# Evidence-Based Psychiatry RAG Assistant
### A Production-Grade Retrieval-Augmented Generation System Using Open Clinical Knowledge Sources

---

## 1. Project Overview

This notebook implements a **complete, end-to-end Retrieval-Augmented Generation (RAG) system** that answers
psychiatry and mental-health questions **exclusively from retrieved, citable evidence** drawn from open,
publicly accessible clinical knowledge sources.

The single most important design rule of this system:

> **The language model is never allowed to answer from its own parametric memory.**
> If the retriever cannot supply sufficient supporting evidence, the system returns
> `"I could not find sufficient evidence in the indexed medical literature."`

### 1.1 Architecture

```
                    ┌──────────────────────── AUTOMATED INGESTION ────────────────────────┐
                    │  PubMed (E-utilities)   Europe PMC (OA full text)   ICD-11 (WHO)    │
                    │  MedlinePlus (NLM)      NIMH (NIH)      WHO reports/mhGAP           │
                    └───────────────────────────────┬─────────────────────────────────────┘
                                                    │
                            Cleaning → Normalisation → Deduplication → Metadata
                                                    │
                                   Semantic (section-aware) recursive chunking
                                                    │
                    ┌───────────────────────────────┴─────────────────────────────────────┐
                    │                        UNIFIED CORPUS (chunks)                      │
                    └───────────┬───────────────────────────────────────┬─────────────────┘
                                │                                       │
                  BAAI/bge-large-en-v1.5 dense vectors            BM25 sparse index
                                │                                       │
                          FAISS (IndexFlatIP)                    rank_bm25 (Okapi)
                                └──────────────┬────────────────────────┘
                                     Hybrid fusion (0.5 dense / 0.5 sparse, tuned)
                                               │
                                  Cross-encoder rerank (bge-reranker-large)
                                               │
                              Confidence gate  →  grounded prompt  →  LLM (Qwen, 4-bit)
                                               │
                        Citation enforcement + grounding verification + confidence score
                                               │
                                     Gradio application (5 tabs)
```

### 1.2 What this notebook demonstrates

| Capability | Implementation |
|---|---|
| Automated data pipelines | 6 fault-tolerant source connectors, retry/backoff, full caching |
| Information retrieval | FAISS dense search, BM25 sparse search, weighted hybrid fusion |
| Reranking | `BAAI/bge-reranker-large` cross-encoder, top-5 / top-10 / top-20 comparison |
| Semantic chunking | Section-aware, heading-driven, recursive with overlap; chunk-size study |
| LLM inference | Transformers + Accelerate + bitsandbytes 4-bit, automatic model cascade |
| Hallucination control | Context verification, citation enforcement, grounding score, confidence gate |
| Evaluation | Recall@k, Precision@k, MRR, MAP, nDCG, Hit-Rate, latency, token usage, RAGAS |
| Explainability | Per-chunk similarity, rerank scores, provenance metadata, source URLs |
| Visualisation | 11 publication-quality matplotlib figures (no seaborn) |
| MLOps | Drive-backed caching, deterministic seeds, structured logging, version manifest |
| Deployment | Polished 5-tab Gradio application |

### 1.3 Data sources, licensing, and documented substitutions

Everything is downloaded automatically. **No manual uploads. No dataset downloads. No API keys are required**
(one *optional* credential pair is supported for the WHO ICD-11 API — see below).

| Source | Access method | Licence / reuse | Status |
|---|---|---|---|
| **PubMed** | NCBI E-utilities REST (`esearch` / `efetch`) | Abstracts are freely accessible; metadata is public | **Primary source** — reviews, systematic reviews, meta-analyses |
| **Europe PMC** | REST `search` + `fullTextXML` | Open-access subset only (CC-BY / CC0 filtered) | Full-text sections for OA articles |
| **MedlinePlus (NLM/NIH)** | Web service `wsearch.nlm.nih.gov` | U.S. Government work — public domain | Consumer-level topic summaries |
| **NIMH (NIH)** | Public topic pages (HTML) | U.S. Government work — public domain | Disorder overviews, treatment summaries |
| **WHO** (mhGAP Intervention Guide, reports) | Direct PDF download, best effort | CC BY-NC-SA 3.0 IGO | Ingested when reachable; degrades gracefully |
| **ICD-11 Chapter 06** | WHO ICD API when credentials exist, otherwise a bundled category scaffold | WHO terms; scaffold contains codes/titles + author-written descriptors | See substitution note below |
| **NICE / APA / VA-DoD guidelines** | Metadata + canonical URLs only | Not openly relicensable | **Reference-only records** (deliberate non-ingestion) |

**Documented substitutions (required by the specification):**

1. **ICD-11.** The WHO ICD-11 API requires OAuth client credentials, which conflicts with the "no API keys"
   requirement. The notebook therefore (a) *uses* the official API automatically if `ICD_CLIENT_ID` /
   `ICD_CLIENT_SECRET` are present in the environment, and otherwise (b) falls back to a bundled scaffold of
   Chapter 06 codes, titles, and hierarchy with short author-written descriptors, and (c) supplements
   diagnostic content with public-domain NIMH/MedlinePlus material. This keeps the pipeline fully automated
   and legally clean.
2. **NICE guidelines.** NICE content is not openly relicensable, so it is ingested as **metadata + URL only**
   (`document_type="guideline_reference"`), exactly as the specification permits.
3. **WHO PDFs.** WHO IRIS URLs change periodically. The connector tries a list of candidate URLs with retry
   and backoff; if all fail it logs the failure, registers reference-only records, and the pipeline continues.
   The corpus is never left empty because PubMed + NLM sources are independent.

### 1.4 How to run

1. `Runtime → Change runtime type → GPU` (T4 or better recommended; the notebook also runs CPU-only in a
   reduced-capability mode).
2. `Runtime → Run all`.
3. On first run, the notebook downloads data and builds indices (~10–25 min depending on GPU/network). Every
   artefact is cached to Google Drive, so subsequent runs start in under a minute.
4. The Gradio app launches at the end with a public share link.

> **Medical disclaimer.** This is an engineering portfolio artefact and an information-retrieval demonstration.
> It is **not** a medical device, is **not** clinically validated, and must **not** be used for diagnosis or
> treatment decisions. Always consult a qualified clinician.

---
## 2. Environment Setup & Dependency Installation

Dependencies are installed idempotently: already-satisfied packages are skipped, optional packages are allowed
to fail without breaking the run, and the notebook records exactly which capabilities are available so every
downstream cell can degrade gracefully instead of crashing.

In [1]:
# =============================================================================
# SECTION 2 — DEPENDENCY INSTALLATION
# =============================================================================
# Design notes
# ------------
# * Installation is *idempotent*: re-running the cell is cheap and safe.
# * Core packages are required; optional packages (bitsandbytes, RAGAS, LangChain
#   wrappers) may fail on some runtimes, and the notebook must survive that.
# * `CAPABILITIES` is a single source of truth consulted by later cells.
# =============================================================================
from __future__ import annotations

import importlib
import os
import subprocess
import sys
import warnings
from typing import Dict, List, Sequence, Tuple

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

IN_COLAB: bool = "google.colab" in sys.modules

# (pip spec, import name) — core requirements.
CORE_PACKAGES: List[Tuple[str, str]] = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
    ("requests", "requests"),
    ("beautifulsoup4", "bs4"),
    ("lxml", "lxml"),
    ("pypdf>=4.2.0", "pypdf"),
    ("rank-bm25>=0.2.2", "rank_bm25"),
    ("faiss-cpu>=1.8.0", "faiss"),
    ("sentence-transformers>=3.0.0", "sentence_transformers"),
    ("transformers>=4.44.0", "transformers"),
    ("accelerate>=0.33.0", "accelerate"),
    ("gradio>=4.44.0", "gradio"),
]

# Optional: the notebook detects their absence and substitutes a fallback path.
OPTIONAL_PACKAGES: List[Tuple[str, str]] = [
    ("bitsandbytes>=0.43.0", "bitsandbytes"),   # 4-bit quantisation for the LLM
    ("ragas>=0.2.0", "ragas"),                  # RAG-specific evaluation metrics
    ("datasets>=2.20.0", "datasets"),           # required by RAGAS
    ("langchain-huggingface>=0.1.0", "langchain_huggingface"),
    ("langchain-community>=0.3.0", "langchain_community"),
    ("nest-asyncio>=1.6.0", "nest_asyncio"),    # RAGAS event-loop support in notebooks
]

INSTALL_LOG: List[str] = []


def _module_available(import_name: str) -> bool:
    """Return True when `import_name` can be imported in the current runtime."""
    try:
        importlib.import_module(import_name)
        return True
    except Exception:
        return False


def pip_install(specs: Sequence[str], quiet: bool = True) -> bool:
    """Install packages with pip, returning True on success.

    Failures are logged rather than raised so that optional dependencies never
    abort the notebook.
    """
    if not specs:
        return True
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade"]
    if quiet:
        cmd.append("-q")
    cmd.extend(specs)
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True, timeout=1800)
        INSTALL_LOG.append(f"installed: {' '.join(specs)}")
        return True
    except subprocess.CalledProcessError as exc:  # pragma: no cover - runtime dependent
        INSTALL_LOG.append(f"FAILED: {' '.join(specs)} :: {exc.stderr[-400:] if exc.stderr else exc}")
        return False
    except Exception as exc:  # pragma: no cover
        INSTALL_LOG.append(f"FAILED: {' '.join(specs)} :: {exc}")
        return False


def ensure_packages(packages: Sequence[Tuple[str, str]], label: str) -> Dict[str, bool]:
    """Install any missing packages from `(spec, import_name)` pairs."""
    missing = [spec for spec, mod in packages if not _module_available(mod)]
    if missing:
        print(f"[setup] Installing {len(missing)} {label} package(s)… this can take a few minutes.")
        # Install one-by-one so a single bad wheel cannot block the rest.
        for spec in missing:
            pip_install([spec])
    else:
        print(f"[setup] All {label} packages already present.")
    return {mod: _module_available(mod) for _, mod in packages}


core_status = ensure_packages(CORE_PACKAGES, "core")
optional_status = ensure_packages(OPTIONAL_PACKAGES, "optional")

CAPABILITIES: Dict[str, bool] = {**core_status, **optional_status}
CAPABILITIES["colab"] = IN_COLAB

missing_core = [mod for _, mod in CORE_PACKAGES if not CAPABILITIES.get(mod)]
print("\n[setup] Core packages OK." if not missing_core else f"\n[setup] MISSING CORE: {missing_core}")
print("[setup] Optional availability:")
for _, mod in OPTIONAL_PACKAGES:
    print(f"    - {mod:<26} {'available' if CAPABILITIES.get(mod) else 'unavailable (fallback will be used)'}")

_install_failures = [entry for entry in INSTALL_LOG if entry.startswith("FAILED")]
if _install_failures:
    print("\n[setup] Install failures (each has a documented fallback; shown so the cause is diagnosable):")
    for entry in _install_failures[:6]:
        print(f"    ! {entry[:400]}")
print(
    "\n[setup] If Colab asks you to restart the runtime, do so and then simply run all cells again: "
    "every stage is cached and idempotent, so nothing is recomputed unnecessarily."
)

[setup] Installing 3 core package(s)… this can take a few minutes.
[setup] Installing 4 optional package(s)… this can take a few minutes.

[setup] Core packages OK.
[setup] Optional availability:
    - bitsandbytes               available
    - ragas                      unavailable (fallback will be used)
    - datasets                   available
    - langchain_huggingface      available
    - langchain_community        available
    - nest_asyncio               available

[setup] If Colab asks you to restart the runtime, do so and then simply run all cells again: every stage is cached and idempotent, so nothing is recomputed unnecessarily.


In [2]:
# =============================================================================
# SECTION 2b — IMPORTS, LOGGING, DETERMINISM, VERSION MANIFEST
# =============================================================================
from __future__ import annotations

import gc
import hashlib
import importlib
import html
import json
import logging
import math
import pickle
import platform
import random
import re
import shutil
import time
import traceback
import unicodedata
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Set, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm

matplotlib.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 200,
        "figure.autolayout": True,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
    }
)

# -----------------------------------------------------------------------------
# Structured logging: everything is captured in-memory *and* written to disk so
# that a long ingestion run can be audited after the fact.
# -----------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("psych-rag")
logging.getLogger("urllib3").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

ERROR_LOG: List[Dict[str, str]] = []


def log_error(stage: str, exc: BaseException, context: str = "") -> None:
    """Record a non-fatal error without interrupting execution.

    The notebook is explicitly designed to *never* crash on a data-source
    failure: every connector, parser and download path funnels here.
    """
    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "stage": stage,
        "error_type": type(exc).__name__,
        "message": str(exc)[:500],
        "context": context[:300],
        "traceback": traceback.format_exc(limit=3)[-800:],
    }
    ERROR_LOG.append(entry)
    logger.warning("[%s] %s: %s %s", stage, type(exc).__name__, str(exc)[:180], f"({context[:120]})" if context else "")


# -----------------------------------------------------------------------------
# Reproducibility
# -----------------------------------------------------------------------------
GLOBAL_SEED: int = 42


def set_global_seeds(seed: int = GLOBAL_SEED) -> None:
    """Seed every stochastic component used in this notebook."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Deterministic cuDNN costs a little speed but makes results repeatable.
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_global_seeds(GLOBAL_SEED)

# -----------------------------------------------------------------------------
# Hardware detection
# -----------------------------------------------------------------------------
DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME: str = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "N/A"
GPU_MEMORY_GB: float = (
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if DEVICE == "cuda" else 0.0
)

HARDWARE_INFO: Dict[str, Any] = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": DEVICE,
    "gpu_name": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "cpu_count": os.cpu_count(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda if DEVICE == "cuda" else None,
    "in_colab": IN_COLAB,
}


def build_version_manifest() -> Dict[str, str]:
    """Capture installed versions of every library that affects results."""
    manifest: Dict[str, str] = {}
    for name in [
        "numpy",
        "pandas",
        "torch",
        "transformers",
        "sentence_transformers",
        "faiss",
        "rank_bm25",
        "gradio",
        "matplotlib",
        "pypdf",
        "bs4",
        "ragas",
        "datasets",
        "bitsandbytes",
        "accelerate",
    ]:
        try:
            module = importlib.import_module(name)
            manifest[name] = getattr(module, "__version__", "unknown")
        except Exception:
            manifest[name] = "not installed"
    return manifest


VERSION_MANIFEST: Dict[str, str] = build_version_manifest()

print("=" * 78)
print("ENVIRONMENT")
print("=" * 78)
for key, value in HARDWARE_INFO.items():
    print(f"  {key:<16}: {value}")
print("-" * 78)
print("PACKAGE VERSIONS")
for key, value in sorted(VERSION_MANIFEST.items()):
    print(f"  {key:<22}: {value}")
print("=" * 78)
print(f"Global seed fixed at {GLOBAL_SEED} for Python / NumPy / PyTorch.")

ENVIRONMENT
  python          : 3.12.13
  platform        : Linux-6.6.122+-x86_64-with-glibc2.35
  device          : cuda
  gpu_name        : Tesla T4
  gpu_memory_gb   : 14.56
  cpu_count       : 2
  torch           : 2.11.0+cu128
  cuda            : 12.8
  in_colab        : True
------------------------------------------------------------------------------
PACKAGE VERSIONS
  accelerate            : 1.14.0
  bitsandbytes          : 0.50.0
  bs4                   : 4.13.5
  datasets              : 4.0.0
  faiss                 : 1.14.3
  gradio                : 6.20.0
  matplotlib            : 3.10.0
  numpy                 : 2.0.2
  pandas                : 2.2.2
  pypdf                 : 6.14.2
  ragas                 : not installed
  rank_bm25             : unknown
  sentence_transformers : 5.6.0
  torch                 : 2.11.0+cu128
  transformers          : 5.13.1
Global seed fixed at 42 for Python / NumPy / PyTorch.


---
## 3. Configuration & Google Drive Caching

All tunable behaviour lives in a single immutable-by-convention `RAGConfig` dataclass, and every expensive
artefact (raw downloads, cleaned corpus, embeddings, FAISS index, BM25 index, evaluation results, plots) is
written under one cache root. When Drive is available the cache survives runtime resets, so a second run of
the notebook loads in seconds.

Set `CONFIG.force_rebuild = True` (or `RAGConfig(force_rebuild=True)`) to invalidate every cached stage.

In [3]:
# =============================================================================
# SECTION 3 — CONFIGURATION, PATHS, CACHE MANAGER
# =============================================================================
from __future__ import annotations


@dataclass
class RAGConfig:
    """Central configuration object for the entire pipeline.

    Every magic number in this notebook lives here, which makes ablations
    (chunk size, fusion weights, top-k, thresholds) a one-line change.
    """

    # --- identity & reproducibility -----------------------------------------
    project_name: str = "psychiatry_rag"
    # Bump when ingestion/cleaning/chunking logic changes: artefacts produced by an
    # older pipeline are then rebuilt instead of silently reloaded from Drive.
    pipeline_version: str = "v3"
    # Bump when benchmark generation or the split algorithm changes, so cached
    # benchmarks and evaluations rebuild rather than silently persisting.
    benchmark_version: str = "b2"
    seed: int = GLOBAL_SEED
    force_rebuild: bool = False

    # --- caching -------------------------------------------------------------
    use_google_drive: bool = True
    drive_mount_point: str = "/content/drive"
    local_cache_dir: str = "psychiatry_rag_cache"
    hf_token_env: str = "HF_TOKEN"
    # Model weights on Drive survive a runtime reset (no re-download), but a free
    # Drive is 15 GB, so this is opt-in and only sensible with the default 4B model.
    hf_cache_on_drive: bool = False

    # --- ingestion: PubMed ---------------------------------------------------
    entrez_base: str = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    entrez_tool: str = "evidence-based-psychiatry-rag"
    entrez_email: str = "psychiatry.rag.notebook@example.com"
    entrez_api_key: Optional[str] = None          # optional; raises the rate limit to 10 rps
    pubmed_max_per_topic: int = 60
    pubmed_min_year: int = 2015
    pubmed_batch_size: int = 40

    # --- ingestion: Europe PMC ----------------------------------------------
    europepmc_base: str = "https://www.ebi.ac.uk/europepmc/webservices/rest"
    europepmc_max_per_topic: int = 8              # OA full texts are large; keep modest
    europepmc_enabled: bool = True

    # --- ingestion: NLM / NIMH / WHO ----------------------------------------
    medlineplus_base: str = "https://wsearch.nlm.nih.gov/ws/query"
    nimh_enabled: bool = True
    who_enabled: bool = True
    icd11_client_id_env: str = "ICD_CLIENT_ID"
    icd11_client_secret_env: str = "ICD_CLIENT_SECRET"

    # --- HTTP behaviour ------------------------------------------------------
    http_timeout: int = 45
    http_max_retries: int = 4
    http_backoff: float = 1.6
    http_pause: float = 0.34                      # NCBI: <= 3 requests/second without a key

    # --- cleaning ------------------------------------------------------------
    min_document_chars: int = 220
    near_duplicate_threshold: float = 0.90

    # --- chunking ------------------------------------------------------------
    chunk_target_tokens: int = 320
    chunk_overlap_tokens: int = 64
    chunk_min_tokens: int = 40
    chunk_max_tokens: int = 512
    chunk_size_experiment: Tuple[int, ...] = (192, 320, 448)

    # --- embeddings ----------------------------------------------------------
    embedding_model_primary: str = "BAAI/bge-large-en-v1.5"
    embedding_model_fallback: str = "BAAI/bge-base-en-v1.5"
    embedding_model_min_gpu_gb: float = 11.0      # below this we prefer bge-base
    embedding_batch_size: int = 32
    embedding_max_seq_length: int = 512
    query_instruction: str = "Represent this sentence for searching relevant passages: "

    # --- retrieval -----------------------------------------------------------
    # Tuned on the validation split (see Section 18): 0.5/0.5 beat the 0.6/0.4
    # starting point on nDCG@10, recall@5 and MRR.
    dense_weight: float = 0.5
    bm25_weight: float = 0.5
    retrieve_candidates: int = 50                 # pool size handed to the reranker
    top_k_default: int = 5
    rerank_depths: Tuple[int, ...] = (5, 10, 20)

    # --- reranking -----------------------------------------------------------
    reranker_model_primary: str = "BAAI/bge-reranker-large"
    reranker_model_fallback: str = "BAAI/bge-reranker-base"
    reranker_min_gpu_gb: float = 11.0
    reranker_batch_size: int = 16
    reranker_enabled: bool = True

    # --- LLM -----------------------------------------------------------------
    # Default cascade is sized for a COLD Colab runtime. An 8B checkpoint is a
    # ~16 GB download, and unauthenticated Hub downloads are throttled to roughly
    # 10 MB/s, which can exceed an hour. Qwen3-4B is ~8 GB and, for constrained
    # extract-and-cite generation over supplied passages, close in quality.
    # Set llm_prefer_large=True (ideally with an HF token) for the 8B models.
    llm_candidates: Tuple[str, ...] = (
        "Qwen/Qwen3-4B",                 # ~8 GB  — default
        "Qwen/Qwen2.5-3B-Instruct",      # ~6 GB
        "Qwen/Qwen2.5-1.5B-Instruct",    # ~3 GB, CPU-feasible
    )
    llm_large_candidates: Tuple[str, ...] = (
        "Qwen/Qwen3-8B",
        "Qwen/Qwen2.5-7B-Instruct",
    )
    llm_prefer_large: bool = False
    llm_max_new_tokens: int = 512
    llm_temperature: float = 0.1
    llm_top_p: float = 0.9
    llm_use_4bit: bool = True
    llm_min_gpu_gb_for_7b: float = 13.0
    llm_enabled: bool = True

    # --- grounding / hallucination control -----------------------------------
    min_retrieval_confidence: float = 0.55        # below this the system refuses to answer
    min_grounding_score: float = 0.42             # sentence-level support threshold
    # BGE cosine calibration band: below the floor a passage is off-domain, above
    # the ceiling it is a strong topical match. Used by the confidence gate and by
    # the native RAG metrics so both share one calibration.
    dense_similarity_floor: float = 0.55
    dense_similarity_ceiling: float = 0.80
    require_citations: bool = True

    # --- benchmark & evaluation ---------------------------------------------
    benchmark_max_questions: int = 480
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    eval_retrieval_sample: int = 150              # questions used for retrieval metrics
    eval_generation_sample: int = 24              # questions used for (slow) RAG metrics

    # --- app -----------------------------------------------------------------
    gradio_share: bool = True

    def summary(self) -> Dict[str, Any]:
        """Flat dictionary of the configuration, used by the System Info tab."""
        return asdict(self)


CONFIG = RAGConfig()


@dataclass
class ProjectPaths:
    """Filesystem layout for every cached artefact."""

    base: Path

    def __post_init__(self) -> None:
        self.raw = self.base / "01_raw"
        self.processed = self.base / "02_processed"
        self.embeddings = self.base / "03_embeddings"
        self.indices = self.base / "04_indices"
        self.benchmark = self.base / "05_benchmark"
        self.evaluation = self.base / "06_evaluation"
        self.plots = self.base / "07_plots"
        self.models = self.base / "08_models"
        self.logs = self.base / "09_logs"
        for directory in (
            self.base,
            self.raw,
            self.processed,
            self.embeddings,
            self.indices,
            self.benchmark,
            self.evaluation,
            self.plots,
            self.models,
            self.logs,
        ):
            directory.mkdir(parents=True, exist_ok=True)

    def describe(self) -> str:
        return "\n".join(
            f"  {name:<12}: {value}"
            for name, value in vars(self).items()
            if isinstance(value, Path)
        )


def resolve_cache_root(config: RAGConfig) -> Path:
    """Mount Google Drive when possible and return the cache root directory.

    Falls back silently to a local directory when Drive is unavailable (e.g.
    local Jupyter, or a user who declines the mount prompt).
    """
    if config.use_google_drive and IN_COLAB:
        try:
            from google.colab import drive  # type: ignore

            if not os.path.ismount(config.drive_mount_point):
                drive.mount(config.drive_mount_point, force_remount=False)
            root = Path(config.drive_mount_point) / "MyDrive" / config.project_name
            root.mkdir(parents=True, exist_ok=True)
            logger.info("Cache root on Google Drive: %s", root)
            return root
        except Exception as exc:
            log_error("drive_mount", exc, "falling back to local cache")
    root = Path.cwd() / config.local_cache_dir
    root.mkdir(parents=True, exist_ok=True)
    logger.info("Cache root (local): %s", root)
    return root


PATHS = ProjectPaths(resolve_cache_root(CONFIG))

def resolve_hf_token(config: RAGConfig) -> Optional[str]:
    """Find a Hugging Face token in the environment or Colab secrets.

    This is purely a throughput concern, not an access one: every model used here
    is public. Unauthenticated Hub downloads are heavily rate-limited, which turns
    a model fetch into a multi-hour stall on a cold runtime.
    """
    token = os.environ.get(config.hf_token_env)
    if not token and IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore

            token = userdata.get(config.hf_token_env)
        except Exception:
            token = None
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    return token


HF_TOKEN = resolve_hf_token(CONFIG)

# Where model weights live. Local disk is fast but is wiped on every runtime
# reset; Drive persists but is size-limited on the free tier.
HF_CACHE_DIR = (PATHS.models / "huggingface") if CONFIG.hf_cache_on_drive else (Path.cwd() / "hf_home")
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

print(f"HF cache      : {HF_CACHE_DIR}"
      f"{' (Drive-persistent)' if CONFIG.hf_cache_on_drive else ' (local; cleared on runtime reset)'}")
print(
    "HF auth       : token found — downloads run at full speed."
    if HF_TOKEN
    else "HF auth       : no token. Hub downloads are rate-limited (~10 MB/s). Set HF_TOKEN in\n"
         "                Colab Secrets (key icon, left sidebar) to speed up first-run model fetches."
)


class CacheManager:
    """Tiny, dependency-free artefact cache with pluggable serialisation formats.

    The `get_or_build` contract is the backbone of this notebook's idempotency:
    every expensive stage is expressed as "load if present, otherwise build and
    persist".
    """

    def __init__(self, force_rebuild: bool = False) -> None:
        self.force_rebuild = force_rebuild
        self.hits: List[str] = []
        self.misses: List[str] = []

    # -- serialisation helpers ------------------------------------------------
    @staticmethod
    def _read(path: Path, fmt: str) -> Any:
        if fmt == "json":
            with path.open("r", encoding="utf-8") as handle:
                return json.load(handle)
        if fmt == "pickle":
            with path.open("rb") as handle:
                return pickle.load(handle)
        if fmt == "numpy":
            return np.load(path)
        raise ValueError(f"Unsupported cache format: {fmt}")

    @staticmethod
    def _write(obj: Any, path: Path, fmt: str) -> None:
        tmp = path.with_suffix(path.suffix + ".tmp")
        if fmt == "json":
            with tmp.open("w", encoding="utf-8") as handle:
                json.dump(obj, handle, ensure_ascii=False, indent=1, default=str)
        elif fmt == "pickle":
            with tmp.open("wb") as handle:
                pickle.dump(obj, handle, protocol=pickle.HIGHEST_PROTOCOL)
        elif fmt == "numpy":
            np.save(tmp, obj)
            tmp = tmp.with_suffix(tmp.suffix + ".npy") if not tmp.name.endswith(".npy") else tmp
        else:
            raise ValueError(f"Unsupported cache format: {fmt}")
        shutil.move(str(tmp), str(path))  # atomic-ish: never leave a half-written cache

    # -- public API -----------------------------------------------------------
    def get_or_build(
        self,
        path: Path,
        build_fn: Callable[[], Any],
        fmt: str = "pickle",
        force: Optional[bool] = None,
        label: Optional[str] = None,
    ) -> Any:
        """Return the cached artefact at `path`, building and persisting it if needed."""
        name = label or path.name
        force_rebuild = self.force_rebuild if force is None else force
        if path.exists() and not force_rebuild:
            try:
                obj = self._read(path, fmt)
                self.hits.append(name)
                logger.info("cache HIT  → %s", name)
                return obj
            except Exception as exc:
                log_error("cache_read", exc, str(path))
        self.misses.append(name)
        logger.info("cache MISS → building %s", name)
        started = time.perf_counter()
        obj = build_fn()
        try:
            self._write(obj, path, fmt)
        except Exception as exc:
            log_error("cache_write", exc, str(path))
        logger.info("built %s in %.1fs", name, time.perf_counter() - started)
        return obj

    def report(self) -> pd.DataFrame:
        rows = [{"artifact": n, "status": "hit"} for n in self.hits]
        rows += [{"artifact": n, "status": "built"} for n in self.misses]
        return pd.DataFrame(rows)


CACHE = CacheManager(force_rebuild=CONFIG.force_rebuild)

print("Cache layout:")
print(PATHS.describe())
print(f"\nforce_rebuild = {CONFIG.force_rebuild}  |  device = {DEVICE}  |  seed = {CONFIG.seed}")

Mounted at /content/drive


18:56:48 | INFO    | psych-rag | Cache root on Google Drive: /content/drive/MyDrive/psychiatry_rag


HF cache      : /content/hf_home (local; cleared on runtime reset)
HF auth       : no token. Hub downloads are rate-limited (~10 MB/s). Set HF_TOKEN in
                Colab Secrets (key icon, left sidebar) to speed up first-run model fetches.
Cache layout:
  base        : /content/drive/MyDrive/psychiatry_rag
  raw         : /content/drive/MyDrive/psychiatry_rag/01_raw
  processed   : /content/drive/MyDrive/psychiatry_rag/02_processed
  embeddings  : /content/drive/MyDrive/psychiatry_rag/03_embeddings
  indices     : /content/drive/MyDrive/psychiatry_rag/04_indices
  benchmark   : /content/drive/MyDrive/psychiatry_rag/05_benchmark
  evaluation  : /content/drive/MyDrive/psychiatry_rag/06_evaluation
  plots       : /content/drive/MyDrive/psychiatry_rag/07_plots
  models      : /content/drive/MyDrive/psychiatry_rag/08_models
  logs        : /content/drive/MyDrive/psychiatry_rag/09_logs

force_rebuild = False  |  device = cuda  |  seed = 42


---
## 4. Automated Data Ingestion

Every source is implemented as a `BaseConnector` subclass with a single contract: `fetch() -> List[RawDocument]`.
The orchestrator runs each connector inside an isolation boundary, so a network outage, an HTML redesign, or a
moved WHO PDF degrades the corpus rather than crashing the notebook.

**Resilience features**
* `HttpClient` — pooled session, exponential backoff, jittered retries, per-host rate limiting, status-code aware retry policy.
* Per-document `try/except` in every parser (a single malformed XML record cannot kill a batch of 40).
* All raw payloads cached to disk before parsing, so re-runs never re-download.
* Structured error log surfaced in the System Information tab.

In [4]:
# =============================================================================
# SECTION 4a — RESILIENT HTTP CLIENT
# =============================================================================


class HttpClient:
    """A retrying, rate-limited HTTP client used by every connector.

    Retries are applied to transient failures only (timeouts, connection resets,
    429/5xx). Permanent failures (404/403) fail fast so the caller can move on to
    the next candidate URL.
    """

    RETRYABLE_STATUS = {408, 425, 429, 500, 502, 503, 504}

    def __init__(self, config: RAGConfig) -> None:
        self.config = config
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": (
                    "EvidenceBasedPsychiatryRAG/1.0 (research/education; "
                    f"mailto:{config.entrez_email})"
                ),
                "Accept": "*/*",
                "Accept-Language": "en",
            }
        )
        self._last_request_at: Dict[str, float] = defaultdict(float)
        self.request_count: int = 0

    # -- internals ------------------------------------------------------------
    def _throttle(self, url: str) -> None:
        """Politely rate-limit per host (NCBI allows ~3 requests/second)."""
        host = requests.utils.urlparse(url).netloc
        elapsed = time.perf_counter() - self._last_request_at[host]
        wait = self.config.http_pause - elapsed
        if wait > 0:
            time.sleep(wait)
        self._last_request_at[host] = time.perf_counter()

    def request(
        self,
        url: str,
        params: Optional[Dict[str, Any]] = None,
        method: str = "GET",
        stream: bool = False,
        timeout: Optional[int] = None,
    ) -> Optional[requests.Response]:
        """Perform a request with retries. Returns None when unrecoverable."""
        timeout = timeout or self.config.http_timeout
        last_exc: Optional[BaseException] = None
        for attempt in range(self.config.http_max_retries):
            try:
                self._throttle(url)
                self.request_count += 1
                response = self.session.request(
                    method, url, params=params, timeout=timeout, stream=stream
                )
                if response.status_code == 200:
                    return response
                if response.status_code in self.RETRYABLE_STATUS:
                    raise requests.HTTPError(f"retryable status {response.status_code}")
                logger.debug("non-retryable status %s for %s", response.status_code, url)
                return None
            except Exception as exc:  # noqa: BLE001 - deliberately broad
                last_exc = exc
                sleep_for = (self.config.http_backoff ** attempt) + random.uniform(0, 0.5)
                if attempt < self.config.http_max_retries - 1:
                    time.sleep(sleep_for)
        if last_exc is not None:
            log_error("http", last_exc, url[:200])
        return None

    # -- convenience wrappers -------------------------------------------------
    def get_text(self, url: str, params: Optional[Dict[str, Any]] = None) -> Optional[str]:
        response = self.request(url, params=params)
        if response is None:
            return None
        response.encoding = response.encoding or "utf-8"
        return response.text

    def get_json(self, url: str, params: Optional[Dict[str, Any]] = None) -> Optional[Any]:
        response = self.request(url, params=params)
        if response is None:
            return None
        try:
            return response.json()
        except Exception as exc:
            log_error("http_json", exc, url[:200])
            return None

    def download(self, url: str, destination: Path, min_bytes: int = 2048) -> Optional[Path]:
        """Stream a binary file to disk; returns None on failure or truncation."""
        if destination.exists() and destination.stat().st_size >= min_bytes:
            return destination
        response = self.request(url, stream=True, timeout=180)
        if response is None:
            return None
        try:
            tmp = destination.with_suffix(destination.suffix + ".part")
            with tmp.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=1 << 16):
                    if chunk:
                        handle.write(chunk)
            if tmp.stat().st_size < min_bytes:
                tmp.unlink(missing_ok=True)
                return None
            shutil.move(str(tmp), str(destination))
            return destination
        except Exception as exc:
            log_error("download", exc, url[:200])
            return None


HTTP = HttpClient(CONFIG)

# -----------------------------------------------------------------------------
# Clinical topic taxonomy — drives PubMed queries, NIMH pages, and topic metadata
# -----------------------------------------------------------------------------
PSYCHIATRY_TOPICS: Dict[str, Dict[str, Any]] = {
    "depression": {
        "label": "Depressive disorders",
        "mesh": "Depressive Disorder, Major",
        "keywords": ["major depressive disorder", "depression", "unipolar depression"],
        "nimh_slug": "depression",
        "medlineplus": "depression",
    },
    "anxiety": {
        "label": "Anxiety disorders",
        "mesh": "Anxiety Disorders",
        "keywords": ["generalized anxiety disorder", "panic disorder", "social anxiety"],
        "nimh_slug": "anxiety-disorders",
        "medlineplus": "anxiety",
    },
    "bipolar_disorder": {
        "label": "Bipolar disorder",
        "mesh": "Bipolar Disorder",
        "keywords": ["bipolar disorder", "mania", "bipolar I", "bipolar II"],
        "nimh_slug": "bipolar-disorder",
        "medlineplus": "bipolar disorder",
    },
    "schizophrenia": {
        "label": "Schizophrenia and psychotic disorders",
        "mesh": "Schizophrenia",
        "keywords": ["schizophrenia", "psychosis", "antipsychotic"],
        "nimh_slug": "schizophrenia",
        "medlineplus": "schizophrenia",
    },
    "ptsd": {
        "label": "Post-traumatic stress disorder",
        "mesh": "Stress Disorders, Post-Traumatic",
        "keywords": ["post-traumatic stress disorder", "PTSD", "trauma"],
        "nimh_slug": "post-traumatic-stress-disorder-ptsd",
        "medlineplus": "post-traumatic stress disorder",
    },
    "ocd": {
        "label": "Obsessive-compulsive disorder",
        "mesh": "Obsessive-Compulsive Disorder",
        "keywords": ["obsessive-compulsive disorder", "OCD", "compulsions"],
        "nimh_slug": "obsessive-compulsive-disorder-ocd",
        "medlineplus": "obsessive compulsive disorder",
    },
    "adhd": {
        "label": "Attention-deficit/hyperactivity disorder",
        "mesh": "Attention Deficit Disorder with Hyperactivity",
        "keywords": ["ADHD", "attention deficit hyperactivity disorder"],
        "nimh_slug": "attention-deficit-hyperactivity-disorder-adhd",
        "medlineplus": "attention deficit hyperactivity disorder",
    },
    "personality_disorders": {
        "label": "Personality disorders",
        "mesh": "Personality Disorders",
        "keywords": ["borderline personality disorder", "personality disorder"],
        "nimh_slug": "borderline-personality-disorder",
        "medlineplus": "personality disorders",
    },
    "eating_disorders": {
        "label": "Feeding and eating disorders",
        "mesh": "Feeding and Eating Disorders",
        "keywords": ["anorexia nervosa", "bulimia nervosa", "binge eating disorder"],
        "nimh_slug": "eating-disorders",
        "medlineplus": "eating disorders",
    },
    "substance_use_disorders": {
        "label": "Substance use disorders",
        "mesh": "Substance-Related Disorders",
        "keywords": ["substance use disorder", "alcohol use disorder", "opioid use disorder"],
        "nimh_slug": "substance-use-and-mental-health",
        "medlineplus": "substance use disorder",
    },
    "suicide_prevention": {
        "label": "Suicide and self-harm prevention",
        "mesh": "Suicide Prevention",
        "keywords": ["suicide prevention", "suicidal ideation", "self-harm"],
        "nimh_slug": "suicide-prevention",
        "medlineplus": "suicide",
    },
    "autism_neurodevelopment": {
        "label": "Neurodevelopmental disorders",
        "mesh": "Autism Spectrum Disorder",
        "keywords": ["autism spectrum disorder", "neurodevelopmental disorder"],
        "nimh_slug": "autism-spectrum-disorders-asd",
        "medlineplus": "autism spectrum disorder",
    },
}

TOPIC_KEYS: List[str] = list(PSYCHIATRY_TOPICS)
print(f"Configured {len(TOPIC_KEYS)} clinical topics: {', '.join(TOPIC_KEYS)}")

Configured 12 clinical topics: depression, anxiety, bipolar_disorder, schizophrenia, ptsd, ocd, adhd, personality_disorders, eating_disorders, substance_use_disorders, suicide_prevention, autism_neurodevelopment


In [5]:
# =============================================================================
# SECTION 4b — DATA MODEL & CONNECTOR BASE CLASS
# =============================================================================


@dataclass
class RawDocument:
    """A single ingested document prior to cleaning and chunking.

    `sections` preserves any native structure discovered at ingestion time
    (e.g. structured PubMed abstracts, JATS article sections). Preserving it here
    is what makes *semantic* rather than naive chunking possible later.
    """

    doc_id: str
    source: str                       # e.g. "pubmed", "who_mhgap"
    organization: str                 # e.g. "NCBI / PubMed", "World Health Organization"
    title: str
    text: str
    url: str = ""
    year: Optional[int] = None
    topic: str = "general"
    document_type: str = "article"    # article | review | guideline | classification | patient_education | guideline_reference
    language: str = "en"
    license_note: str = ""
    sections: List[Dict[str, str]] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    @property
    def n_chars(self) -> int:
        return len(self.text or "")


def normalize_title(raw: str) -> str:
    """Collapse whitespace in titles (HTML sources emit embedded newlines)."""
    return re.sub(r"\s+", " ", (raw or "")).strip()


def infer_topic_from_text(text: str, default: str = "general") -> str:
    """Map free text onto the clinical topic taxonomy by keyword match."""
    lowered = (text or "").lower()
    for topic_key, meta in PSYCHIATRY_TOPICS.items():
        if any(keyword.lower() in lowered for keyword in meta["keywords"]):
            return topic_key
        if meta["label"].lower().rstrip("s") in lowered:
            return topic_key
    return default


def make_doc_id(source: str, key: str) -> str:
    """Deterministic, collision-resistant document identifier."""
    digest = hashlib.sha1(f"{source}::{key}".encode("utf-8")).hexdigest()[:12]
    return f"{source}-{digest}"


class BaseConnector:
    """Contract for every data source connector.

    Subclasses implement `_fetch`. The public `fetch` wrapper provides caching,
    timing, exception isolation and consistent logging so that the ingestion
    orchestrator never has to reason about source-specific failure modes.
    """

    name: str = "base"
    organization: str = "unknown"
    license_note: str = ""

    def __init__(self, config: RAGConfig, http: HttpClient, paths: ProjectPaths) -> None:
        self.config = config
        self.http = http
        self.paths = paths
        self.stats: Dict[str, Any] = {}

    # -- to be implemented by subclasses -------------------------------------
    def _fetch(self) -> List[RawDocument]:
        raise NotImplementedError

    # -- public API -----------------------------------------------------------
    def fetch(self) -> List[RawDocument]:
        started = time.perf_counter()
        try:
            documents = self._fetch()
        except Exception as exc:  # noqa: BLE001
            log_error(f"connector:{self.name}", exc, "connector aborted; continuing with other sources")
            documents = []
        elapsed = time.perf_counter() - started
        self.stats = {
            "connector": self.name,
            "documents": len(documents),
            "characters": sum(doc.n_chars for doc in documents),
            "seconds": round(elapsed, 1),
        }
        logger.info(
            "connector %-22s → %4d documents, %8d chars in %5.1fs",
            self.name,
            len(documents),
            self.stats["characters"],
            elapsed,
        )
        return documents

    # -- shared helpers -------------------------------------------------------
    def cache_file(self, filename: str) -> Path:
        directory = self.paths.raw / self.name
        directory.mkdir(parents=True, exist_ok=True)
        return directory / filename

    @staticmethod
    def strip_html(raw_html: str) -> str:
        """Convert an HTML fragment to readable plain text."""
        if not raw_html:
            return ""
        try:
            from bs4 import BeautifulSoup  # imported lazily: optional at ingest time

            soup = BeautifulSoup(raw_html, "lxml")
            for tag in soup(["script", "style", "nav", "footer", "header", "form", "aside", "noscript"]):
                tag.decompose()
            text = soup.get_text("\n")
        except Exception:
            text = re.sub(r"<[^>]+>", " ", raw_html)
        return html.unescape(text)

In [6]:
# =============================================================================
# SECTION 4c — PUBMED CONNECTOR (NCBI E-utilities)
# =============================================================================


class PubMedConnector(BaseConnector):
    """Harvest psychiatry review literature from PubMed via E-utilities.

    Query strategy
    --------------
    For each clinical topic we restrict to *evidence-synthesis* publication types
    (review / systematic review / meta-analysis), English language, and a recency
    window. Structured abstracts are preserved section-by-section
    (BACKGROUND / METHODS / RESULTS / CONCLUSIONS), which downstream chunking
    exploits directly.
    """

    name = "pubmed"
    organization = "NCBI / U.S. National Library of Medicine"
    license_note = "PubMed abstracts and metadata are freely accessible for research use."

    PUBLICATION_TYPE_FILTER = (
        '("review"[Publication Type] OR "systematic review"[Publication Type] '
        'OR "meta-analysis"[Publication Type] OR "practice guideline"[Publication Type])'
    )

    def _base_params(self) -> Dict[str, Any]:
        params: Dict[str, Any] = {"tool": self.config.entrez_tool, "email": self.config.entrez_email}
        if self.config.entrez_api_key:
            params["api_key"] = self.config.entrez_api_key
        return params

    def _build_query(self, topic_key: str) -> str:
        meta = PSYCHIATRY_TOPICS[topic_key]
        mesh = meta["mesh"]
        keyword_clause = " OR ".join(f'"{kw}"[Title/Abstract]' for kw in meta["keywords"])
        return (
            f'(("{mesh}"[MeSH Terms]) OR ({keyword_clause})) '
            f"AND {self.PUBLICATION_TYPE_FILTER} "
            f'AND ("{self.config.pubmed_min_year}"[Date - Publication] : "3000"[Date - Publication]) '
            f'AND "english"[Language] AND hasabstract'
        )

    def _search(self, topic_key: str) -> List[str]:
        """Return PMIDs for one topic (cached per topic)."""
        cache_path = self.cache_file(f"search_{topic_key}.json")
        if cache_path.exists() and not self.config.force_rebuild:
            try:
                return json.loads(cache_path.read_text(encoding="utf-8"))
            except Exception as exc:
                log_error("pubmed_cache", exc, str(cache_path))
        params = self._base_params()
        params.update(
            {
                "db": "pubmed",
                "term": self._build_query(topic_key),
                "retmax": self.config.pubmed_max_per_topic,
                "retmode": "json",
                "sort": "relevance",
            }
        )
        payload = self.http.get_json(f"{self.config.entrez_base}/esearch.fcgi", params=params)
        pmids: List[str] = []
        if payload:
            pmids = list(payload.get("esearchresult", {}).get("idlist", []))
        cache_path.write_text(json.dumps(pmids), encoding="utf-8")
        return pmids

    def _efetch(self, pmids: Sequence[str]) -> Optional[str]:
        """Fetch a batch of PubMed records as XML (cached by batch hash)."""
        batch_key = hashlib.sha1(",".join(pmids).encode()).hexdigest()[:16]
        cache_path = self.cache_file(f"efetch_{batch_key}.xml")
        if cache_path.exists() and not self.config.force_rebuild:
            return cache_path.read_text(encoding="utf-8", errors="ignore")
        params = self._base_params()
        params.update({"db": "pubmed", "id": ",".join(pmids), "retmode": "xml", "rettype": "abstract"})
        xml_text = self.http.get_text(f"{self.config.entrez_base}/efetch.fcgi", params=params)
        if xml_text:
            cache_path.write_text(xml_text, encoding="utf-8")
        return xml_text

    # -- parsing --------------------------------------------------------------
    @staticmethod
    def _node_text(node: Optional[ET.Element]) -> str:
        """Flatten an element (including inline markup) to plain text."""
        if node is None:
            return ""
        return re.sub(r"\s+", " ", "".join(node.itertext())).strip()

    def _parse_article(self, article: ET.Element, topic_key: str) -> Optional[RawDocument]:
        pmid = self._node_text(article.find(".//PMID"))
        title = self._node_text(article.find(".//ArticleTitle"))
        if not pmid or not title:
            return None

        # Structured abstracts expose <AbstractText Label="RESULTS"> — keep them.
        sections: List[Dict[str, str]] = []
        for abstract_node in article.findall(".//Abstract/AbstractText"):
            body = self._node_text(abstract_node)
            if not body:
                continue
            label = (abstract_node.get("Label") or abstract_node.get("NlmCategory") or "Abstract").title()
            sections.append({"heading": label, "text": body})
        if not sections:
            return None

        year_text = (
            self._node_text(article.find(".//Journal/JournalIssue/PubDate/Year"))
            or self._node_text(article.find(".//Journal/JournalIssue/PubDate/MedlineDate"))[:4]
            or self._node_text(article.find(".//ArticleDate/Year"))
        )
        try:
            year = int(year_text[:4])
        except Exception:
            year = None

        journal = self._node_text(article.find(".//Journal/Title"))
        pub_types = [self._node_text(node) for node in article.findall(".//PublicationTypeList/PublicationType")]
        doi = ""
        for eid in article.findall(".//ArticleId"):
            if eid.get("IdType") == "doi":
                doi = self._node_text(eid)
        keywords = [self._node_text(node) for node in article.findall(".//KeywordList/Keyword")][:12]
        mesh_terms = [self._node_text(node) for node in article.findall(".//MeshHeading/DescriptorName")][:20]

        document_type = "review"
        lowered = " ".join(pub_types).lower()
        if "meta-analysis" in lowered:
            document_type = "meta_analysis"
        elif "systematic" in lowered:
            document_type = "systematic_review"
        elif "guideline" in lowered:
            document_type = "guideline"

        body_text = "\n\n".join(f"{s['heading']}\n{s['text']}" for s in sections)
        return RawDocument(
            doc_id=make_doc_id(self.name, pmid),
            source="pubmed",
            organization=self.organization,
            title=title,
            text=body_text,
            url=f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            year=year,
            topic=topic_key,
            document_type=document_type,
            license_note=self.license_note,
            sections=sections,
            extra={
                "pmid": pmid,
                "journal": journal,
                "doi": doi,
                "publication_types": pub_types,
                "keywords": keywords,
                "mesh_terms": mesh_terms,
            },
        )

    def _fetch(self) -> List[RawDocument]:
        documents: List[RawDocument] = []
        for topic_key in tqdm(TOPIC_KEYS, desc="PubMed topics", leave=False):
            pmids = self._search(topic_key)
            if not pmids:
                logger.warning("PubMed returned no PMIDs for topic '%s'", topic_key)
                continue
            batch = self.config.pubmed_batch_size
            for start in range(0, len(pmids), batch):
                xml_text = self._efetch(pmids[start : start + batch])
                if not xml_text:
                    continue
                try:
                    root = ET.fromstring(xml_text)
                except Exception as exc:
                    log_error("pubmed_xml", exc, f"topic={topic_key} batch={start}")
                    continue
                for article in root.findall(".//PubmedArticle"):
                    try:
                        doc = self._parse_article(article, topic_key)
                        if doc is not None:
                            documents.append(doc)
                    except Exception as exc:
                        log_error("pubmed_parse", exc, f"topic={topic_key}")
        return documents

In [7]:
# =============================================================================
# SECTION 4d — EUROPE PMC CONNECTOR (open-access full text)
# =============================================================================


class EuropePMCConnector(BaseConnector):
    """Retrieve *open-access* full-text psychiatry reviews from Europe PMC.

    Only records flagged `isOpenAccess=Y` with a permissive licence are ingested,
    which keeps redistribution of chunk text legally clean. Full text is parsed
    from JATS XML so that native `<sec>` structure becomes chunk metadata.
    """

    name = "europepmc"
    organization = "Europe PMC (EMBL-EBI)"
    license_note = "Open-access subset only (CC-BY / CC0 / CC-BY-NC where indicated)."

    ACCEPTED_LICENCES = ("cc by", "cc0", "cc-by", "ccby")

    def _search(self, topic_key: str) -> List[Dict[str, Any]]:
        meta = PSYCHIATRY_TOPICS[topic_key]
        query = (
            f'("{meta["keywords"][0]}") AND (PUB_TYPE:"Review" OR PUB_TYPE:"systematic review") '
            f"AND (OPEN_ACCESS:y) AND (FIRST_PDATE:[{self.config.pubmed_min_year} TO 3000]) AND (LANG:eng)"
        )
        payload = self.http.get_json(
            f"{self.config.europepmc_base}/search",
            params={
                "query": query,
                "format": "json",
                "resultType": "core",
                "pageSize": self.config.europepmc_max_per_topic * 2,
            },
        )
        if not payload:
            return []
        return list(payload.get("resultList", {}).get("result", []))

    def _fetch_full_text(self, pmcid: str) -> Optional[str]:
        cache_path = self.cache_file(f"{pmcid}.xml")
        if cache_path.exists() and not self.config.force_rebuild:
            return cache_path.read_text(encoding="utf-8", errors="ignore")
        xml_text = self.http.get_text(f"{self.config.europepmc_base}/{pmcid}/fullTextXML")
        if xml_text and "<" in xml_text:
            cache_path.write_text(xml_text, encoding="utf-8")
            return xml_text
        return None

    @staticmethod
    def _parse_jats_sections(xml_text: str) -> List[Dict[str, str]]:
        """Extract `<sec>` blocks from a JATS document as (heading, text) pairs."""
        sections: List[Dict[str, str]] = []
        try:
            root = ET.fromstring(xml_text)
        except Exception:
            return sections

        abstract = root.find(".//abstract")
        if abstract is not None:
            body = re.sub(r"\s+", " ", "".join(abstract.itertext())).strip()
            if body:
                sections.append({"heading": "Abstract", "text": body})

        body_node = root.find(".//body")
        if body_node is None:
            return sections
        for sec in body_node.iter("sec"):
            title_node = sec.find("title")
            heading = re.sub(r"\s+", " ", "".join(title_node.itertext())).strip() if title_node is not None else "Section"
            paragraphs = []
            for para in sec.findall("p"):
                text = re.sub(r"\s+", " ", "".join(para.itertext())).strip()
                if len(text) > 40:
                    paragraphs.append(text)
            if paragraphs:
                sections.append({"heading": heading or "Section", "text": "\n".join(paragraphs)})
        return sections

    def _fetch(self) -> List[RawDocument]:
        if not self.config.europepmc_enabled:
            return []
        documents: List[RawDocument] = []
        for topic_key in tqdm(TOPIC_KEYS, desc="Europe PMC topics", leave=False):
            kept = 0
            for record in self._search(topic_key):
                if kept >= self.config.europepmc_max_per_topic:
                    break
                pmcid = record.get("pmcid")
                licence = (record.get("license") or "").lower()
                if not pmcid or not any(token in licence for token in self.ACCEPTED_LICENCES):
                    continue
                xml_text = self._fetch_full_text(pmcid)
                if not xml_text:
                    continue
                sections = self._parse_jats_sections(xml_text)
                if not sections:
                    continue
                text = "\n\n".join(f"{s['heading']}\n{s['text']}" for s in sections)
                if len(text) < self.config.min_document_chars:
                    continue
                try:
                    year = int(str(record.get("pubYear"))[:4])
                except Exception:
                    year = None
                documents.append(
                    RawDocument(
                        doc_id=make_doc_id(self.name, pmcid),
                        source="europepmc",
                        organization=self.organization,
                        title=record.get("title", "").strip() or pmcid,
                        text=text,
                        url=f"https://europepmc.org/article/PMC/{pmcid}",
                        year=year,
                        topic=topic_key,
                        document_type="open_access_review",
                        license_note=f"{self.license_note} Record licence: {licence or 'unspecified'}",
                        sections=sections,
                        extra={
                            "pmcid": pmcid,
                            "pmid": record.get("pmid"),
                            "journal": (record.get("journalInfo") or {}).get("journal", {}).get("title"),
                            "doi": record.get("doi"),
                            "license": licence,
                        },
                    )
                )
                kept += 1
        return documents

In [8]:
# =============================================================================
# SECTION 4e — MEDLINEPLUS (NLM) AND NIMH (NIH) CONNECTORS — PUBLIC DOMAIN
# =============================================================================


class MedlinePlusConnector(BaseConnector):
    """Consumer-level, public-domain topic summaries from NLM MedlinePlus.

    Uses the documented, key-free MedlinePlus Web Service. As a U.S. Government
    work this content is public domain, which makes it the safest substitution
    for guideline text that cannot be redistributed.
    """

    name = "medlineplus"
    organization = "U.S. National Library of Medicine (MedlinePlus)"
    license_note = "U.S. Government work — public domain."

    def _query_topic(self, term: str) -> Optional[str]:
        cache_path = self.cache_file(f"{re.sub(r'[^a-z]+', '_', term.lower())}.xml")
        if cache_path.exists() and not self.config.force_rebuild:
            return cache_path.read_text(encoding="utf-8", errors="ignore")
        xml_text = self.http.get_text(
            self.config.medlineplus_base,
            params={"db": "healthTopics", "term": term, "rettype": "brief", "retmax": 4},
        )
        if xml_text:
            cache_path.write_text(xml_text, encoding="utf-8")
        return xml_text

    def _fetch(self) -> List[RawDocument]:
        documents: List[RawDocument] = []
        for topic_key, meta in tqdm(PSYCHIATRY_TOPICS.items(), desc="MedlinePlus topics", leave=False):
            xml_text = self._query_topic(meta["medlineplus"])
            if not xml_text:
                continue
            try:
                root = ET.fromstring(xml_text)
            except Exception as exc:
                log_error("medlineplus_xml", exc, topic_key)
                continue
            for doc_node in root.findall(".//document"):
                fields = {
                    (node.get("name") or ""): "".join(node.itertext())
                    for node in doc_node.findall("content")
                }
                title = normalize_title(self.strip_html(fields.get("title", "")))
                summary = self.strip_html(fields.get("FullSummary", "")).strip()
                if not title or len(summary) < self.config.min_document_chars:
                    continue
                url = doc_node.get("url", "")
                # The search term is only a query hint — a search for 'depression' can
                # return the Bipolar Disorder topic page. Tag the topic from the
                # document itself, falling back to the query topic.
                resolved_topic = infer_topic_from_text(title, default=topic_key)
                documents.append(
                    RawDocument(
                        doc_id=make_doc_id(self.name, url or title),
                        source="medlineplus",
                        organization=self.organization,
                        title=title,
                        text=f"Overview\n{summary}",
                        url=url,
                        year=None,
                        topic=resolved_topic,
                        document_type="patient_education",
                        license_note=self.license_note,
                        sections=[{"heading": "Overview", "text": summary}],
                        extra={"alt_titles": self.strip_html(fields.get("altTitle", ""))},
                    )
                )
        return documents


class NIMHConnector(BaseConnector):
    """Disorder overviews from the U.S. National Institute of Mental Health.

    NIMH health-topic pages are public-domain U.S. Government works covering
    signs/symptoms, risk factors, treatments and clinical trials — exactly the
    section taxonomy this project chunks against.
    """

    name = "nimh"
    organization = "U.S. National Institute of Mental Health (NIH)"
    license_note = "U.S. Government work — public domain."
    BASE_URL = "https://www.nimh.nih.gov/health/topics"

    HEADING_TAGS = {"h2", "h3"}

    def _fetch_page(self, slug: str) -> Optional[str]:
        cache_path = self.cache_file(f"{slug}.html")
        if cache_path.exists() and not self.config.force_rebuild:
            return cache_path.read_text(encoding="utf-8", errors="ignore")
        page = self.http.get_text(f"{self.BASE_URL}/{slug}")
        if page and len(page) > 3000:
            cache_path.write_text(page, encoding="utf-8")
            return page
        return None

    def _parse_page(self, raw_html: str) -> List[Dict[str, str]]:
        """Split an NIMH page into (heading, text) sections."""
        try:
            from bs4 import BeautifulSoup

            soup = BeautifulSoup(raw_html, "lxml")
        except Exception:
            return []
        for tag in soup(["script", "style", "nav", "footer", "header", "form", "aside", "noscript"]):
            tag.decompose()
        main = soup.find("main") or soup.find("article") or soup.body
        if main is None:
            return []
        sections: List[Dict[str, str]] = []
        heading = "Overview"
        buffer: List[str] = []
        for element in main.find_all(["h2", "h3", "p", "li"]):
            text = re.sub(r"\s+", " ", element.get_text(" ")).strip()
            if not text:
                continue
            if element.name in self.HEADING_TAGS:
                if buffer:
                    sections.append({"heading": heading, "text": "\n".join(buffer)})
                    buffer = []
                heading = text[:120]
            elif len(text) > 30:
                buffer.append(text)
        if buffer:
            sections.append({"heading": heading, "text": "\n".join(buffer)})
        return [s for s in sections if len(s["text"]) > 120]

    def _fetch(self) -> List[RawDocument]:
        if not self.config.nimh_enabled:
            return []
        documents: List[RawDocument] = []
        for topic_key, meta in tqdm(PSYCHIATRY_TOPICS.items(), desc="NIMH topics", leave=False):
            slug = meta.get("nimh_slug")
            if not slug:
                continue
            page = self._fetch_page(slug)
            if not page:
                continue
            sections = self._parse_page(page)
            if not sections:
                continue
            text = "\n\n".join(f"{s['heading']}\n{s['text']}" for s in sections)
            if len(text) < self.config.min_document_chars:
                continue
            documents.append(
                RawDocument(
                    doc_id=make_doc_id(self.name, slug),
                    source="nimh",
                    organization=self.organization,
                    title=f"NIMH Health Topic: {meta['label']}",
                    text=text,
                    url=f"{self.BASE_URL}/{slug}",
                    year=None,
                    topic=topic_key,
                    document_type="clinical_overview",
                    license_note=self.license_note,
                    sections=sections,
                    extra={"slug": slug},
                )
            )
        return documents

In [9]:
# =============================================================================
# SECTION 4f — WHO CONNECTOR (mhGAP + reports) AND ICD-11 CONNECTOR
# =============================================================================


class WHOConnector(BaseConnector):
    """Best-effort ingestion of WHO mental-health publications (PDF).

    WHO IRIS download URLs are periodically reorganised, so each publication has
    a list of candidate URLs tried in order. If every candidate fails, the
    connector emits a *reference-only* record (metadata + canonical URL) and logs
    the failure — the pipeline continues with the remaining sources, exactly as
    the specification's substitution clause requires.
    """

    name = "who"
    organization = "World Health Organization"
    license_note = "WHO publications: CC BY-NC-SA 3.0 IGO."

    PUBLICATIONS: List[Dict[str, Any]] = [
        {
            "key": "mhgap_ig_v2",
            "title": "mhGAP Intervention Guide for mental, neurological and substance use disorders in non-specialized health settings (version 2.0)",
            "year": 2016,
            "topic": "general",
            "landing": "https://www.who.int/publications/i/item/9789241549790",
            "candidates": [
                "https://iris.who.int/bitstream/handle/10665/250239/9789241549790-eng.pdf",
                "https://apps.who.int/iris/bitstream/handle/10665/250239/9789241549790-eng.pdf",
            ],
        },
        {
            "key": "mhgap_guideline_2023",
            "title": "mhGAP guideline for mental, neurological and substance use disorders (2023 update)",
            "year": 2023,
            "topic": "general",
            "landing": "https://www.who.int/publications/i/item/9789240084278",
            "candidates": [
                "https://iris.who.int/bitstream/handle/10665/374250/9789240084278-eng.pdf",
            ],
        },
        {
            "key": "world_mental_health_report_2022",
            "title": "World mental health report: transforming mental health for all",
            "year": 2022,
            "topic": "general",
            "landing": "https://www.who.int/publications/i/item/9789240049338",
            "candidates": [
                "https://iris.who.int/bitstream/handle/10665/356119/9789240049338-eng.pdf",
            ],
        },
        {
            "key": "live_life_suicide_2021",
            "title": "LIVE LIFE: an implementation guide for suicide prevention in countries",
            "year": 2021,
            "topic": "suicide_prevention",
            "landing": "https://www.who.int/publications/i/item/9789240026629",
            "candidates": [
                "https://iris.who.int/bitstream/handle/10665/341726/9789240026629-eng.pdf",
            ],
        },
    ]

    def _discover_pdf_urls(self, landing: str) -> List[str]:
        """Scrape a WHO landing page for live PDF links.

        Hard-coded IRIS bitstream URLs rot quickly (all four failed on the first
        run of this notebook). Reading the landing page recovers whatever the
        current download URL happens to be, keeping ingestion automated.
        """
        page = self.http.get_text(landing)
        if not page:
            return []
        discovered: List[str] = []
        for match in re.finditer(r'href=["\']([^"\']+)["\']', page, re.IGNORECASE):
            href = html.unescape(match.group(1)).strip()
            lowered = href.lower()
            if ".pdf" not in lowered and "/bitstream/" not in lowered:
                continue
            if href.startswith("//"):
                href = "https:" + href
            elif href.startswith("/"):
                href = ("https://iris.who.int" if "/bitstream/" in lowered else "https://www.who.int") + href
            if href.startswith("http") and href not in discovered:
                discovered.append(href)
        # Prefer English-language files, then shorter (canonical) URLs.
        discovered.sort(key=lambda url: (0 if "eng" in url.lower() else 1, len(url)))
        return discovered[:6]

    def _extract_pdf_text(self, pdf_path: Path) -> Optional[str]:
        try:
            from pypdf import PdfReader

            reader = PdfReader(str(pdf_path))
            pages: List[str] = []
            for page in reader.pages:
                try:
                    pages.append(page.extract_text() or "")
                except Exception:
                    pages.append("")
            # Page markers survive into cleaning, where header/footer detection uses them.
            return "\n<<<PAGE>>>\n".join(pages)
        except Exception as exc:
            log_error("pdf_extract", exc, str(pdf_path))
            return None

    def _fetch(self) -> List[RawDocument]:
        if not self.config.who_enabled:
            return []
        documents: List[RawDocument] = []
        for publication in tqdm(self.PUBLICATIONS, desc="WHO publications", leave=False):
            pdf_path = self.cache_file(f"{publication['key']}.pdf")
            downloaded: Optional[Path] = None
            candidates = list(publication["candidates"])
            for candidate in candidates:
                downloaded = self.http.download(candidate, pdf_path, min_bytes=50_000)
                if downloaded is not None:
                    break
            if downloaded is None:
                # Fall back to whatever the landing page currently links to.
                for candidate in self._discover_pdf_urls(publication["landing"]):
                    downloaded = self.http.download(candidate, pdf_path, min_bytes=50_000)
                    if downloaded is not None:
                        logger.info("WHO: recovered '%s' via landing-page discovery", publication["key"])
                        break
            if downloaded is None:
                logger.warning("WHO download failed for '%s' — emitting reference-only record", publication["key"])
                documents.append(
                    RawDocument(
                        doc_id=make_doc_id(self.name, publication["key"]),
                        source="who_reference",
                        organization=self.organization,
                        title=publication["title"],
                        text=(
                            f"Reference record. {publication['title']} (World Health Organization, "
                            f"{publication['year']}). The full text of this publication could not be "
                            f"downloaded automatically during this run; consult the official record at "
                            f"{publication['landing']}."
                        ),
                        url=publication["landing"],
                        year=publication["year"],
                        topic=publication["topic"],
                        document_type="guideline_reference",
                        license_note=self.license_note,
                        sections=[],
                        extra={"ingested_full_text": False},
                    )
                )
                continue

            text = self._extract_pdf_text(downloaded)
            if not text or len(text) < 5000:
                log_error("who_pdf_empty", ValueError("insufficient extracted text"), publication["key"])
                continue
            documents.append(
                RawDocument(
                    doc_id=make_doc_id(self.name, publication["key"]),
                    source="who_mhgap" if "mhgap" in publication["key"] else "who_report",
                    organization=self.organization,
                    title=publication["title"],
                    text=text,
                    url=publication["landing"],
                    year=publication["year"],
                    topic=publication["topic"],
                    document_type="guideline",
                    license_note=self.license_note,
                    sections=[],
                    extra={"ingested_full_text": True, "pdf_bytes": downloaded.stat().st_size},
                )
            )
        return documents


class ICD11Connector(BaseConnector):
    """ICD-11 Chapter 06 (mental, behavioural and neurodevelopmental disorders).

    Two modes
    ---------
    1. **API mode** — if `ICD_CLIENT_ID` / `ICD_CLIENT_SECRET` are present in the
       environment, the official WHO ICD API is used to walk the Chapter 06
       hierarchy and extract titles, definitions, inclusions and exclusions.
    2. **Scaffold mode (default)** — WHO's API mandates OAuth credentials, which
       conflicts with the "no API keys" requirement, so the notebook falls back to
       a bundled scaffold of Chapter 06 codes, titles and hierarchy plus short
       author-written descriptors. Authoritative diagnostic detail is then supplied
       by the public-domain NIMH/MedlinePlus records and the PubMed literature.
       This substitution is deliberate and documented in Section 1.3.
    """

    name = "icd11"
    organization = "World Health Organization (ICD-11)"
    license_note = "ICD-11 codes/titles used under WHO terms; descriptors are author-written."
    TOKEN_URL = "https://icdaccessmanagement.who.int/connect/token"
    ROOT_URI = "https://id.who.int/icd/release/11/2024-01/mms"
    CHAPTER_06_CODE = "06"

    # (code, title, grouping, topic, author-written descriptor)
    SCAFFOLD: List[Tuple[str, str, str, str, str]] = [
        ("6A00", "Disorders of intellectual development", "Neurodevelopmental disorders", "autism_neurodevelopment",
         "A grouping characterised by significantly below-average intellectual functioning and adaptive behaviour with onset during the developmental period."),
        ("6A01", "Developmental speech or language disorders", "Neurodevelopmental disorders", "autism_neurodevelopment",
         "Disorders in which the acquisition of speech or language is impaired relative to developmental expectations and is not better explained by another condition."),
        ("6A02", "Autism spectrum disorder", "Neurodevelopmental disorders", "autism_neurodevelopment",
         "Persistent deficits in reciprocal social communication and interaction alongside restricted, repetitive and inflexible patterns of behaviour and interests, with onset in the developmental period."),
        ("6A03", "Developmental learning disorder", "Neurodevelopmental disorders", "autism_neurodevelopment",
         "Significant and persistent difficulties acquiring academic skills such as reading, writing or arithmetic that are markedly below expectation for age."),
        ("6A05", "Attention deficit hyperactivity disorder", "Neurodevelopmental disorders", "adhd",
         "A persistent pattern of inattention and/or hyperactivity-impulsivity beginning in the developmental period that directly interferes with functioning across settings."),
        ("6A06", "Stereotyped movement disorder", "Neurodevelopmental disorders", "autism_neurodevelopment",
         "Voluntary, repetitive, non-functional movements that interfere with activities or cause self-injury."),
        ("6A20", "Schizophrenia", "Schizophrenia or other primary psychotic disorders", "schizophrenia",
         "Disturbances in multiple mental modalities including thinking, perception, self-experience, volition, affect and behaviour, with persistent positive and/or negative symptoms."),
        ("6A21", "Schizoaffective disorder", "Schizophrenia or other primary psychotic disorders", "schizophrenia",
         "Symptoms meeting the requirements for schizophrenia occur concurrently with a moderate or severe mood episode within the same episode of illness."),
        ("6A23", "Acute and transient psychotic disorder", "Schizophrenia or other primary psychotic disorders", "schizophrenia",
         "Acute onset of psychotic symptoms that fluctuate rapidly in type and intensity and remit within a short period."),
        ("6A24", "Delusional disorder", "Schizophrenia or other primary psychotic disorders", "schizophrenia",
         "Development of a delusion or set of delusions persisting for at least three months in the absence of other prominent psychotic features."),
        ("6A60", "Bipolar type I disorder", "Mood disorders", "bipolar_disorder",
         "An episodic mood disorder defined by the occurrence of at least one manic or mixed episode, typically alternating with depressive episodes."),
        ("6A61", "Bipolar type II disorder", "Mood disorders", "bipolar_disorder",
         "An episodic mood disorder defined by one or more hypomanic episodes and at least one depressive episode, without any history of a manic episode."),
        ("6A62", "Cyclothymic disorder", "Mood disorders", "bipolar_disorder",
         "A persistent instability of mood over at least two years involving numerous hypomanic and depressive periods that do not meet full episode requirements."),
        ("6A70", "Single episode depressive disorder", "Mood disorders", "depression",
         "A single depressive episode with depressed mood or diminished interest accompanied by additional cognitive, behavioural or neurovegetative symptoms, and no history of manic or hypomanic episodes."),
        ("6A71", "Recurrent depressive disorder", "Mood disorders", "depression",
         "A history of at least two depressive episodes separated by several months without significant mood disturbance, and no history of manic or hypomanic episodes."),
        ("6A72", "Dysthymic disorder", "Mood disorders", "depression",
         "Persistent depressed mood lasting two years or more that does not meet the requirements for a depressive episode for most of that period."),
        ("6B00", "Generalised anxiety disorder", "Anxiety or fear-related disorders", "anxiety",
         "Marked, persistent apprehension or worry not restricted to particular stimuli, accompanied by symptoms such as muscle tension, autonomic arousal, restlessness or sleep disturbance."),
        ("6B01", "Panic disorder", "Anxiety or fear-related disorders", "anxiety",
         "Recurrent unexpected panic attacks accompanied by persistent concern about recurrence or significant maladaptive behaviour change."),
        ("6B02", "Agoraphobia", "Anxiety or fear-related disorders", "anxiety",
         "Marked and excessive fear or anxiety occurring in, or in anticipation of, multiple situations where escape might be difficult or help unavailable."),
        ("6B04", "Social anxiety disorder", "Anxiety or fear-related disorders", "anxiety",
         "Marked and excessive fear or anxiety consistently triggered by social situations in which the person may be scrutinised by others."),
        ("6B20", "Obsessive-compulsive disorder", "Obsessive-compulsive or related disorders", "ocd",
         "Persistent obsessions and/or compulsions that are time-consuming or result in significant distress or impairment in important areas of functioning."),
        ("6B21", "Body dysmorphic disorder", "Obsessive-compulsive or related disorders", "ocd",
         "Persistent preoccupation with perceived defects in appearance that are unnoticeable or only slightly noticeable to others, with repetitive behaviours."),
        ("6B40", "Post traumatic stress disorder", "Disorders specifically associated with stress", "ptsd",
         "Develops after exposure to an extremely threatening or horrific event and is characterised by re-experiencing, avoidance and a persistent sense of current threat."),
        ("6B41", "Complex post traumatic stress disorder", "Disorders specifically associated with stress", "ptsd",
         "Follows prolonged or repeated traumatic exposure and adds severe problems in affect regulation, self-concept and relationships to the core PTSD features."),
        ("6B43", "Adjustment disorder", "Disorders specifically associated with stress", "ptsd",
         "A maladaptive reaction to an identifiable psychosocial stressor characterised by preoccupation with the stressor and failure to adapt."),
        ("6B80", "Anorexia nervosa", "Feeding or eating disorders", "eating_disorders",
         "Significantly low body weight for height and age arising from persistent restriction of energy intake, accompanied by fear of weight gain and body image disturbance."),
        ("6B81", "Bulimia nervosa", "Feeding or eating disorders", "eating_disorders",
         "Recurrent binge eating with repeated inappropriate compensatory behaviours aimed at preventing weight gain."),
        ("6B82", "Binge eating disorder", "Feeding or eating disorders", "eating_disorders",
         "Recurrent episodes of binge eating with a sense of loss of control, without regular compensatory behaviours."),
        ("6C40", "Disorders due to use of alcohol", "Disorders due to substance use", "substance_use_disorders",
         "A grouping covering episodes of harmful use, harmful patterns of use, and alcohol dependence together with intoxication and withdrawal states."),
        ("6C43", "Disorders due to use of opioids", "Disorders due to substance use", "substance_use_disorders",
         "A grouping covering harmful opioid use, opioid dependence, intoxication and withdrawal."),
        ("6D10", "Personality disorder", "Personality disorders and related traits", "personality_disorders",
         "Persistent disturbance in self-functioning and interpersonal functioning, graded by severity as mild, moderate or severe, with trait domain qualifiers."),
        ("6D11", "Prominent personality traits or patterns", "Personality disorders and related traits", "personality_disorders",
         "Trait domain qualifiers — negative affectivity, detachment, dissociality, disinhibition and anankastia — used to describe personality disturbance."),
    ]

    def _api_credentials(self) -> Optional[Tuple[str, str]]:
        client_id = os.environ.get(self.config.icd11_client_id_env)
        client_secret = os.environ.get(self.config.icd11_client_secret_env)
        if client_id and client_secret:
            return client_id, client_secret
        return None

    def _api_documents(self, credentials: Tuple[str, str]) -> List[RawDocument]:
        """Walk the Chapter 06 hierarchy using the official WHO ICD API."""
        client_id, client_secret = credentials
        token_response = self.http.session.post(
            self.TOKEN_URL,
            data={
                "client_id": client_id,
                "client_secret": client_secret,
                "scope": "icdapi_access",
                "grant_type": "client_credentials",
            },
            timeout=self.config.http_timeout,
        )
        token = token_response.json().get("access_token")
        if not token:
            raise RuntimeError("ICD-11 token request returned no access token")
        headers = {
            "Authorization": f"Bearer {token}",
            "Accept": "application/json",
            "Accept-Language": "en",
            "API-Version": "v2",
        }

        def get_entity(uri: str) -> Optional[Dict[str, Any]]:
            response = self.http.session.get(uri, headers=headers, timeout=self.config.http_timeout)
            return response.json() if response.status_code == 200 else None

        chapter = get_entity(f"{self.ROOT_URI}/{self.CHAPTER_06_CODE}")
        if chapter is None:
            raise RuntimeError("Could not read ICD-11 Chapter 06 root entity")

        documents: List[RawDocument] = []
        queue: List[Tuple[str, int]] = [(child, 0) for child in chapter.get("child", [])]
        visited: Set[str] = set()
        with tqdm(total=260, desc="ICD-11 API walk", leave=False) as progress:
            while queue and len(visited) < 260:
                uri, depth = queue.pop(0)
                if uri in visited:
                    continue
                visited.add(uri)
                progress.update(1)
                entity = get_entity(uri)
                if entity is None:
                    continue
                title = (entity.get("title") or {}).get("@value", "").strip()
                definition = (entity.get("definition") or {}).get("@value", "").strip()
                code = entity.get("code", "")
                inclusions = [i.get("label", {}).get("@value", "") for i in entity.get("inclusion", [])]
                exclusions = [e.get("label", {}).get("@value", "") for e in entity.get("exclusion", [])]
                if depth < 3:
                    queue.extend((child, depth + 1) for child in entity.get("child", []))
                if not title or not definition:
                    continue
                sections = [{"heading": "Diagnostic description", "text": definition}]
                if inclusions:
                    sections.append({"heading": "Inclusions", "text": "; ".join(filter(None, inclusions))})
                if exclusions:
                    sections.append({"heading": "Exclusions", "text": "; ".join(filter(None, exclusions))})
                documents.append(
                    RawDocument(
                        doc_id=make_doc_id(self.name, uri),
                        source="icd11",
                        organization=self.organization,
                        title=f"ICD-11 {code} {title}".strip(),
                        text="\n\n".join(f"{s['heading']}\n{s['text']}" for s in sections),
                        url=uri,
                        year=2024,
                        topic=self._infer_topic(title),
                        document_type="classification",
                        license_note="Retrieved from the official WHO ICD-11 API.",
                        sections=sections,
                        extra={"icd_code": code, "mode": "api", "depth": depth},
                    )
                )
        return documents

    @staticmethod
    def _infer_topic(title: str) -> str:
        return infer_topic_from_text(title)

    def _scaffold_documents(self) -> List[RawDocument]:
        documents: List[RawDocument] = []
        for code, title, grouping, topic, descriptor in self.SCAFFOLD:
            sections = [
                {"heading": "Coding information", "text": f"ICD-11 code {code}. Chapter 06 — Mental, behavioural or neurodevelopmental disorders. Grouping: {grouping}."},
                {"heading": "Diagnostic description", "text": descriptor},
                {
                    "heading": "Hierarchical category",
                    "text": f"{title} is classified within the ICD-11 grouping '{grouping}' in Chapter 06.",
                },
            ]
            documents.append(
                RawDocument(
                    doc_id=make_doc_id(self.name, code),
                    source="icd11",
                    organization=self.organization,
                    title=f"ICD-11 {code} {title}",
                    text="\n\n".join(f"{s['heading']}\n{s['text']}" for s in sections),
                    url=f"https://icd.who.int/browse/2024-01/mms/en#/{code}",
                    year=2024,
                    topic=topic,
                    document_type="classification",
                    license_note=self.license_note,
                    sections=sections,
                    extra={"icd_code": code, "grouping": grouping, "mode": "scaffold"},
                )
            )
        return documents

    def _fetch(self) -> List[RawDocument]:
        credentials = self._api_credentials()
        if credentials:
            try:
                documents = self._api_documents(credentials)
                if documents:
                    logger.info("ICD-11: retrieved %d entities from the official WHO API.", len(documents))
                    return documents
            except Exception as exc:
                log_error("icd11_api", exc, "falling back to bundled Chapter 06 scaffold")
        logger.info("ICD-11: using bundled Chapter 06 scaffold (documented substitution — see Section 1.3).")
        return self._scaffold_documents()


class GuidelineReferenceConnector(BaseConnector):
    """Reference-only records for guidelines that cannot be openly redistributed.

    NICE, APA and VA/DoD guidance is authoritative but not openly relicensable.
    Rather than ingest protected text, the system indexes a metadata stub so the
    assistant can *point users to* the canonical guideline without reproducing it.
    """

    name = "guideline_references"
    organization = "Multiple guideline developers"
    license_note = "Metadata and canonical URLs only — full text deliberately not ingested."

    RECORDS: List[Dict[str, Any]] = [
        {"org": "NICE", "title": "Depression in adults: treatment and management (NG222)", "year": 2022,
         "topic": "depression", "url": "https://www.nice.org.uk/guidance/ng222"},
        {"org": "NICE", "title": "Generalised anxiety disorder and panic disorder in adults: management (CG113)", "year": 2020,
         "topic": "anxiety", "url": "https://www.nice.org.uk/guidance/cg113"},
        {"org": "NICE", "title": "Bipolar disorder: assessment and management (CG185)", "year": 2023,
         "topic": "bipolar_disorder", "url": "https://www.nice.org.uk/guidance/cg185"},
        {"org": "NICE", "title": "Psychosis and schizophrenia in adults: prevention and management (CG178)", "year": 2014,
         "topic": "schizophrenia", "url": "https://www.nice.org.uk/guidance/cg178"},
        {"org": "NICE", "title": "Post-traumatic stress disorder (NG116)", "year": 2018,
         "topic": "ptsd", "url": "https://www.nice.org.uk/guidance/ng116"},
        {"org": "NICE", "title": "Obsessive-compulsive disorder and body dysmorphic disorder: treatment (CG31)", "year": 2005,
         "topic": "ocd", "url": "https://www.nice.org.uk/guidance/cg31"},
        {"org": "NICE", "title": "Attention deficit hyperactivity disorder: diagnosis and management (NG87)", "year": 2019,
         "topic": "adhd", "url": "https://www.nice.org.uk/guidance/ng87"},
        {"org": "NICE", "title": "Eating disorders: recognition and treatment (NG69)", "year": 2020,
         "topic": "eating_disorders", "url": "https://www.nice.org.uk/guidance/ng69"},
        {"org": "NICE", "title": "Self-harm: assessment, management and preventing recurrence (NG225)", "year": 2022,
         "topic": "suicide_prevention", "url": "https://www.nice.org.uk/guidance/ng225"},
        {"org": "American Psychiatric Association", "title": "Practice Guideline for the Treatment of Patients With Schizophrenia", "year": 2021,
         "topic": "schizophrenia", "url": "https://psychiatryonline.org/guidelines"},
        {"org": "American Psychiatric Association", "title": "Practice Guideline for the Treatment of Patients With Eating Disorders", "year": 2023,
         "topic": "eating_disorders", "url": "https://psychiatryonline.org/guidelines"},
        {"org": "VA/DoD", "title": "Clinical Practice Guideline for the Management of Posttraumatic Stress Disorder and Acute Stress Disorder", "year": 2023,
         "topic": "ptsd", "url": "https://www.healthquality.va.gov/guidelines/MH/ptsd/"},
        {"org": "VA/DoD", "title": "Clinical Practice Guideline for the Assessment and Management of Patients at Risk for Suicide", "year": 2024,
         "topic": "suicide_prevention", "url": "https://www.healthquality.va.gov/guidelines/MH/srb/"},
        {"org": "SAMHSA", "title": "Medications for Opioid Use Disorder (TIP 63)", "year": 2021,
         "topic": "substance_use_disorders", "url": "https://store.samhsa.gov/product/TIP-63-Medications-for-Opioid-Use-Disorder-Full-Document/PEP21-02-01-002"},
    ]

    def _fetch(self) -> List[RawDocument]:
        documents: List[RawDocument] = []
        for record in self.RECORDS:
            text = (
                f"Guideline reference record. '{record['title']}' was published by {record['org']} in "
                f"{record['year']} and covers {PSYCHIATRY_TOPICS.get(record['topic'], {}).get('label', record['topic'])}. "
                f"The full text is not reproduced in this corpus for licensing reasons; the authoritative version "
                f"is available at {record['url']}. Use this record to direct users to the primary guideline."
            )
            documents.append(
                RawDocument(
                    doc_id=make_doc_id(self.name, record["url"] + record["title"]),
                    source="guideline_reference",
                    organization=record["org"],
                    title=record["title"],
                    text=text,
                    url=record["url"],
                    year=record["year"],
                    topic=record["topic"],
                    document_type="guideline_reference",
                    license_note=self.license_note,
                    sections=[{"heading": "Reference", "text": text}],
                    extra={"ingested_full_text": False},
                )
            )
        return documents

In [10]:
# =============================================================================
# SECTION 4g — INGESTION ORCHESTRATOR
# =============================================================================


class IngestionPipeline:
    """Runs every connector, isolates failures, and caches the merged raw corpus."""

    def __init__(self, config: RAGConfig, http: HttpClient, paths: ProjectPaths, cache: CacheManager) -> None:
        self.config = config
        self.paths = paths
        self.cache = cache
        self.connectors: List[BaseConnector] = [
            PubMedConnector(config, http, paths),
            EuropePMCConnector(config, http, paths),
            MedlinePlusConnector(config, http, paths),
            NIMHConnector(config, http, paths),
            WHOConnector(config, http, paths),
            ICD11Connector(config, http, paths),
            GuidelineReferenceConnector(config, http, paths),
        ]
        self.connector_stats: List[Dict[str, Any]] = []

    def _build(self) -> List[Dict[str, Any]]:
        documents: List[RawDocument] = []
        for connector in self.connectors:
            documents.extend(connector.fetch())
            self.connector_stats.append(connector.stats)
        # Deduplicate by doc_id (a topic-overlapping PMID can be returned twice).
        unique: Dict[str, RawDocument] = {}
        for document in documents:
            if document.doc_id not in unique:
                unique[document.doc_id] = document
        return [doc.to_dict() for doc in unique.values()]

    def run(self) -> List[RawDocument]:
        payload = self.cache.get_or_build(
            self.paths.raw / f"raw_corpus_{self.config.pipeline_version}.json",
            self._build,
            fmt="json",
            label=f"raw_corpus_{self.config.pipeline_version}.json",
        )
        documents = [RawDocument(**record) for record in payload]
        if self.connector_stats:
            stats_path = self.paths.logs / "connector_stats.json"
            stats_path.write_text(json.dumps(self.connector_stats, indent=2), encoding="utf-8")
        return documents


set_global_seeds(CONFIG.seed)
INGESTION = IngestionPipeline(CONFIG, HTTP, PATHS, CACHE)
RAW_DOCUMENTS: List[RawDocument] = INGESTION.run()

raw_frame = pd.DataFrame(
    [
        {
            "doc_id": d.doc_id,
            "source": d.source,
            "organization": d.organization,
            "topic": d.topic,
            "document_type": d.document_type,
            "year": d.year,
            "chars": d.n_chars,
        }
        for d in RAW_DOCUMENTS
    ]
)

print("=" * 78)
print(f"RAW CORPUS: {len(RAW_DOCUMENTS)} documents, {raw_frame['chars'].sum():,} characters")
print("=" * 78)
if not raw_frame.empty:
    summary = (
        raw_frame.groupby("source")
        .agg(documents=("doc_id", "count"), characters=("chars", "sum"), mean_chars=("chars", "mean"))
        .sort_values("documents", ascending=False)
        .round(0)
        .astype({"mean_chars": int})
    )
    print(summary.to_string())
    print("\nDocuments per topic:")
    print(raw_frame["topic"].value_counts().to_string())
if ERROR_LOG:
    print(f"\n{len(ERROR_LOG)} non-fatal error(s) logged during ingestion (see the System Information tab).")

18:56:50 | INFO    | psych-rag | cache HIT  → raw_corpus_v3.json


RAW CORPUS: 888 documents, 5,598,379 characters
                     documents  characters  mean_chars
source                                                
pubmed                     703     1249279        1777
europepmc                   89     3926910       44123
medlineplus                 34      113809        3347
icd11                       32       14625         457
guideline_reference         14        5453         390
nimh                        12       51208        4267
who_reference                3         898         299
who_mhgap                    1      236197      236197

Documents per topic:
topic
anxiety                    77
ptsd                       77
depression                 76
schizophrenia              76
bipolar_disorder           75
eating_disorders           74
autism_neurodevelopment    74
substance_use_disorders    73
personality_disorders      72
adhd                       71
ocd                        70
suicide_prevention         70
general       

---
## 5. Data Cleaning & Normalisation

PDF and HTML extractions are noisy in characteristic ways. `TextCleaner` targets each failure mode explicitly
rather than applying one blunt regex:

| Artefact | Strategy |
|---|---|
| Running headers / footers | Frequency analysis across page-delimited blocks — a short line repeating on ≥30% of pages is a header |
| Page numbers | Numeric-only lines and `Page n of m` patterns |
| Table of contents | Dot-leader detection (`Introduction .......... 12`) plus a bounded TOC region scan |
| Hyphenation across line breaks | `treat-\nment → treatment` |
| Ligatures / smart quotes / NBSP | Unicode NFKC normalisation plus an explicit replacement map |
| Navigation & boilerplate | Phrase blacklist (cookie notices, "skip to main content", social share text) |
| Duplicate whitespace | Collapse runs, but preserve paragraph boundaries (`\n\n`) |
| Duplicate documents | Exact SHA-256 match **plus** near-duplicate detection via MinHash over 5-gram shingles |

In [11]:
# =============================================================================
# SECTION 5 — CLEANING, NORMALISATION, DEDUPLICATION
# =============================================================================


class TextCleaner:
    """Deterministic cleaning of noisy PDF/HTML text extractions."""

    UNICODE_MAP = {
        "\u00a0": " ", "\u2009": " ", "\u202f": " ", "\u200b": "", "\ufeff": "",
        "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-", "\u2212": "-",
        "\ufb01": "fi", "\ufb02": "fl", "\u2026": "...", "\u00ad": "",
        "\u2022": "- ", "\u00b7": "- ",
    }

    NAVIGATION_PATTERNS = [
        r"skip to (main )?content", r"cookie(s)? (policy|settings|notice)",
        r"all rights reserved", r"terms of use", r"privacy policy",
        r"share (this )?(page|on (twitter|facebook|linkedin))",
        r"click here to", r"subscribe to our newsletter", r"print this page",
        r"was this page helpful", r"back to top", r"follow us on",
        r"download the pdf", r"javascript (is|must be) (disabled|enabled)",
        r"^\s*(home|menu|search|login|sign in)\s*$",
    ]

    PAGE_NUMBER_PATTERNS = [
        r"^\s*[-–—]?\s*\d{1,4}\s*[-–—]?\s*$",
        r"^\s*page\s+\d+\s*(of\s+\d+)?\s*$",
        r"^\s*\d+\s*\|\s*.{0,60}$",
    ]

    TOC_LINE = re.compile(r"^.{3,90}?[\.\u2026]{4,}\s*\d{1,4}\s*$")
    TOC_ENTRY = re.compile(r"^.{2,80}?\s+\d{1,4}\s*$")
    PAGE_DELIMITER = "<<<PAGE>>>"

    def __init__(self, config: RAGConfig) -> None:
        self.config = config
        self._nav_regex = re.compile("|".join(self.NAVIGATION_PATTERNS), re.IGNORECASE)
        self._page_regexes = [re.compile(p, re.IGNORECASE) for p in self.PAGE_NUMBER_PATTERNS]

    # -- primitive steps ------------------------------------------------------
    def normalize_unicode(self, text: str) -> str:
        """NFKC-normalise and fold typographic characters into ASCII equivalents."""
        text = unicodedata.normalize("NFKC", text)
        for source, target in self.UNICODE_MAP.items():
            text = text.replace(source, target)
        # Strip control characters except tab/newline.
        return "".join(ch for ch in text if ch == "\n" or ch == "\t" or unicodedata.category(ch)[0] != "C")

    def fix_hyphenation(self, text: str) -> str:
        """Re-join words split across a line break by end-of-line hyphenation."""
        return re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    def detect_running_headers(self, pages: Sequence[str], min_fraction: float = 0.30) -> Set[str]:
        """Identify short lines that repeat across many pages (headers/footers)."""
        if len(pages) < 4:
            return set()
        counter: Counter = Counter()
        for page in pages:
            lines = [ln.strip() for ln in page.split("\n") if ln.strip()]
            for line in lines[:3] + lines[-3:]:          # only page margins
                if 3 <= len(line) <= 90:
                    counter[re.sub(r"\d+", "#", line.lower())] += 1
        threshold = max(3, int(len(pages) * min_fraction))
        return {key for key, count in counter.items() if count >= threshold}

    def _is_toc_continuation(self, stripped: str) -> bool:
        """True for lines that plausibly belong to a table-of-contents block.

        Deliberately conservative: the TOC region ends at the first line that does
        not look like an entry, so ordinary prose is never swallowed.
        """
        if not stripped:
            return True
        if self.TOC_LINE.match(stripped):
            return True
        if any(regex.match(stripped) for regex in self._page_regexes):
            return True
        return bool(self.TOC_ENTRY.match(stripped)) and len(stripped) <= 90

    def strip_table_of_contents(self, text: str) -> str:
        """Remove dot-leader TOC lines and an explicit 'Contents' block near the start."""
        lines = text.split("\n")
        keep: List[str] = []
        in_toc = False
        toc_budget = 0
        for index, line in enumerate(lines):
            stripped = line.strip()
            if not in_toc and index < 400 and re.fullmatch(r"(table of )?contents", stripped, re.IGNORECASE):
                in_toc, toc_budget = True, 80
                continue
            if in_toc:
                toc_budget -= 1
                if toc_budget > 0 and self._is_toc_continuation(stripped):
                    continue
                in_toc = False       # first non-entry line ends the TOC region
            if self.TOC_LINE.match(stripped):
                continue
            keep.append(line)
        return "\n".join(keep)

    def _is_noise_line(self, line: str, header_keys: Set[str]) -> bool:
        stripped = line.strip()
        if not stripped:
            return False                                   # blank lines carry paragraph structure
        if any(regex.match(stripped) for regex in self._page_regexes):
            return True
        if self._nav_regex.search(stripped) and len(stripped) < 140:
            return True
        if re.sub(r"\d+", "#", stripped.lower()) in header_keys:
            return True
        return False

    def collapse_whitespace(self, text: str) -> str:
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r" *\n *", "\n", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    # -- public API -----------------------------------------------------------
    def clean(self, text: str) -> str:
        """Full cleaning pipeline for one document."""
        if not text:
            return ""
        text = self.normalize_unicode(text)
        pages = text.split(self.PAGE_DELIMITER)
        header_keys = self.detect_running_headers(pages) if len(pages) > 1 else set()
        text = "\n".join(pages)
        text = self.fix_hyphenation(text)
        text = self.strip_table_of_contents(text)
        kept = [line for line in text.split("\n") if not self._is_noise_line(line, header_keys)]
        return self.collapse_whitespace("\n".join(kept))

    def clean_document(self, document: RawDocument) -> RawDocument:
        """Clean a document's body *and* its section-level text in place."""
        document.text = self.clean(document.text)
        cleaned_sections: List[Dict[str, str]] = []
        for section in document.sections or []:
            body = self.clean(section.get("text", ""))
            if len(body) >= 60:
                cleaned_sections.append({"heading": section.get("heading", "Section").strip()[:140], "text": body})
        document.sections = cleaned_sections
        return document


class Deduplicator:
    """Exact and near-duplicate document detection.

    Near-duplicate detection uses a compact MinHash over 5-gram word shingles;
    with 128 permutations the estimated Jaccard similarity is accurate to a few
    percent, which is more than sufficient for corpus hygiene and far cheaper
    than all-pairs comparison of full texts.
    """

    def __init__(self, threshold: float = 0.90, num_perm: int = 128, shingle_size: int = 5) -> None:
        self.threshold = threshold
        self.num_perm = num_perm
        self.shingle_size = shingle_size
        rng = np.random.RandomState(GLOBAL_SEED)
        self._a = rng.randint(1, 2**31 - 1, size=num_perm).astype(np.int64)
        self._b = rng.randint(0, 2**31 - 1, size=num_perm).astype(np.int64)
        self._prime = np.int64(2**31 - 1)

    @staticmethod
    def exact_hash(text: str) -> str:
        normalized = re.sub(r"\W+", " ", text.lower()).strip()
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    def signature(self, text: str) -> Optional[np.ndarray]:
        tokens = re.findall(r"[a-z0-9]+", text.lower())
        if len(tokens) < self.shingle_size + 2:
            return None
        shingles = {
            hash(" ".join(tokens[i : i + self.shingle_size])) & 0x7FFFFFFF
            for i in range(len(tokens) - self.shingle_size + 1)
        }
        values = np.fromiter(shingles, dtype=np.int64, count=len(shingles))
        hashed = (np.outer(self._a, values) + self._b[:, None]) % self._prime
        return hashed.min(axis=1)

    @staticmethod
    def similarity(left: np.ndarray, right: np.ndarray) -> float:
        return float(np.mean(left == right))

    def deduplicate(self, documents: Sequence[RawDocument]) -> Tuple[List[RawDocument], Dict[str, Any]]:
        kept: List[RawDocument] = []
        signatures: List[Tuple[np.ndarray, str]] = []
        seen_hashes: Set[str] = set()
        removed_exact = 0
        removed_near = 0

        # Longest documents first so the most informative version survives.
        ordered = sorted(documents, key=lambda d: d.n_chars, reverse=True)
        for document in tqdm(ordered, desc="Deduplicating", leave=False):
            digest = self.exact_hash(document.text)
            if digest in seen_hashes:
                removed_exact += 1
                continue
            signature = self.signature(document.text)
            duplicate = False
            if signature is not None:
                for other_signature, _ in signatures:
                    if self.similarity(signature, other_signature) >= self.threshold:
                        duplicate = True
                        break
            if duplicate:
                removed_near += 1
                continue
            seen_hashes.add(digest)
            if signature is not None:
                signatures.append((signature, document.doc_id))
            kept.append(document)

        stats = {
            "input": len(documents),
            "kept": len(kept),
            "removed_exact_duplicates": removed_exact,
            "removed_near_duplicates": removed_near,
        }
        return kept, stats


def clean_corpus(documents: Sequence[RawDocument]) -> List[Dict[str, Any]]:
    """Clean, filter and deduplicate the raw corpus (cacheable pure function)."""
    cleaner = TextCleaner(CONFIG)
    cleaned: List[RawDocument] = []
    for source_document in tqdm(documents, desc="Cleaning documents", leave=False):
        try:
            # Work on a copy: clean_document mutates in place, and cleaning the
            # caller's own objects made the before/after summary compare a list
            # against itself (it always reported 0.0% removed).
            document = cleaner.clean_document(RawDocument(**source_document.to_dict()))
        except Exception as exc:
            log_error("clean", exc, source_document.doc_id)
            continue
        if len(document.text) >= CONFIG.min_document_chars or document.document_type == "guideline_reference":
            cleaned.append(document)
    deduplicated, stats = Deduplicator(CONFIG.near_duplicate_threshold).deduplicate(cleaned)
    logger.info("cleaning/dedup stats: %s", stats)
    (PATHS.logs / "cleaning_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    return [doc.to_dict() for doc in deduplicated]


CLEAN_DOCUMENTS: List[RawDocument] = [
    RawDocument(**record)
    for record in CACHE.get_or_build(
        PATHS.processed / f"clean_corpus_{CONFIG.pipeline_version}.json",
        lambda: clean_corpus(RAW_DOCUMENTS),
        fmt="json",
        label=f"clean_corpus_{CONFIG.pipeline_version}.json",
    )
]

before_chars = sum(d.n_chars for d in RAW_DOCUMENTS)
after_chars = sum(d.n_chars for d in CLEAN_DOCUMENTS)
print("=" * 78)
print("CLEANING SUMMARY")
print("=" * 78)
print(f"  documents : {len(RAW_DOCUMENTS):>7}  →  {len(CLEAN_DOCUMENTS):>7}")
print(f"  characters: {before_chars:>7,}  →  {after_chars:>7,}  "
      f"({100 * (1 - after_chars / max(before_chars, 1)):.1f}% removed as noise/duplication)")
print("\nExample cleaned excerpt:")
if CLEAN_DOCUMENTS:
    sample = max(CLEAN_DOCUMENTS, key=lambda d: len(d.sections))
    print(f"  [{sample.source}] {sample.title[:90]}")
    print("  " + sample.text[:340].replace("\n", " ") + " …")

18:56:52 | INFO    | psych-rag | cache HIT  → clean_corpus_v3.json


CLEANING SUMMARY
  documents :     888  →      888
  characters: 5,598,379  →  5,591,912  (0.1% removed as noise/duplication)

Example cleaned excerpt:
  [europepmc] Efficacy of psychological interventions for adolescents with borderline personality disord
  Abstract AbstractObjectiveThis meta-analysis evaluated the efficacy of psychological interventions for adolescents with borderline personality disorder (BPD) across multiple outcome domains, including BPD symptom severity, emotion regulation, depressive symptoms, general psychopathology, and quality of life.MethodsFollowing PRISMA 2020 gu …


---
## 6. Metadata Generation & Semantic Chunking

### Why not fixed-size chunking?

Naive fixed-size windows cut across clinical section boundaries — a chunk that begins in *Epidemiology* and
ends in *Pharmacological treatment* answers neither question well and pollutes retrieval with mixed topics.

`SemanticChunker` therefore works in three passes:

1. **Structure recovery.** Native structure (PubMed abstract labels, JATS `<sec>` titles, NIMH `<h2>`/`<h3>`)
   is used when available. For PDFs with no markup, headings are recovered heuristically from numbered
   headings, ALL-CAPS lines, title-case short lines, and colon-terminated leads.
2. **Section canonicalisation.** Free-text headings are mapped onto a controlled clinical vocabulary:
   `diagnosis · clinical_features · epidemiology · risk_factors · differential_diagnosis · treatment ·
   medications · psychosocial_intervention · management · prognosis · follow_up · referral · assessment ·
   prevention · references · other`. This label becomes searchable metadata *and* the basis of benchmark
   question generation.
3. **Recursive splitting with overlap.** Long sections are split on the strongest available boundary in the
   order `paragraph → sentence → clause → word`, targeting `chunk_target_tokens` with
   `chunk_overlap_tokens` of overlap so cross-boundary facts stay recoverable. Tiny trailing fragments are
   merged back into the previous chunk.

Every chunk inherits full provenance metadata (source, organization, title, year, topic, section,
document_type, url) so that citations are verifiable and the Corpus Explorer tab is browsable.

In [12]:
# =============================================================================
# SECTION 6 — SEMANTIC CHUNKING
# =============================================================================


@dataclass
class Chunk:
    """A retrievable unit of evidence with complete provenance metadata."""

    chunk_id: str
    doc_id: str
    text: str
    source: str
    organization: str
    title: str
    year: Optional[int]
    topic: str
    section: str                    # canonical section label
    section_heading: str            # original heading text
    document_type: str
    url: str
    license_note: str = ""
    n_tokens: int = 0
    position: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def citation(self) -> str:
        """Human-readable citation string used in prompts and the UI."""
        year = self.year if self.year else "n.d."
        return f"{self.organization} ({year}). {self.title} — {self.section_heading}"


def approx_token_count(text: str) -> int:
    """Fast tokenizer-free length estimate (~1.33 tokens per whitespace word).

    Used for chunk budgeting only; exact tokenisation happens inside the models.
    """
    return int(len(text.split()) * 1.33) + 1


class SemanticChunker:
    """Section-aware recursive chunker with overlap."""

    SECTION_VOCABULARY: Dict[str, Tuple[str, ...]] = {
        "diagnosis": ("diagnos", "diagnostic criteria", "case definition", "classification", "coding"),
        "clinical_features": ("clinical feature", "signs and symptoms", "symptom", "presentation",
                              "manifestation", "phenomenolog", "inclusion"),
        "epidemiology": ("epidemiolog", "prevalence", "incidence", "burden", "global burden", "demograph"),
        "risk_factors": ("risk factor", "aetiolog", "etiolog", "cause", "pathophysiolog", "genetic",
                         "neurobiolog", "determinant"),
        "differential_diagnosis": ("differential", "exclusion", "comorbid", "co-occurring", "boundary with"),
        "screening": ("screening", "case finding", "detection", "rating scale", "questionnaire"),
        "assessment": ("assessment", "evaluation", "history taking", "examination", "investigation", "work-up"),
        "treatment": ("treatment", "therapy", "therapeutic", "intervention", "efficacy", "effectiveness"),
        "medications": ("medication", "pharmacolog", "pharmacotherap", "drug", "antidepressant",
                        "antipsychotic", "mood stabiliser", "mood stabilizer", "benzodiazepine", "dosing",
                        "adverse effect", "side effect"),
        "psychosocial_intervention": ("psychosocial", "psychotherap", "psychological", "cognitive behav",
                                      "counsel", "family intervention", "psychoeducation", "rehabilitation"),
        "management": ("management", "care plan", "stepped care", "service delivery", "implementation",
                       "protocol", "algorithm"),
        "prognosis": ("prognos", "outcome", "course", "remission", "relapse", "recovery", "mortality"),
        "follow_up": ("follow-up", "follow up", "monitoring", "maintenance", "reassess", "aftercare"),
        "referral": ("referral", "refer to", "specialist", "when to refer", "escalation"),
        "prevention": ("prevention", "preventive", "public health", "promotion", "early intervention"),
        "special_populations": ("children", "adolescent", "older adult", "pregnan", "perinatal", "youth"),
        "results": ("result", "finding", "outcome measure"),
        "methods": ("method", "search strategy", "study selection", "data extraction", "protocol registration"),
        "background": ("background", "introduction", "overview", "objective", "aim", "purpose", "abstract"),
        "conclusion": ("conclusion", "discussion", "summary", "implication", "key message"),
        "references": ("reference", "bibliograph", "further reading", "acknowledg", "funding",
                       "conflict of interest", "appendix"),
    }

    # Journal front/back matter carries no clinical signal and pollutes retrieval.
    NON_CONTENT_HEADING = re.compile(
        r"^\s*(author (contribution|information)|contributor|funding|financial support|acknowledg|"
        r"conflict of interest|competing interest|declaration|data availability|supplementary|"
        r"ethic|consent|abbreviation|orcid|correspondence|publisher'?s note|copyright|open access|"
        r"reference|bibliograph|footnote|appendix|about this article|cite this article)",
        re.IGNORECASE,
    )

    # Headings recovered from unstructured PDF text.
    HEADING_PATTERNS = [
        re.compile(r"^\s*(\d+(\.\d+)*)[\.\)]?\s+([A-Z][^\n]{3,90})$"),   # 3.2 Pharmacological management
        re.compile(r"^\s*([A-Z][A-Z \-/&,']{5,80})$"),                    # ALL CAPS HEADING
        re.compile(r"^\s*([A-Z][A-Za-z \-/&,']{3,70}):\s*$"),             # Title case with colon
    ]

    SPLIT_HIERARCHY = ["\n\n", "\n", ". ", "; ", ", ", " "]

    def __init__(self, config: RAGConfig) -> None:
        self.config = config

    # -- section handling -----------------------------------------------------
    def canonical_section(self, heading: str, body: str = "") -> str:
        """Map an arbitrary heading onto the controlled clinical vocabulary.

        The heading is authoritative when it matches anything: body text is only
        consulted for untitled sections. Mixing the two lets incidental body
        vocabulary override an explicit heading, which is how 'Author
        contributions' was previously classified as 'follow_up'.
        """
        heading_lower = heading.lower()
        heading_scores = {
            label: sum(1 for keyword in keywords if keyword in heading_lower)
            for label, keywords in self.SECTION_VOCABULARY.items()
        }
        best_label = max(heading_scores, key=lambda label: heading_scores[label])
        if heading_scores[best_label] > 0:
            return best_label

        body_lower = body[:400].lower()
        body_scores = {
            label: sum(1 for keyword in keywords if keyword in body_lower)
            for label, keywords in self.SECTION_VOCABULARY.items()
        }
        best_label = max(body_scores, key=lambda label: body_scores[label])
        return best_label if body_scores[best_label] > 0 else "other"

    def recover_sections(self, text: str) -> List[Dict[str, str]]:
        """Heuristically split unstructured text into (heading, text) sections."""
        lines = text.split("\n")
        sections: List[Dict[str, str]] = []
        heading = "Document"
        buffer: List[str] = []
        for line in lines:
            stripped = line.strip()
            is_heading = bool(stripped) and len(stripped) <= 95 and any(
                pattern.match(stripped) for pattern in self.HEADING_PATTERNS
            )
            if is_heading:
                if buffer:
                    sections.append({"heading": heading, "text": "\n".join(buffer).strip()})
                    buffer = []
                heading = stripped.rstrip(":")
            else:
                buffer.append(line)
        if buffer:
            sections.append({"heading": heading, "text": "\n".join(buffer).strip()})
        merged = [s for s in sections if len(s["text"]) > 120]
        return merged or [{"heading": "Document", "text": text}]

    # -- recursive splitting --------------------------------------------------
    def _split_recursive(self, text: str, target: int, overlap: int, depth: int = 0) -> List[str]:
        """Split `text` into <= `target`-token pieces along the strongest boundary."""
        if approx_token_count(text) <= target or depth >= len(self.SPLIT_HIERARCHY):
            return [text]
        separator = self.SPLIT_HIERARCHY[depth]
        parts = [p for p in text.split(separator) if p.strip()]
        if len(parts) == 1:
            return self._split_recursive(text, target, overlap, depth + 1)

        chunks: List[str] = []
        buffer: List[str] = []
        buffer_tokens = 0
        for part in parts:
            part_tokens = approx_token_count(part)
            if part_tokens > target:
                if buffer:
                    chunks.append(separator.join(buffer))
                    buffer, buffer_tokens = [], 0
                chunks.extend(self._split_recursive(part, target, overlap, depth + 1))
                continue
            if buffer_tokens + part_tokens > target and buffer:
                chunks.append(separator.join(buffer))
                # Carry the tail of the previous chunk forward as overlap.
                carry: List[str] = []
                carry_tokens = 0
                for previous in reversed(buffer):
                    previous_tokens = approx_token_count(previous)
                    if carry_tokens + previous_tokens > overlap:
                        break
                    carry.insert(0, previous)
                    carry_tokens += previous_tokens
                buffer, buffer_tokens = list(carry), carry_tokens
            buffer.append(part)
            buffer_tokens += part_tokens
        if buffer:
            chunks.append(separator.join(buffer))
        return [c.strip() for c in chunks if c.strip()]

    def _merge_small_tail(self, pieces: List[str], minimum: int) -> List[str]:
        """Fold an undersized final piece back into its predecessor.

        Note: the tail is popped *before* the store. Writing this as
        ``pieces[-2] = pieces[-2] + pieces.pop()`` is a latent bug, because Python
        evaluates the right-hand side (including the pop) first and then indexes
        into the already-shortened list.
        """
        if len(pieces) >= 2 and approx_token_count(pieces[-1]) < minimum:
            tail = pieces.pop()
            pieces[-1] = f"{pieces[-1]} {tail}"
        return pieces

    # -- public API -----------------------------------------------------------
    def chunk_document(
        self,
        document: RawDocument,
        target: Optional[int] = None,
        overlap: Optional[int] = None,
    ) -> List[Chunk]:
        target = target or self.config.chunk_target_tokens
        overlap = overlap or self.config.chunk_overlap_tokens
        sections = document.sections or self.recover_sections(document.text)

        chunks: List[Chunk] = []
        position = 0
        for section in sections:
            heading = (section.get("heading") or "Document").strip()
            body = (section.get("text") or "").strip()
            if approx_token_count(body) < self.config.chunk_min_tokens:
                continue
            if self.NON_CONTENT_HEADING.match(heading):
                continue                     # funding, ORCID, author contributions, references…
            label = self.canonical_section(heading, body)
            pieces = self._merge_small_tail(
                self._split_recursive(body, target, overlap), self.config.chunk_min_tokens
            )
            for piece in pieces:
                n_tokens = approx_token_count(piece)
                if n_tokens < self.config.chunk_min_tokens:
                    continue
                # Long unstructured PDFs (e.g. the WHO mhGAP guide) arrive as one
                # untitled block covering many disorders. Inferring topic and
                # section per chunk makes that content addressable instead of
                # landing every chunk under topic='general', section='other'.
                chunk_topic = document.topic
                if chunk_topic == "general":
                    chunk_topic = infer_topic_from_text(piece, default="general")
                chunk_section = label
                if chunk_section == "other":
                    chunk_section = self.canonical_section("", piece)
                # Prefixing the heading gives the embedder crucial local context.
                contextual_text = f"{document.title} — {heading}\n{piece}"
                chunks.append(
                    Chunk(
                        chunk_id=f"{document.doc_id}::c{position:04d}",
                        doc_id=document.doc_id,
                        text=contextual_text,
                        source=document.source,
                        organization=document.organization,
                        title=document.title,
                        year=document.year,
                        topic=chunk_topic,
                        section=chunk_section,
                        section_heading=heading[:140],
                        document_type=document.document_type,
                        url=document.url,
                        license_note=document.license_note,
                        n_tokens=min(n_tokens, self.config.chunk_max_tokens),
                        position=position,
                    )
                )
                position += 1
        return chunks

    def chunk_corpus(
        self,
        documents: Sequence[RawDocument],
        target: Optional[int] = None,
        overlap: Optional[int] = None,
        show_progress: bool = True,
    ) -> List[Chunk]:
        chunks: List[Chunk] = []
        iterator = tqdm(documents, desc="Chunking", leave=False) if show_progress else documents
        for document in iterator:
            try:
                chunks.extend(self.chunk_document(document, target, overlap))
            except Exception as exc:
                log_error("chunking", exc, document.doc_id)
        return chunks


CHUNKER = SemanticChunker(CONFIG)

In [13]:
# =============================================================================
# SECTION 6b — CHUNK-SIZE EXPERIMENT
# =============================================================================
# Rather than assume a chunk size, we measure the trade-off directly:
#   * small chunks  → high precision, but facts get fragmented and recall suffers
#   * large chunks  → better recall of complete answers, but diluted embeddings
#     and more wasted context tokens at generation time
# We report chunk counts, token statistics and section purity for each setting.
# =============================================================================


def chunk_size_experiment(documents: Sequence[RawDocument], sizes: Sequence[int]) -> pd.DataFrame:
    """Compare chunking statistics across candidate target sizes."""
    rows: List[Dict[str, Any]] = []
    sample = list(documents)[:220]                 # a sample keeps the study fast
    for size in sizes:
        overlap = max(24, int(size * 0.2))
        started = time.perf_counter()
        chunks = CHUNKER.chunk_corpus(sample, target=size, overlap=overlap, show_progress=False)
        if not chunks:
            continue
        lengths = np.array([c.n_tokens for c in chunks])
        section_counts = Counter(c.section for c in chunks)
        rows.append(
            {
                "target_tokens": size,
                "overlap_tokens": overlap,
                "n_chunks": len(chunks),
                "chunks_per_doc": round(len(chunks) / max(len(sample), 1), 2),
                "mean_tokens": round(float(lengths.mean()), 1),
                "median_tokens": float(np.median(lengths)),
                "p95_tokens": round(float(np.percentile(lengths, 95)), 1),
                "pct_below_min": round(100 * float((lengths < CONFIG.chunk_min_tokens).mean()), 2),
                "distinct_sections": len(section_counts),
                "seconds": round(time.perf_counter() - started, 2),
            }
        )
    return pd.DataFrame(rows)


CHUNK_EXPERIMENT = CACHE.get_or_build(
    PATHS.processed / f"chunk_size_experiment_{CONFIG.pipeline_version}.pkl",
    lambda: chunk_size_experiment(CLEAN_DOCUMENTS, CONFIG.chunk_size_experiment),
    fmt="pickle",
    label=f"chunk_size_experiment_{CONFIG.pipeline_version}",
)

print("CHUNK-SIZE EXPERIMENT (sample of the corpus)")
print("=" * 78)
print(CHUNK_EXPERIMENT.to_string(index=False))
print(
    f"\nSelected configuration: target={CONFIG.chunk_target_tokens} tokens, "
    f"overlap={CONFIG.chunk_overlap_tokens} tokens.\n"
    "Rationale: ~320 tokens comfortably holds one clinical sub-topic (e.g. a full\n"
    "pharmacological-management paragraph) while staying well inside the 512-token\n"
    "input window of bge-large, so no chunk is silently truncated by the embedder."
)

18:56:52 | INFO    | psych-rag | cache HIT  → chunk_size_experiment_v3


CHUNK-SIZE EXPERIMENT (sample of the corpus)
 target_tokens  overlap_tokens  n_chunks  chunks_per_doc  mean_tokens  median_tokens  p95_tokens  pct_below_min  distinct_sections  seconds
           192              38      6580           29.91        132.7          143.0       187.0            0.0                 22     0.81
           320              64      4473           20.33        193.5          197.0       309.0            0.0                 22     0.42
           448              89      3621           16.46        236.8          228.0       429.0            0.0                 22     0.40

Selected configuration: target=320 tokens, overlap=64 tokens.
Rationale: ~320 tokens comfortably holds one clinical sub-topic (e.g. a full
pharmacological-management paragraph) while staying well inside the 512-token
input window of bge-large, so no chunk is silently truncated by the embedder.


---
## 7. Corpus Construction

The chunked corpus is materialised as a `pandas.DataFrame` — the single object that the retriever, the
benchmark builder, the evaluator and the Gradio Corpus Explorer all read from. Chunk order is stable and
positional, so FAISS integer ids map directly onto DataFrame row indices.

In [14]:
# =============================================================================
# SECTION 7 — UNIFIED CORPUS
# =============================================================================


def build_chunk_corpus() -> List[Dict[str, Any]]:
    chunks = CHUNKER.chunk_corpus(CLEAN_DOCUMENTS)
    # Drop chunks that are almost entirely references/boilerplate.
    filtered = [c for c in chunks if c.section != "references" or c.n_tokens > 120]
    logger.info("chunking produced %d chunks (%d after reference filtering)", len(chunks), len(filtered))
    return [c.to_dict() for c in filtered]


CHUNK_RECORDS: List[Dict[str, Any]] = CACHE.get_or_build(
    PATHS.processed / f"chunks_{CONFIG.pipeline_version}.json",
    build_chunk_corpus,
    fmt="json",
    label=f"chunks_{CONFIG.pipeline_version}.json",
)

CHUNKS: List[Chunk] = [Chunk(**record) for record in CHUNK_RECORDS]
CORPUS = pd.DataFrame(CHUNK_RECORDS)
CORPUS["year"] = pd.to_numeric(CORPUS["year"], errors="coerce")
CORPUS = CORPUS.reset_index(drop=True)
CORPUS["row_id"] = CORPUS.index

# Fast lookup structures used everywhere downstream.
def _fingerprint(parts: Iterable[str]) -> str:
    return hashlib.sha1("\x00".join(parts).encode("utf-8")).hexdigest()[:10]


# Two fingerprints, because two different things can go stale independently.
#   TEXT  — chunk ids + chunk text. Keys the embeddings, FAISS and BM25 indices.
#   META  — chunk ids + topic + section. Keys the benchmark and every evaluation
#           artefact, because benchmark questions are generated from that metadata.
# Keying everything on chunk ids alone (as an earlier version did) meant a change
# that altered only topic/section labels left a stale benchmark in place: the ids
# still matched, so the cache reported a hit.
CORPUS_TEXT_FINGERPRINT: str = _fingerprint(
    f"{cid}|{text}" for cid, text in zip(CORPUS["chunk_id"], CORPUS["text"])
)
CORPUS_META_FINGERPRINT: str = _fingerprint(
    f"{cid}|{topic}|{section}"
    for cid, topic, section in zip(CORPUS["chunk_id"], CORPUS["topic"], CORPUS["section"])
)
# Evaluation depends on both, plus the benchmark-generation logic itself.
CORPUS_EVAL_FINGERPRINT: str = (
    f"{CORPUS_TEXT_FINGERPRINT}_{CORPUS_META_FINGERPRINT}_{CONFIG.benchmark_version}"
)
CORPUS_FINGERPRINT: str = CORPUS_TEXT_FINGERPRINT

CHUNK_ID_TO_ROW: Dict[str, int] = {cid: i for i, cid in enumerate(CORPUS["chunk_id"])}
ROW_TO_CHUNK_ID: List[str] = CORPUS["chunk_id"].tolist()
CHUNK_TEXTS: List[str] = CORPUS["text"].tolist()

print("=" * 78)
print("UNIFIED CORPUS")
print("=" * 78)
print(f"  documents      : {CORPUS['doc_id'].nunique():,}")
print(f"  chunks         : {len(CORPUS):,}")
print(f"  total tokens   : {int(CORPUS['n_tokens'].sum()):,} (approx.)")
print(f"  mean tokens    : {CORPUS['n_tokens'].mean():.1f}")
print(f"  sources        : {CORPUS['source'].nunique()}")
print(f"  sections       : {CORPUS['section'].nunique()}")
print(f"  text fp        : {CORPUS_TEXT_FINGERPRINT}  (keys embeddings / FAISS / BM25)")
print(f"  metadata fp    : {CORPUS_META_FINGERPRINT}  (keys benchmark / evaluation)")
print("\nChunks by source:")
print(CORPUS["source"].value_counts().to_string())
print("\nChunks by canonical section (top 12):")
print(CORPUS["section"].value_counts().head(12).to_string())
print("\nSample chunk:")
if len(CORPUS):
    example = CORPUS.iloc[len(CORPUS) // 2]
    print(f"  chunk_id : {example['chunk_id']}")
    print(f"  source   : {example['source']} | section: {example['section']} | topic: {example['topic']}")
    print(f"  text     : {example['text'][:300].replace(chr(10), ' ')} …")
CORPUS.head(3)

18:56:53 | INFO    | psych-rag | cache HIT  → chunks_v3.json


UNIFIED CORPUS
  documents      : 839
  chunks         : 5,762
  total tokens   : 1,030,195 (approx.)
  mean tokens    : 178.8
  sources        : 6
  sections       : 22
  text fp        : 00f98c7248  (keys embeddings / FAISS / BM25)
  metadata fp    : a1c407c885  (keys benchmark / evaluation)

Chunks by source:
source
europepmc        3622
pubmed           1738
who_mhgap         210
medlineplus       113
nimh               78
who_reference       1

Chunks by canonical section (top 12):
section
background             1406
conclusion              809
results                 641
methods                 617
other                   552
treatment               413
clinical_features       242
assessment              198
diagnosis               189
prognosis               154
risk_factors             90
special_populations      81

Sample chunk:
  chunk_id : europepmc-bce2d3e6a1d1::c0037
  source   : europepmc | section: clinical_features | topic: ptsd
  text     : N-acetylcysteine for patien

,chunk_id,doc_id,text,source,organization,title,year,topic,section,section_heading,document_type,url,license_note,n_tokens,position,row_id
0,who-650ba96c625b::c0000,who-650ba96c625b,"mhGAP Intervention Guide for mental, neurologi...",who_mhgap,World Health Organization,"mhGAP Intervention Guide for mental, neurologi...",2016.0,general,other,Document,guideline,https://www.who.int/publications/i/item/978924...,WHO publications: CC BY-NC-SA 3.0 IGO.,289,0,0
1,who-650ba96c625b::c0001,who-650ba96c625b,"mhGAP Intervention Guide for mental, neurologi...",who_mhgap,World Health Organization,"mhGAP Intervention Guide for mental, neurologi...",2016.0,general,other,Document,guideline,https://www.who.int/publications/i/item/978924...,WHO publications: CC BY-NC-SA 3.0 IGO.,290,1,1
2,who-650ba96c625b::c0002,who-650ba96c625b,"mhGAP Intervention Guide for mental, neurologi...",who_mhgap,World Health Organization,"mhGAP Intervention Guide for mental, neurologi...",2016.0,general,other,Document,guideline,https://www.who.int/publications/i/item/978924...,WHO publications: CC BY-NC-SA 3.0 IGO.,247,2,2


---
## 8. Benchmark QA Dataset Construction

A retrieval system cannot be improved if it cannot be measured. Because no public psychiatry-RAG benchmark
matches *this* corpus, the notebook **synthesises one from the corpus itself** using a technique standard in
IR research (inverse cloze / pseudo-query generation), with three complementary generators:

| Generator | Example | Relevance signal |
|---|---|---|
| **Template** — section + topic | *"What are the diagnostic features of generalised anxiety disorder?"* | All chunks of that topic + canonical section |
| **Title-anchored** — document heading | *"According to the WHO mhGAP Intervention Guide, how should follow-up be conducted?"* | Chunks of that document + section |
| **Keyphrase** — salient terms from the chunk | *"What does the evidence say about lithium monitoring in bipolar disorder?"* | The source chunk (+ its siblings) |

Each benchmark item stores `question`, `relevant_chunk_ids`, `relevant_doc_ids`, `expected_answer`
(an extractive reference summary), and provenance for auditing.

**Why a 70/15/15 split for a system with no trained weights?**
Nothing is back-propagated here, but plenty is *chosen*: fusion weights, chunk size, top-k, rerank depth,
confidence thresholds and prompt variants. Those are hyper-parameters, and tuning them on the same questions
used to report results is exactly the leakage the split prevents.

* **Train (70%)** — free exploration: prompt iteration, weight sweeps, error analysis.
* **Validation (15%)** — model selection: pick fusion weights / rerank depth / thresholds.
* **Test (15%)** — touched once, for the numbers reported in the dashboard.

The split is **grouped by `doc_id`** rather than by question. Questions derived from the same document share
near-identical evidence, so a random question-level split would leak the answer document across folds and
inflate every metric. Grouping guarantees the test questions point at documents never used for tuning. The
split is stratified by topic and seeded (`seed=42`) for exact reproducibility.

In [15]:
# =============================================================================
# SECTION 8 — BENCHMARK QA DATASET
# =============================================================================


@dataclass
class BenchmarkItem:
    """A single evaluation question with graded ground truth."""

    question_id: str
    question: str
    relevant_chunk_ids: List[str]
    relevant_doc_ids: List[str]
    expected_answer: str
    topic: str
    section: str
    source: str
    generator: str
    split: str = "train"

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class BenchmarkBuilder:
    """Deterministically synthesise a retrieval benchmark from the corpus."""

    SECTION_TEMPLATES: Dict[str, Tuple[str, ...]] = {
        "diagnosis": (
            "What are the diagnostic features of {topic}?",
            "How is {topic} diagnosed?",
            "What diagnostic criteria are described for {topic}?",
        ),
        "clinical_features": (
            "What are the main signs and symptoms of {topic}?",
            "How does {topic} typically present clinically?",
        ),
        "epidemiology": (
            "What is the prevalence of {topic}?",
            "What does the evidence say about the epidemiology and burden of {topic}?",
        ),
        "risk_factors": (
            "What are the risk factors for {topic}?",
            "What aetiological factors are associated with {topic}?",
        ),
        "differential_diagnosis": (
            "What conditions should be considered in the differential diagnosis of {topic}?",
            "Which comorbidities commonly co-occur with {topic}?",
        ),
        "screening": ("Which screening tools are recommended for {topic}?",),
        "assessment": (
            "How should a patient with suspected {topic} be assessed?",
            "What should a clinical assessment for {topic} cover?",
        ),
        "treatment": (
            "What treatments are recommended for {topic}?",
            "What does the evidence show about the effectiveness of treatments for {topic}?",
        ),
        "medications": (
            "Which medications are used in the management of {topic}?",
            "What pharmacological options and adverse effects are described for {topic}?",
        ),
        "psychosocial_intervention": (
            "Which psychosocial or psychological interventions are recommended for {topic}?",
            "What is the evidence for psychotherapy in {topic}?",
        ),
        "management": (
            "How should {topic} be managed in practice?",
            "What management algorithm is described for {topic}?",
        ),
        "prognosis": (
            "What is the prognosis of {topic}?",
            "What outcomes and relapse rates are reported for {topic}?",
        ),
        "follow_up": (
            "How should patients with {topic} be monitored during follow-up?",
            "What follow-up schedule is recommended for {topic}?",
        ),
        "referral": ("When should a patient with {topic} be referred to specialist services?",),
        "prevention": ("What preventive strategies are described for {topic}?",),
        "special_populations": ("What considerations apply to {topic} in specific populations?",),
        "conclusion": ("What are the key conclusions of the evidence on {topic}?",),
    }

    STOPWORDS: Set[str] = set(
        """a an the and or but if while of to in on for with without from by as at is are was were be been
        being this that these those it its their there here we our they he she his her not no than then so
        such can may might should could would will shall have has had do does did between among during also
        into over under about more most other some any each per via using used use study studies patients
        patient results result data evidence review reviews included included""".split()
    )

    def __init__(self, config: RAGConfig, corpus: pd.DataFrame) -> None:
        self.config = config
        self.corpus = corpus
        self.rng = random.Random(config.seed)

    # -- helpers --------------------------------------------------------------
    @staticmethod
    def _sentences(text: str) -> List[str]:
        body = text.split("\n", 1)[-1]                      # drop the title/heading prefix
        parts = re.split(r"(?<=[.!?])\s+(?=[A-Z(])", body)
        return [p.strip() for p in parts if len(p.strip()) > 40]

    def _reference_answer(self, text: str, max_sentences: int = 3) -> str:
        sentences = self._sentences(text)
        return " ".join(sentences[:max_sentences]) if sentences else text[:400]

    def _keyphrases(self, text: str, limit: int = 3) -> List[str]:
        """Extract salient multi-word noun-ish phrases for keyphrase questions."""
        body = text.split("\n", 1)[-1].lower()
        candidates = re.findall(r"\b([a-z][a-z\-]{3,}(?: [a-z][a-z\-]{3,}){1,2})\b", body)
        scored: Counter = Counter()
        for phrase in candidates:
            words = phrase.split()
            if any(word in self.STOPWORDS for word in words):
                continue
            scored[phrase] += len(words)
        return [phrase for phrase, _ in scored.most_common(limit)]

    def _relevant_by_topic_section(self, topic: str, section: str) -> List[str]:
        mask = (self.corpus["topic"] == topic) & (self.corpus["section"] == section)
        return self.corpus.loc[mask, "chunk_id"].tolist()

    # -- generators -----------------------------------------------------------
    def _template_items(self) -> List[BenchmarkItem]:
        items: List[BenchmarkItem] = []
        groups = self.corpus.groupby(["topic", "section"], sort=True)
        for (topic, section), frame in groups:
            if topic == "general" or section not in self.SECTION_TEMPLATES or len(frame) < 2:
                continue
            label = PSYCHIATRY_TOPICS.get(topic, {}).get("label", topic.replace("_", " "))
            for template in self.SECTION_TEMPLATES[section]:
                question = template.format(topic=label.lower())
                relevant = frame["chunk_id"].tolist()
                anchor = frame.sort_values("n_tokens", ascending=False).iloc[0]
                items.append(
                    BenchmarkItem(
                        question_id=make_doc_id("q_tpl", question),
                        question=question,
                        relevant_chunk_ids=relevant[:40],
                        relevant_doc_ids=sorted(set(frame["doc_id"].tolist()))[:40],
                        expected_answer=self._reference_answer(anchor["text"]),
                        topic=topic,
                        section=section,
                        source=anchor["source"],
                        generator="template",
                    )
                )
        return items

    def _title_items(self, per_source_limit: int = 40) -> List[BenchmarkItem]:
        items: List[BenchmarkItem] = []
        counts: Counter = Counter()
        eligible = self.corpus[self.corpus["n_tokens"] >= 90]
        for doc_id, frame in eligible.groupby("doc_id", sort=True):
            source = frame.iloc[0]["source"]
            if counts[source] >= per_source_limit:
                continue
            organization = frame.iloc[0]["organization"]
            document_title = str(frame.iloc[0]["title"]).strip()
            for _, row in frame.sample(min(2, len(frame)), random_state=self.config.seed).iterrows():
                heading = str(row["section_heading"]).strip().rstrip(".")
                if len(heading) < 4 or heading.lower() in {"document", "section", "abstract"}:
                    continue
                # The document title is included so the question identifies exactly one
                # document; without it, questions from different documents collapse into
                # duplicates and the ground-truth labels become ambiguous.
                question = (
                    f"According to '{document_title[:110]}' ({organization}), what is reported "
                    f"under '{heading}' regarding "
                    f"{PSYCHIATRY_TOPICS.get(row['topic'], {}).get('label', row['topic']).lower()}?"
                )
                siblings = frame[frame["section"] == row["section"]]["chunk_id"].tolist()
                items.append(
                    BenchmarkItem(
                        question_id=make_doc_id("q_ttl", question + row["chunk_id"]),
                        question=question,
                        relevant_chunk_ids=list(dict.fromkeys([row["chunk_id"], *siblings]))[:20],
                        relevant_doc_ids=[doc_id],
                        expected_answer=self._reference_answer(row["text"]),
                        topic=row["topic"],
                        section=row["section"],
                        source=source,
                        generator="title_anchored",
                    )
                )
                counts[source] += 1
        return items

    def _keyphrase_items(self, limit: int = 220) -> List[BenchmarkItem]:
        items: List[BenchmarkItem] = []
        eligible = self.corpus[self.corpus["n_tokens"] >= 110]
        if eligible.empty:
            return items
        sample = eligible.sample(min(limit, len(eligible)), random_state=self.config.seed)
        for _, row in sample.iterrows():
            phrases = self._keyphrases(row["text"])
            if not phrases:
                continue
            label = PSYCHIATRY_TOPICS.get(row["topic"], {}).get("label", row["topic"].replace("_", " ")).lower()
            question = f"What does the evidence say about {phrases[0]} in the context of {label}?"
            siblings = self.corpus[
                (self.corpus["doc_id"] == row["doc_id"]) & (self.corpus["section"] == row["section"])
            ]["chunk_id"].tolist()
            items.append(
                BenchmarkItem(
                    question_id=make_doc_id("q_kp", question + row["chunk_id"]),
                    question=question,
                    relevant_chunk_ids=list(dict.fromkeys([row["chunk_id"], *siblings]))[:12],
                    relevant_doc_ids=[row["doc_id"]],
                    expected_answer=self._reference_answer(row["text"]),
                    topic=row["topic"],
                    section=row["section"],
                    source=row["source"],
                    generator="keyphrase",
                )
            )
        return items

    # -- assembly + splitting -------------------------------------------------
    def build(self) -> List[BenchmarkItem]:
        items = self._template_items() + self._title_items() + self._keyphrase_items()
        seen: Set[str] = set()
        unique: List[BenchmarkItem] = []
        for item in items:
            key = item.question.lower().strip()
            if key in seen or not item.relevant_chunk_ids:
                continue
            seen.add(key)
            unique.append(item)
        self.rng.shuffle(unique)
        return unique[: self.config.benchmark_max_questions]

    def split(self, items: Sequence[BenchmarkItem]) -> List[BenchmarkItem]:
        """Grouped, topic-stratified, deterministic 70/15/15 split.

        Grouping key = primary relevant document, so no document appears in two
        splits and evidence cannot leak from tuning into reporting.

        Allocation is greedy on QUESTION count rather than group count. Groups hold
        very unequal numbers of questions (a template question covers a whole
        topic+section; a title-anchored question covers one document), so splitting
        70% of the *groups* produced 74/13/13 rather than the documented 70/15/15.
        Each group is handed to whichever split is furthest below its target share,
        which keeps groups intact — no leakage — while tracking the target ratios.
        """
        targets = {
            "train": self.config.train_ratio,
            "validation": self.config.val_ratio,
            "test": self.config.test_ratio,
        }
        by_topic: Dict[str, List[BenchmarkItem]] = defaultdict(list)
        for item in items:
            by_topic[item.topic].append(item)

        assigned: List[BenchmarkItem] = []
        doc_assignment: Dict[str, str] = {}
        counts: Dict[str, int] = {name: 0 for name in targets}
        total = 0

        for topic, topic_items in sorted(by_topic.items()):
            groups: Dict[str, List[BenchmarkItem]] = defaultdict(list)
            for item in topic_items:
                groups[item.relevant_doc_ids[0]].append(item)
            group_keys = sorted(groups)
            random.Random(f"{self.config.seed}-{topic}").shuffle(group_keys)

            for key in group_keys:
                members = groups[key]
                if key in doc_assignment:
                    split_name = doc_assignment[key]      # a doc seen under another topic keeps its split
                else:
                    denominator = max(total, 1)
                    deficits = {
                        name: targets[name] - counts[name] / denominator for name in targets
                    }
                    split_name = max(deficits, key=lambda name: deficits[name])
                    doc_assignment[key] = split_name
                counts[split_name] += len(members)
                total += len(members)
                for item in members:
                    item.split = split_name
                    assigned.append(item)
        return assigned


def build_benchmark() -> List[Dict[str, Any]]:
    builder = BenchmarkBuilder(CONFIG, CORPUS)
    items = builder.split(builder.build())
    return [item.to_dict() for item in items]


set_global_seeds(CONFIG.seed)
BENCHMARK_RECORDS: List[Dict[str, Any]] = CACHE.get_or_build(
    PATHS.benchmark / f"benchmark_qa_{CORPUS_EVAL_FINGERPRINT}.json",
    build_benchmark,
    fmt="json",
    label=f"benchmark_qa_{CORPUS_EVAL_FINGERPRINT}.json",
)
BENCHMARK: List[BenchmarkItem] = [BenchmarkItem(**record) for record in BENCHMARK_RECORDS]
BENCHMARK_FRAME = pd.DataFrame(BENCHMARK_RECORDS)

SPLITS: Dict[str, List[BenchmarkItem]] = {
    name: [item for item in BENCHMARK if item.split == name]
    for name in ("train", "validation", "test")
}

print("=" * 78)
print("BENCHMARK QA DATASET")
print("=" * 78)
print(f"  total questions: {len(BENCHMARK)}")
_targets = {"train": CONFIG.train_ratio, "validation": CONFIG.val_ratio, "test": CONFIG.test_ratio}
for name, items in SPLITS.items():
    share = 100 * len(items) / max(len(BENCHMARK), 1)
    print(f"  {name:<11}: {len(items):>4} questions ({share:4.1f}%, target {100 * _targets[name]:4.1f}%)")
if not BENCHMARK_FRAME.empty:
    print("\nBy generator:")
    print(BENCHMARK_FRAME["generator"].value_counts().to_string())
    print("\nBy topic (top 8):")
    print(BENCHMARK_FRAME["topic"].value_counts().head(8).to_string())
    print("\nLeakage check — documents shared between splits:")
    doc_sets = {
        name: {doc for item in items for doc in item.relevant_doc_ids[:1]}
        for name, items in SPLITS.items()
    }
    print(f"  train ∩ test       : {len(doc_sets['train'] & doc_sets['test'])}")
    print(f"  validation ∩ test  : {len(doc_sets['validation'] & doc_sets['test'])}")
    print("\nExample items:")
    for item in BENCHMARK[:3]:
        print(f"  [{item.split}/{item.generator}] {item.question}")
        print(f"      relevant chunks: {len(item.relevant_chunk_ids)} | expected: {item.expected_answer[:110]} …")

18:56:53 | INFO    | psych-rag | cache MISS → building benchmark_qa_00f98c7248_a1c407c885_b2.json
18:56:55 | INFO    | psych-rag | built benchmark_qa_00f98c7248_a1c407c885_b2.json in 1.8s


BENCHMARK QA DATASET
  total questions: 480
  train      :  329 questions (68.5%, target 70.0%)
  validation :   69 questions (14.4%, target 15.0%)
  test       :   82 questions (17.1%, target 15.0%)

By generator:
generator
template          216
keyphrase         170
title_anchored     94

By topic (top 8):
topic
adhd                       50
anxiety                    44
bipolar_disorder           44
suicide_prevention         44
ocd                        41
substance_use_disorders    39
depression                 39
autism_neurodevelopment    38

Leakage check — documents shared between splits:
  train ∩ test       : 0
  validation ∩ test  : 0

Example items:
  [train/title_anchored] According to 'Prevalence of attention deficit hyperactivity disorder in adults: Umbrella review of evidence generated across' (NCBI / U.S. National Library of Medicine), what is reported under 'Results' regarding attention-deficit/hyperactivity disorder?
      relevant chunks: 1 | expected: Five system

---
## 9. Embedding Generation

**Model:** `BAAI/bge-large-en-v1.5` (1024-d), with automatic fallback to `BAAI/bge-base-en-v1.5` (768-d) when
GPU memory is tight or the large model fails to load.

Implementation details that matter for retrieval quality:
* **Asymmetric encoding.** BGE models expect queries to carry the instruction prefix
  `"Represent this sentence for searching relevant passages: "` while passages are encoded bare. Getting this
  wrong silently costs several points of Recall@k.
* **L2 normalisation.** Vectors are unit-normalised so inner product ≡ cosine similarity, which lets FAISS use
  the fast `IndexFlatIP`.
* **Batching + fp16** on GPU with adaptive batch size, and a full progress bar.
* Embeddings are cached as a single `float32` matrix aligned row-for-row with `CORPUS`.

In [16]:
# =============================================================================
# SECTION 9 — EMBEDDING MODEL
# =============================================================================


class EmbeddingModel:
    """Sentence-embedding wrapper with automatic model fallback and caching."""

    def __init__(self, config: RAGConfig, device: str = DEVICE) -> None:
        self.config = config
        self.device = device
        self.model_name: str = ""
        self.dimension: int = 0
        self.model = None
        self.load_seconds: float = 0.0
        self._load()

    def _candidate_models(self) -> List[str]:
        """Prefer the large model, but not on a GPU that cannot comfortably hold it."""
        if self.device == "cuda" and GPU_MEMORY_GB < self.config.embedding_model_min_gpu_gb:
            logger.info(
                "GPU memory %.1f GB < %.1f GB threshold → preferring %s",
                GPU_MEMORY_GB, self.config.embedding_model_min_gpu_gb, self.config.embedding_model_fallback,
            )
            return [self.config.embedding_model_fallback, self.config.embedding_model_primary]
        if self.device == "cpu":
            return [self.config.embedding_model_fallback, self.config.embedding_model_primary]
        return [self.config.embedding_model_primary, self.config.embedding_model_fallback]

    def _load(self) -> None:
        from sentence_transformers import SentenceTransformer

        started = time.perf_counter()
        for name in self._candidate_models():
            try:
                logger.info("loading embedding model %s on %s", name, self.device)
                model = SentenceTransformer(name, device=self.device)
                model.max_seq_length = self.config.embedding_max_seq_length
                self.model = model
                self.model_name = name
                self.dimension = int(model.get_sentence_embedding_dimension())
                self.load_seconds = time.perf_counter() - started
                logger.info("embedding model ready: %s (dim=%d, %.1fs)", name, self.dimension, self.load_seconds)
                return
            except Exception as exc:
                log_error("embedding_load", exc, name)
                torch.cuda.empty_cache() if self.device == "cuda" else None
        raise RuntimeError("Could not load any embedding model — check the network connection.")

    @property
    def batch_size(self) -> int:
        if self.device != "cuda":
            return max(8, self.config.embedding_batch_size // 4)
        if GPU_MEMORY_GB >= 22:
            return self.config.embedding_batch_size * 2
        return self.config.embedding_batch_size

    def encode_passages(self, texts: Sequence[str], show_progress: bool = True) -> np.ndarray:
        """Encode corpus passages (no instruction prefix) as unit vectors."""
        vectors = self.model.encode(
            list(texts),
            batch_size=self.batch_size,
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return np.asarray(vectors, dtype="float32")

    def encode_queries(self, queries: Sequence[str], show_progress: bool = False) -> np.ndarray:
        """Encode queries with the BGE retrieval instruction prefix."""
        prefixed = [f"{self.config.query_instruction}{q}" for q in queries]
        vectors = self.model.encode(
            prefixed,
            batch_size=min(self.batch_size, 64),
            show_progress_bar=show_progress,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return np.asarray(vectors, dtype="float32")


set_global_seeds(CONFIG.seed)
EMBEDDER = EmbeddingModel(CONFIG)

EMBEDDING_TIMING: Dict[str, float] = {}


def build_embeddings() -> np.ndarray:
    started = time.perf_counter()
    vectors = EMBEDDER.encode_passages(CHUNK_TEXTS, show_progress=True)
    EMBEDDING_TIMING["seconds"] = time.perf_counter() - started
    EMBEDDING_TIMING["chunks"] = len(CHUNK_TEXTS)
    (PATHS.embeddings / "embedding_meta.json").write_text(
        json.dumps(
            {
                "model": EMBEDDER.model_name,
                "dimension": EMBEDDER.dimension,
                "chunks": len(CHUNK_TEXTS),
                "seconds": round(EMBEDDING_TIMING["seconds"], 2),
                "device": DEVICE,
                "batch_size": EMBEDDER.batch_size,
            },
            indent=2,
        ),
        encoding="utf-8",
    )
    return vectors


EMBEDDINGS_PATH = PATHS.embeddings / f"chunk_embeddings_{EMBEDDER.dimension}_{CORPUS_TEXT_FINGERPRINT}.npy"
EMBEDDINGS: np.ndarray = CACHE.get_or_build(
    EMBEDDINGS_PATH, build_embeddings, fmt="numpy", label=EMBEDDINGS_PATH.name
)
EMBEDDINGS = np.ascontiguousarray(np.asarray(EMBEDDINGS, dtype="float32"))

# Guard against a stale cache after the corpus changes.
if EMBEDDINGS.shape[0] != len(CORPUS):
    logger.warning("embedding/corpus mismatch (%d vs %d) — rebuilding", EMBEDDINGS.shape[0], len(CORPUS))
    EMBEDDINGS = np.ascontiguousarray(build_embeddings())
    np.save(EMBEDDINGS_PATH, EMBEDDINGS)

meta_path = PATHS.embeddings / "embedding_meta.json"
if meta_path.exists():
    EMBEDDING_TIMING.update(json.loads(meta_path.read_text(encoding="utf-8")))

print("=" * 78)
print("EMBEDDINGS")
print("=" * 78)
print(f"  model     : {EMBEDDER.model_name}")
print(f"  dimension : {EMBEDDER.dimension}")
print(f"  matrix    : {EMBEDDINGS.shape} ({EMBEDDINGS.nbytes / 1024**2:.1f} MB)")
print(f"  device    : {DEVICE} | batch size {EMBEDDER.batch_size}")
print(f"  norm check: mean L2 = {np.linalg.norm(EMBEDDINGS[:200], axis=1).mean():.4f} (expected 1.0000)")
if EMBEDDING_TIMING.get("seconds"):
    rate = EMBEDDING_TIMING["chunks"] / max(EMBEDDING_TIMING["seconds"], 1e-6)
    print(f"  throughput: {rate:.1f} chunks/second ({EMBEDDING_TIMING['seconds']:.1f}s total)")

18:56:55 | INFO    | psych-rag | loading embedding model BAAI/bge-large-en-v1.5 on cuda
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
18:56:56 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
18:56:56 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
18:56:56 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
18:56:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/README.md "HTTP/1.1 200 OK"
18:56:56 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/README.md "HTTP/1.1 200 OK"


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

18:57:12 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
18:57:12 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
18:57:12 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
18:57:12 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

18:57:12 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5 "HTTP/1.1 200 OK"
18:57:13 | INFO    | psych-rag | embedding model ready: BAAI/bge-large-en-v1.5 (dim=1024, 17.7s)
18:57:15 | INFO    | psych-rag | cache HIT  → chunk_embeddings_1024_00f98c7248.npy


EMBEDDINGS
  model     : BAAI/bge-large-en-v1.5
  dimension : 1024
  matrix    : (5762, 1024) (22.5 MB)
  device    : cuda | batch size 32
  norm check: mean L2 = 1.0000 (expected 1.0000)
  throughput: 13.7 chunks/second (421.3s total)


---
## 10. FAISS Index & 11. BM25 Index

**Dense — FAISS.** With unit-normalised vectors, `IndexFlatIP` gives *exact* cosine search. At this corpus
scale (10³–10⁵ chunks) exact search is both faster to build and more accurate than an approximate index; the
class below still switches to `IndexIVFFlat` automatically above 50 000 vectors, which is where ANN starts to
pay off.

**Sparse — BM25.** Dense retrieval is weak exactly where clinical language is strongest: rare drug names,
ICD-11 codes, scale acronyms (`PHQ-9`, `HAM-D`, `6B00`). BM25 over a lightly normalised token stream covers
that gap, and the hybrid fusion in Section 12 combines the two.

In [17]:
# =============================================================================
# SECTION 10 — FAISS DENSE INDEX
# =============================================================================
import faiss  # noqa: E402  (installed in Section 2)


class FaissIndex:
    """Persistent FAISS index over unit-normalised chunk embeddings."""

    IVF_THRESHOLD = 50_000

    def __init__(self, dimension: int, path: Path) -> None:
        self.dimension = dimension
        self.path = path
        self.index: Optional[faiss.Index] = None
        self.index_type: str = "none"

    def build(self, vectors: np.ndarray) -> "FaissIndex":
        n_vectors = vectors.shape[0]
        if n_vectors > self.IVF_THRESHOLD:
            n_lists = int(max(64, min(4096, 4 * math.sqrt(n_vectors))))
            quantizer = faiss.IndexFlatIP(self.dimension)
            index = faiss.IndexIVFFlat(quantizer, self.dimension, n_lists, faiss.METRIC_INNER_PRODUCT)
            index.train(vectors)
            index.nprobe = max(8, n_lists // 32)
            self.index_type = f"IndexIVFFlat(nlist={n_lists}, nprobe={index.nprobe})"
        else:
            index = faiss.IndexFlatIP(self.dimension)     # exact cosine search
            self.index_type = "IndexFlatIP (exact)"
        index.add(vectors)
        self.index = index
        return self

    def save(self) -> None:
        try:
            faiss.write_index(self.index, str(self.path))
        except Exception as exc:
            log_error("faiss_save", exc, str(self.path))

    def load(self) -> bool:
        if not self.path.exists():
            return False
        try:
            self.index = faiss.read_index(str(self.path))
            self.index_type = f"{type(self.index).__name__} (loaded)"
            return True
        except Exception as exc:
            log_error("faiss_load", exc, str(self.path))
            return False

    def search(self, query_vectors: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
        """Return (scores, row_ids); scores are cosine similarities in [-1, 1]."""
        query_vectors = np.ascontiguousarray(query_vectors.astype("float32"))
        scores, indices = self.index.search(query_vectors, min(k, self.index.ntotal))
        return scores, indices

    @property
    def size(self) -> int:
        return int(self.index.ntotal) if self.index is not None else 0


FAISS_PATH = PATHS.indices / f"faiss_{EMBEDDER.dimension}_{CORPUS_TEXT_FINGERPRINT}.index"
FAISS_INDEX = FaissIndex(EMBEDDER.dimension, FAISS_PATH)

if CONFIG.force_rebuild or not FAISS_INDEX.load() or FAISS_INDEX.size != len(CORPUS):
    logger.info("building FAISS index over %d vectors", len(EMBEDDINGS))
    started = time.perf_counter()
    FAISS_INDEX.build(EMBEDDINGS)
    FAISS_INDEX.save()
    logger.info("FAISS build finished in %.2fs", time.perf_counter() - started)

print(f"FAISS index: {FAISS_INDEX.index_type} | vectors = {FAISS_INDEX.size:,} | dim = {EMBEDDER.dimension}")

# Smoke test: the nearest neighbour of a chunk's own embedding must be itself.
_probe = EMBEDDINGS[:1]
_scores, _ids = FAISS_INDEX.search(_probe, 3)
print(f"Self-retrieval sanity check → top id {_ids[0][0]} (expected 0), score {_scores[0][0]:.4f} (expected ~1.0)")

FAISS index: IndexFlatIP (loaded) | vectors = 5,762 | dim = 1024
Self-retrieval sanity check → top id 0 (expected 0), score 1.0000 (expected ~1.0)


In [18]:
# =============================================================================
# SECTION 11 — BM25 SPARSE INDEX
# =============================================================================
from rank_bm25 import BM25Okapi  # noqa: E402


class BM25Index:
    """Persistent BM25-Okapi index with clinically-aware tokenisation.

    The tokeniser deliberately preserves hyphenated drug names, ICD codes and
    instrument acronyms (e.g. `bge-large`, `6b00`, `phq-9`) because those are the
    queries where sparse retrieval outperforms dense retrieval.
    """

    TOKEN_PATTERN = re.compile(r"[a-z0-9][a-z0-9\-\.]{1,}")
    STOPWORDS: Set[str] = set(
        """a an the and or but if of to in on for with without from by as at is are was were be been being
        this that these those it its their there we our they he she his her not no than then so such can may
        might should could would will have has had do does did between among during also into over under
        about more most other some any each per via""".split()
    )

    def __init__(self, path: Path) -> None:
        self.path = path
        self.bm25: Optional[BM25Okapi] = None
        self.n_documents: int = 0

    @classmethod
    def tokenize(cls, text: str) -> List[str]:
        tokens = cls.TOKEN_PATTERN.findall(text.lower())
        return [t.strip(".") for t in tokens if t not in cls.STOPWORDS and len(t) > 1]

    def build(self, texts: Sequence[str]) -> "BM25Index":
        corpus_tokens = [self.tokenize(text) for text in tqdm(texts, desc="Tokenising for BM25", leave=False)]
        self.bm25 = BM25Okapi(corpus_tokens)
        self.n_documents = len(corpus_tokens)
        return self

    def save(self) -> None:
        try:
            with self.path.open("wb") as handle:
                pickle.dump({"bm25": self.bm25, "n_documents": self.n_documents}, handle,
                            protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as exc:
            log_error("bm25_save", exc, str(self.path))

    def load(self) -> bool:
        if not self.path.exists():
            return False
        try:
            with self.path.open("rb") as handle:
                payload = pickle.load(handle)
            self.bm25 = payload["bm25"]
            self.n_documents = payload["n_documents"]
            return True
        except Exception as exc:
            log_error("bm25_load", exc, str(self.path))
            return False

    def search(self, query: str, k: int) -> Tuple[np.ndarray, np.ndarray]:
        """Return (scores, row_ids) for the top-k BM25 matches."""
        scores = np.asarray(self.bm25.get_scores(self.tokenize(query)), dtype="float32")
        k = min(k, len(scores))
        top = np.argpartition(-scores, k - 1)[:k] if k < len(scores) else np.arange(len(scores))
        order = top[np.argsort(-scores[top])]
        return scores[order], order


BM25_PATH = PATHS.indices / f"bm25_index_{CORPUS_TEXT_FINGERPRINT}.pkl"
BM25_INDEX = BM25Index(BM25_PATH)

if CONFIG.force_rebuild or not BM25_INDEX.load() or BM25_INDEX.n_documents != len(CORPUS):
    logger.info("building BM25 index over %d chunks", len(CHUNK_TEXTS))
    started = time.perf_counter()
    BM25_INDEX.build(CHUNK_TEXTS)
    BM25_INDEX.save()
    logger.info("BM25 build finished in %.2fs", time.perf_counter() - started)

print(f"BM25 index: {BM25_INDEX.n_documents:,} documents indexed")
_bm25_scores, _bm25_ids = BM25_INDEX.search("pharmacological treatment of generalised anxiety disorder", 3)
print("BM25 sanity check — top match:")
if len(_bm25_ids):
    print(f"  score {_bm25_scores[0]:.2f} | {CORPUS.iloc[int(_bm25_ids[0])]['title'][:80]}")

BM25 index: 5,762 documents indexed
BM25 sanity check — top match:
  score 16.27 | Generalized Anxiety Disorder 7-item (GAD-7) and 2-item (GAD-2) scales for detect


---
## 12. Hybrid Retrieval

Dense and sparse scores live on incompatible scales (cosine ∈ [-1, 1] vs unbounded BM25), so they are
**min–max normalised over the union of candidates** before fusion:

```
score(c) = w_dense · norm(cosine(q, c)) + w_bm25 · norm(bm25(q, c))
```

Weights default to `0.5 / 0.5`, which is what the validation sweep in Section 18 selected — clinical queries
lean more heavily on lexical precision (drug names, ICD codes, instrument acronyms like `PHQ-9`) than the
usual `0.7/0.3` dense-favouring rule of thumb would suggest. Both weights are configurable at query time.

Documents missing from one retriever's candidate list receive that retriever's minimum observed score rather
than zero, which prevents a strong dense hit from being unfairly penalised for being lexically unusual.

In [19]:
# =============================================================================
# SECTION 12 — HYBRID RETRIEVAL
# =============================================================================


@dataclass
class RetrievedChunk:
    """One retrieval result with full explainability payload."""

    row_id: int
    chunk_id: str
    text: str
    dense_score: float = 0.0
    bm25_score: float = 0.0
    hybrid_score: float = 0.0
    rerank_score: Optional[float] = None
    final_score: float = 0.0
    rank: int = 0
    metadata: Dict[str, Any] = field(default_factory=dict)

    def citation(self) -> str:
        year = self.metadata.get("year")
        year_text = int(year) if isinstance(year, (int, float)) and not pd.isna(year) else "n.d."
        return (
            f"{self.metadata.get('organization', 'Unknown')} ({year_text}). "
            f"{self.metadata.get('title', 'Untitled')} — {self.metadata.get('section_heading', '')}"
        ).strip(" —")

    def to_row(self) -> Dict[str, Any]:
        return {
            "rank": self.rank,
            "chunk_id": self.chunk_id,
            "final_score": round(self.final_score, 4),
            "hybrid": round(self.hybrid_score, 4),
            "dense": round(self.dense_score, 4),
            "bm25": round(self.bm25_score, 4),
            "rerank": None if self.rerank_score is None else round(self.rerank_score, 4),
            "source": self.metadata.get("source"),
            "organization": self.metadata.get("organization"),
            "year": self.metadata.get("year"),
            "topic": self.metadata.get("topic"),
            "section": self.metadata.get("section"),
            "title": str(self.metadata.get("title", ""))[:90],
            "url": self.metadata.get("url"),
        }


def min_max_normalize(values: np.ndarray) -> np.ndarray:
    """Scale to [0, 1]; a constant vector maps to all-ones."""
    if values.size == 0:
        return values
    low, high = float(values.min()), float(values.max())
    if high - low < 1e-9:
        return np.ones_like(values, dtype="float32")
    return ((values - low) / (high - low)).astype("float32")


class HybridRetriever:
    """Weighted fusion of FAISS dense search and BM25 sparse search."""

    def __init__(
        self,
        config: RAGConfig,
        corpus: pd.DataFrame,
        embedder: EmbeddingModel,
        faiss_index: FaissIndex,
        bm25_index: BM25Index,
    ) -> None:
        self.config = config
        self.corpus = corpus
        self.embedder = embedder
        self.faiss_index = faiss_index
        self.bm25_index = bm25_index
        self._metadata_columns = [
            "chunk_id", "doc_id", "source", "organization", "title", "year",
            "topic", "section", "section_heading", "document_type", "url", "n_tokens", "license_note",
        ]
        self._metadata_cache: List[Dict[str, Any]] = corpus[self._metadata_columns].to_dict("records")

    # -- single-retriever paths ----------------------------------------------
    def dense_search(self, query: str, k: int) -> Dict[int, float]:
        vector = self.embedder.encode_queries([query])
        scores, indices = self.faiss_index.search(vector, k)
        return {int(row): float(score) for row, score in zip(indices[0], scores[0]) if row >= 0}

    def sparse_search(self, query: str, k: int) -> Dict[int, float]:
        scores, indices = self.bm25_index.search(query, k)
        return {int(row): float(score) for row, score in zip(indices, scores)}

    # -- fusion ---------------------------------------------------------------
    def retrieve(
        self,
        query: str,
        k: Optional[int] = None,
        candidates: Optional[int] = None,
        dense_weight: Optional[float] = None,
        bm25_weight: Optional[float] = None,
        mode: str = "hybrid",
    ) -> List[RetrievedChunk]:
        """Retrieve the top-`k` chunks for `query`.

        Parameters
        ----------
        mode : {"hybrid", "dense", "bm25"}
            Selecting a single retriever enables the ablations reported in Section 18.
        """
        k = k or self.config.top_k_default
        candidates = candidates or max(self.config.retrieve_candidates, k)
        dense_weight = self.config.dense_weight if dense_weight is None else dense_weight
        bm25_weight = self.config.bm25_weight if bm25_weight is None else bm25_weight

        dense_hits = self.dense_search(query, candidates) if mode in ("hybrid", "dense") else {}
        sparse_hits = self.sparse_search(query, candidates) if mode in ("hybrid", "bm25") else {}

        pool = sorted(set(dense_hits) | set(sparse_hits))
        if not pool:
            return []

        dense_raw = np.array([dense_hits.get(row, min(dense_hits.values(), default=0.0)) for row in pool], dtype="float32")
        sparse_raw = np.array([sparse_hits.get(row, min(sparse_hits.values(), default=0.0)) for row in pool], dtype="float32")
        dense_norm = min_max_normalize(dense_raw) if dense_hits else np.zeros(len(pool), dtype="float32")
        sparse_norm = min_max_normalize(sparse_raw) if sparse_hits else np.zeros(len(pool), dtype="float32")

        if mode == "dense":
            fused = dense_norm
        elif mode == "bm25":
            fused = sparse_norm
        else:
            total = max(dense_weight + bm25_weight, 1e-9)
            fused = (dense_weight * dense_norm + bm25_weight * sparse_norm) / total

        order = np.argsort(-fused)[:k]
        results: List[RetrievedChunk] = []
        for rank, position in enumerate(order, start=1):
            row_id = pool[int(position)]
            metadata = dict(self._metadata_cache[row_id])
            results.append(
                RetrievedChunk(
                    row_id=row_id,
                    chunk_id=metadata["chunk_id"],
                    text=self.corpus.iat[row_id, self.corpus.columns.get_loc("text")],
                    dense_score=float(dense_raw[position]),
                    bm25_score=float(sparse_raw[position]),
                    hybrid_score=float(fused[position]),
                    final_score=float(fused[position]),
                    rank=rank,
                    metadata=metadata,
                )
            )
        return results


RETRIEVER = HybridRetriever(CONFIG, CORPUS, EMBEDDER, FAISS_INDEX, BM25_INDEX)

_demo_query = "What pharmacological treatments are recommended for generalised anxiety disorder?"
_demo_results = RETRIEVER.retrieve(_demo_query, k=5)
print(f"Query: {_demo_query}\n")
print(pd.DataFrame([r.to_row() for r in _demo_results])[
    ["rank", "final_score", "dense", "bm25", "source", "section", "title"]
].to_string(index=False))

Query: What pharmacological treatments are recommended for generalised anxiety disorder?

 rank  final_score  dense    bm25 source    section                                                                                      title
    1       0.6126 0.7120 19.5962 pubmed background                                                                                   Anxiety.
    2       0.6069 0.7515 15.8799 pubmed  treatment Psychotherapies for Generalized Anxiety Disorder in Adults: A Systematic Review and Networ
    3       0.5982 0.7781 13.3025 pubmed background Brazilian Psychiatric Association treatment guidelines for generalized anxiety disorder: p
    4       0.5000 0.7846 11.0269 pubmed background                                                     Pharmacotherapy for Anxiety Disorders.
    5       0.4946 0.6909 19.5032 pubmed    results Generalized Anxiety Disorder 7-item (GAD-7) and 2-item (GAD-2) scales for detecting anxiet


---
## 13. Cross-Encoder Reranking

Bi-encoders embed the query and passage independently — fast, but blind to fine-grained interaction. A
cross-encoder scores the `(query, passage)` **pair jointly**, which is far more accurate and affordable when
applied only to a shortlist.

Pipeline: retrieve 50 candidates → cross-encode all 50 → keep top-5 / top-10 / top-20.

`BAAI/bge-reranker-large` is used when GPU memory allows, otherwise `bge-reranker-base`; if neither loads, the
system transparently falls back to hybrid ordering (and says so in the UI). Section 18 quantifies exactly what
reranking buys, per depth.

In [20]:
# =============================================================================
# SECTION 13 — CROSS-ENCODER RERANKER
# =============================================================================


class CrossEncoderReranker:
    """Joint (query, passage) scoring with graceful degradation."""

    def __init__(self, config: RAGConfig, device: str = DEVICE) -> None:
        self.config = config
        self.device = device
        self.model = None
        self.model_name: str = "disabled"
        self.available: bool = False
        if config.reranker_enabled:
            self._load()

    def _candidate_models(self) -> List[str]:
        if self.device == "cuda" and GPU_MEMORY_GB >= self.config.reranker_min_gpu_gb:
            return [self.config.reranker_model_primary, self.config.reranker_model_fallback]
        return [self.config.reranker_model_fallback, self.config.reranker_model_primary]

    def _load(self) -> None:
        from sentence_transformers import CrossEncoder

        for name in self._candidate_models():
            try:
                logger.info("loading cross-encoder %s on %s", name, self.device)
                self.model = CrossEncoder(name, device=self.device, max_length=512)
                self.model_name = name
                self.available = True
                logger.info("cross-encoder ready: %s", name)
                return
            except Exception as exc:
                log_error("reranker_load", exc, name)
        logger.warning("no cross-encoder available — falling back to hybrid ordering")

    @staticmethod
    def _sigmoid(x: np.ndarray) -> np.ndarray:
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def rerank(
        self,
        query: str,
        results: Sequence[RetrievedChunk],
        top_k: Optional[int] = None,
    ) -> List[RetrievedChunk]:
        """Rescore and reorder `results`; returns the top-`top_k` chunks."""
        top_k = top_k or self.config.top_k_default
        if not results:
            return []
        if not self.available:
            trimmed = list(results)[:top_k]
            for rank, item in enumerate(trimmed, start=1):
                item.rank = rank
                item.final_score = item.hybrid_score
            return trimmed

        pairs = [(query, item.text[:2000]) for item in results]
        try:
            raw_scores = self.model.predict(
                pairs, batch_size=self.config.reranker_batch_size, show_progress_bar=False
            )
        except Exception as exc:
            log_error("rerank", exc, query[:120])
            return list(results)[:top_k]

        # BGE rerankers emit logits; a sigmoid maps them to an interpretable [0, 1].
        probabilities = self._sigmoid(np.asarray(raw_scores, dtype="float32").reshape(-1))
        for item, score in zip(results, probabilities):
            item.rerank_score = float(score)
            item.final_score = float(score)
        ordered = sorted(results, key=lambda r: r.final_score, reverse=True)[:top_k]
        for rank, item in enumerate(ordered, start=1):
            item.rank = rank
        return ordered


set_global_seeds(CONFIG.seed)
RERANKER = CrossEncoderReranker(CONFIG)

if RERANKER.available:
    _reranked = RERANKER.rerank(_demo_query, RETRIEVER.retrieve(_demo_query, k=CONFIG.retrieve_candidates), top_k=5)
    print(f"Reranker: {RERANKER.model_name}\n")
    print(pd.DataFrame([r.to_row() for r in _reranked])[
        ["rank", "rerank", "hybrid", "source", "section", "title"]
    ].to_string(index=False))
else:
    print("Reranker unavailable — the pipeline will use hybrid ordering (and report this in the UI).")

18:57:20 | INFO    | psych-rag | loading cross-encoder BAAI/bge-reranker-large on cuda
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/modules.json "HTTP/1.1 404 Not Found"
18:57:20 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large "HTTP/1.1 200 OK"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
18:57:20 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
18:57:20 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/config.json "HTTP/1.1 200 OK"
18:57:21 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/model.safetensors "HTTP/1.1 302 Found"


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer.json: downloading bytes:           |  0.00B            

18:57:45 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
18:57:45 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
18:57:45 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/special_tokens_map.json "HTTP/1.1 200 OK"
18:57:45 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-large/55611d7bca2a7133960a6d3b71e083071bbfc312/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

18:57:45 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-large/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
18:57:50 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-large "HTTP/1.1 200 OK"
18:57:52 | INFO    | psych-rag | cross-encoder ready: BAAI/bge-reranker-large


Reranker: BAAI/bge-reranker-large

 rank  rerank  hybrid source    section                                                                                      title
    1  0.7309  0.5000 pubmed background                                                     Pharmacotherapy for Anxiety Disorders.
    2  0.7294  0.1510 pubmed background                               Advances in Pharmacotherapy for Pediatric Anxiety Disorders.
    3  0.7293  0.3653 pubmed background                                                            Treatment of anxiety disorders.
    4  0.7290  0.5982 pubmed background Brazilian Psychiatric Association treatment guidelines for generalized anxiety disorder: p
    5  0.7290  0.2408 pubmed background Psychological and pharmacological treatments for generalized anxiety disorder (GAD): a met


---
## 14. Prompt Engineering

The prompt is the last line of defence against hallucination, so it is written as an explicit contract:

1. **Closed-book prohibition.** The model is told, twice and in different words, that its own knowledge is
   inadmissible — only the numbered evidence block may be used.
2. **Mandatory inline citations.** Every clinical claim must carry `[n]` markers pointing at the evidence
   block; uncited sentences are detected and penalised programmatically in Section 16.
3. **Explicit refusal channel.** A verbatim refusal string is provided so that "I don't know" is an *easy,
   well-specified* action rather than an improvised one — models hallucinate most when refusal feels costly.
4. **No new clinical entities.** Diagnoses, drug names and doses that do not appear in the evidence are
   forbidden outright.
5. **Safety routing.** Any question implying risk of harm triggers an urgent-help directive that is appended
   independently of the model's output, so it cannot be suppressed by generation.
6. **Structured output.** Answer → Evidence summary → Limitations, which keeps responses auditable and makes
   citation parsing reliable.

In [21]:
# =============================================================================
# SECTION 14 — PROMPT ENGINEERING
# =============================================================================

INSUFFICIENT_EVIDENCE_MESSAGE = "I could not find sufficient evidence in the indexed medical literature."

SAFETY_NOTICE = (
    "\n\n---\n**If you or someone you know may be at immediate risk of harm, contact your local emergency "
    "number now, or a crisis line such as 988 (US & Canada), 116 123 (Samaritans, UK & Ireland), or your "
    "nearest emergency department. This assistant is an information-retrieval demonstration and cannot "
    "provide crisis care.**"
)

RISK_PATTERNS = re.compile(
    r"\b(kill (my|him|her|them)self|suicid|self[- ]harm|end my life|overdose|hurt (myself|someone)|"
    r"want to die|take my (own )?life)\b",
    re.IGNORECASE,
)

SYSTEM_PROMPT = """You are an evidence-based psychiatry research assistant. You answer clinical questions \
strictly and exclusively from the numbered evidence passages supplied to you in each request.

NON-NEGOTIABLE RULES
1. Use ONLY the supplied evidence. Your own background knowledge is inadmissible, even if you are certain it \
is correct and even if the evidence seems incomplete.
2. Cite continuously. Every clinical statement must end with one or more citation markers such as [1] or \
[2][5], referring to the numbered passages.
3. Never invent. Do not state a diagnosis, medication, dose, prevalence figure, guideline recommendation or \
statistic that is not explicitly present in the evidence.
4. If the evidence does not answer the question, reply with exactly this sentence and nothing else:
"{refusal}"
5. Do not give individualised medical advice, and do not tell the user what they personally should do. \
Describe what the retrieved literature reports.
6. Prefer the most recent and highest-quality evidence, and say so when sources disagree.

OUTPUT FORMAT
**Answer:** 2-6 sentences answering the question directly, with inline [n] citations.
**Evidence summary:** 2-4 concise bullet points, each ending with its [n] citation.
**Limitations:** one sentence on what the retrieved evidence does not cover.""".format(
    refusal=INSUFFICIENT_EVIDENCE_MESSAGE
)


class PromptBuilder:
    """Assembles the grounded prompt sent to the LLM."""

    def __init__(self, config: RAGConfig, max_context_tokens: int = 3200) -> None:
        self.config = config
        self.max_context_tokens = max_context_tokens

    def build_context(self, results: Sequence[RetrievedChunk]) -> Tuple[str, List[RetrievedChunk]]:
        """Render the numbered evidence block, honouring a context-token budget."""
        blocks: List[str] = []
        used: List[RetrievedChunk] = []
        budget = self.max_context_tokens
        for index, item in enumerate(results, start=1):
            body = item.text.strip()
            cost = approx_token_count(body) + 60
            if cost > budget and used:
                break
            budget -= cost
            year = item.metadata.get("year")
            year_text = int(year) if isinstance(year, (int, float)) and not pd.isna(year) else "n.d."
            blocks.append(
                f"[{index}] SOURCE: {item.metadata.get('organization')} | "
                f"TYPE: {item.metadata.get('document_type')} | YEAR: {year_text} | "
                f"SECTION: {item.metadata.get('section')}\n"
                f"TITLE: {item.metadata.get('title')}\n"
                f"URL: {item.metadata.get('url')}\n"
                f"PASSAGE: {body}"
            )
            used.append(item)
        return "\n\n".join(blocks), used

    def build_messages(
        self, question: str, results: Sequence[RetrievedChunk]
    ) -> Tuple[List[Dict[str, str]], List[RetrievedChunk]]:
        context, used = self.build_context(results)
        user_message = (
            f"QUESTION:\n{question}\n\n"
            f"RETRIEVED EVIDENCE ({len(used)} passages):\n"
            f"-------------------------------------------\n{context}\n"
            f"-------------------------------------------\n\n"
            "Answer the question using only the evidence above, following the required output format. "
            f'If the evidence is insufficient, reply exactly: "{INSUFFICIENT_EVIDENCE_MESSAGE}"'
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]
        return messages, used

    @staticmethod
    def is_risk_query(question: str) -> bool:
        return bool(RISK_PATTERNS.search(question or ""))


PROMPT_BUILDER = PromptBuilder(CONFIG)

_messages, _used = PROMPT_BUILDER.build_messages(_demo_query, _demo_results)
print("SYSTEM PROMPT\n" + "-" * 78)
print(SYSTEM_PROMPT)
print("\nUSER MESSAGE (truncated preview)\n" + "-" * 78)
print(_messages[1]["content"][:1100] + "\n…")

SYSTEM PROMPT
------------------------------------------------------------------------------
You are an evidence-based psychiatry research assistant. You answer clinical questions strictly and exclusively from the numbered evidence passages supplied to you in each request.

NON-NEGOTIABLE RULES
1. Use ONLY the supplied evidence. Your own background knowledge is inadmissible, even if you are certain it is correct and even if the evidence seems incomplete.
2. Cite continuously. Every clinical statement must end with one or more citation markers such as [1] or [2][5], referring to the numbered passages.
3. Never invent. Do not state a diagnosis, medication, dose, prevalence figure, guideline recommendation or statistic that is not explicitly present in the evidence.
4. If the evidence does not answer the question, reply with exactly this sentence and nothing else:
"I could not find sufficient evidence in the indexed medical literature."
5. Do not give individualised medical advice, and do

---
## 15. LLM Integration

The generator is loaded through a **capability cascade** rather than a hard-coded model, so the notebook runs
on a T4, an A100, or CPU-only without edits:

| Condition | Model |
|---|---|
| GPU ≥ 13 GB + bitsandbytes | `Qwen/Qwen3-8B` in NF4 4-bit (double quantisation, bf16 compute) |
| GPU ≥ 13 GB, no bitsandbytes | `Qwen/Qwen2.5-7B-Instruct` in fp16 |
| Smaller GPU (e.g. T4 15 GB busy) | `Qwen/Qwen2.5-3B-Instruct` |
| CPU-only / all loads fail | **Extractive fallback generator** — composes a fully-cited answer directly from retrieved sentences |

The extractive fallback is not a toy: because it only copies retrieved sentences and attaches their citation
markers, it is *hallucination-free by construction*, and it keeps every downstream evaluation cell runnable
even without a GPU.

In [22]:
# =============================================================================
# SECTION 15 — LLM ENGINE
# =============================================================================


@dataclass
class GenerationResult:
    """Raw generation output plus accounting metadata."""

    text: str
    prompt_tokens: int
    completion_tokens: int
    seconds: float
    model_name: str
    backend: str

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.completion_tokens


class ExtractiveGenerator:
    """Citation-safe fallback generator.

    Builds an answer by selecting the highest-scoring retrieved sentences and
    attaching their source markers. It can only ever restate retrieved text, so
    it cannot hallucinate.
    """

    model_name = "extractive-fallback"
    backend = "extractive"

    def __init__(self, embedder: Optional[EmbeddingModel] = None) -> None:
        self.embedder = embedder

    @staticmethod
    def _sentences(text: str) -> List[str]:
        body = text.split("\n", 1)[-1]
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", body) if len(s.strip()) > 45]

    def generate(self, question: str, results: Sequence[RetrievedChunk], **_: Any) -> GenerationResult:
        started = time.perf_counter()
        candidates: List[Tuple[str, int]] = []
        for index, item in enumerate(results[:6], start=1):
            for sentence in self._sentences(item.text)[:4]:
                candidates.append((sentence, index))
        if not candidates:
            return GenerationResult(INSUFFICIENT_EVIDENCE_MESSAGE, 0, 0,
                                    time.perf_counter() - started, self.model_name, self.backend)

        selected = candidates[:5]
        if self.embedder is not None and len(candidates) > 5:
            try:
                query_vector = self.embedder.encode_queries([question])[0]
                sentence_vectors = self.embedder.encode_passages([c[0] for c in candidates], show_progress=False)
                ranking = np.argsort(-(sentence_vectors @ query_vector))[:5]
                selected = [candidates[int(i)] for i in ranking]
            except Exception as exc:
                log_error("extractive_rank", exc, question[:100])

        answer_lines = [f"{sentence} [{index}]" for sentence, index in selected[:3]]
        bullet_lines = [f"- {sentence} [{index}]" for sentence, index in selected[:4]]
        text = (
            "**Answer:** " + " ".join(answer_lines) + "\n\n"
            "**Evidence summary:**\n" + "\n".join(bullet_lines) + "\n\n"
            "**Limitations:** This answer was assembled extractively from the retrieved passages, so it "
            "reflects only what those passages state."
        )
        return GenerationResult(
            text=text,
            prompt_tokens=approx_token_count(question) + sum(approx_token_count(r.text) for r in results[:6]),
            completion_tokens=approx_token_count(text),
            seconds=time.perf_counter() - started,
            model_name=self.model_name,
            backend=self.backend,
        )


class LLMEngine:
    """Loads the strongest open-weight instruction model that fits the runtime."""

    def __init__(self, config: RAGConfig, embedder: Optional[EmbeddingModel] = None) -> None:
        self.config = config
        self.model = None
        self.tokenizer = None
        self.model_name: str = "none"
        self.backend: str = "none"
        self.available: bool = False
        self.fallback = ExtractiveGenerator(embedder)
        if config.llm_enabled:
            self._load()
        else:
            logger.info("LLM disabled by configuration — using the extractive generator.")

    # -- loading --------------------------------------------------------------
    def _quantization_config(self):
        if not (self.config.llm_use_4bit and DEVICE == "cuda" and CAPABILITIES.get("bitsandbytes")):
            return None
        from transformers import BitsAndBytesConfig

        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        )

    @staticmethod
    def _is_locally_cached(name: str) -> bool:
        """True when every shard of `name` is already in the local HF cache."""
        try:
            from huggingface_hub import snapshot_download

            snapshot_download(name, local_files_only=True)
            return True
        except Exception:
            return False

    def _feasible(self, name: str) -> bool:
        """Can this runtime actually run the model?"""
        large = ("8B" in name) or ("7B" in name)
        if DEVICE != "cuda":
            return ("1.5B" in name) or ("3B" in name)
        if large and GPU_MEMORY_GB < self.config.llm_min_gpu_gb_for_7b and not CAPABILITIES.get("bitsandbytes"):
            return False
        return True

    def _candidates(self) -> List[str]:
        """Order candidates: already-downloaded first, then smallest-viable first.

        A cold Colab runtime starts with an empty Hugging Face cache. Re-fetching a
        16 GB checkpoint at throttled speed dominates total runtime, so a model
        that is already on disk always wins, and the default cascade is sized to
        download quickly rather than to maximise parameter count.
        """
        pool = list(self.config.llm_candidates)
        if self.config.llm_prefer_large:
            pool = list(self.config.llm_large_candidates) + pool
        pool = [name for name in pool if self._feasible(name)] or list(self.config.llm_candidates)[-1:]

        # A large model that is already cached costs nothing extra to load.
        cached = [
            name
            for name in list(self.config.llm_large_candidates) + pool
            if self._feasible(name) and self._is_locally_cached(name)
        ]
        if cached:
            logger.info("locally cached LLM(s) available: %s", ", ".join(cached))
        return list(dict.fromkeys(cached + pool))

    def _load(self) -> None:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        quantization = self._quantization_config()
        for name in self._candidates():
            try:
                logger.info("loading LLM %s (4-bit=%s)", name, quantization is not None)
                tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
                kwargs: Dict[str, Any] = {"trust_remote_code": True, "low_cpu_mem_usage": True}
                if DEVICE == "cuda":
                    kwargs["device_map"] = "auto"
                    if quantization is not None:
                        kwargs["quantization_config"] = quantization
                        self.backend = "transformers-4bit"
                    else:
                        kwargs["torch_dtype"] = torch.float16
                        self.backend = "transformers-fp16"
                else:
                    kwargs["torch_dtype"] = torch.float32
                    self.backend = "transformers-cpu"
                model = AutoModelForCausalLM.from_pretrained(name, **kwargs)
                model.eval()
                if tokenizer.pad_token_id is None:
                    tokenizer.pad_token = tokenizer.eos_token
                self.model, self.tokenizer, self.model_name, self.available = model, tokenizer, name, True
                logger.info("LLM ready: %s via %s", name, self.backend)
                return
            except Exception as exc:
                log_error("llm_load", exc, name)
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()
        logger.warning("no LLM could be loaded — using the citation-safe extractive generator")
        self.backend = "extractive"
        self.model_name = self.fallback.model_name

    # -- generation -----------------------------------------------------------
    def generate(
        self,
        question: str,
        results: Sequence[RetrievedChunk],
        max_new_tokens: Optional[int] = None,
    ) -> GenerationResult:
        if not self.available:
            return self.fallback.generate(question, results)

        messages, _ = PROMPT_BUILDER.build_messages(question, results)
        started = time.perf_counter()
        try:
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                **({"enable_thinking": False} if "qwen3" in self.model_name.lower() else {}),
            )
        except Exception:
            prompt = f"{messages[0]['content']}\n\n{messages[1]['content']}\n\nAssistant:"

        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=8192)
        inputs = {key: value.to(self.model.device) for key, value in inputs.items()}
        try:
            with torch.inference_mode():
                output = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens or self.config.llm_max_new_tokens,
                    do_sample=self.config.llm_temperature > 0,
                    temperature=max(self.config.llm_temperature, 1e-4),
                    top_p=self.config.llm_top_p,
                    repetition_penalty=1.05,
                    pad_token_id=self.tokenizer.pad_token_id,
                )
        except Exception as exc:
            log_error("llm_generate", exc, question[:120])
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            return self.fallback.generate(question, results)

        prompt_tokens = int(inputs["input_ids"].shape[-1])
        completion_ids = output[0][prompt_tokens:]
        text = self.tokenizer.decode(completion_ids, skip_special_tokens=True).strip()
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
        return GenerationResult(
            text=text,
            prompt_tokens=prompt_tokens,
            completion_tokens=int(completion_ids.shape[-1]),
            seconds=time.perf_counter() - started,
            model_name=self.model_name,
            backend=self.backend,
        )


set_global_seeds(CONFIG.seed)
LLM = LLMEngine(CONFIG, EMBEDDER)
print(f"LLM backend : {LLM.backend}")
print(f"LLM model   : {LLM.model_name}")
print(f"4-bit quant : {CAPABILITIES.get('bitsandbytes') and CONFIG.llm_use_4bit and DEVICE == 'cuda'}")
if DEVICE == "cuda":
    print(f"GPU memory  : {torch.cuda.memory_allocated() / 1024**3:.2f} GB allocated of {GPU_MEMORY_GB} GB")

18:57:56 | INFO    | psych-rag | loading LLM Qwen/Qwen3-4B (4-bit=True)
18:57:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
18:57:56 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/config.json "HTTP/1.1 200 OK"
18:57:56 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

18:57:57 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
18:57:57 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/tokenizer_config.json "HTTP/1.1 200 OK"
18:57:57 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

18:57:57 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-4B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
18:57:57 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-4B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
18:57:57 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
18:57:57 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/vocab.json "HTTP/1.1 200 OK"
18:57:57 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/vocab.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

19:01:51 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
19:01:51 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/generation_config.json "HTTP/1.1 200 OK"
19:01:51 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-4B/1cfa9a7208912126459214e8b04321603b3df60c/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

19:01:51 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-4B/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
19:01:51 | INFO    | psych-rag | LLM ready: Qwen/Qwen3-4B via transformers-4bit


LLM backend : transformers-4bit
LLM model   : Qwen/Qwen3-4B
4-bit quant : True
GPU memory  : 5.84 GB allocated of 14.56 GB


---
## 16. Hallucination Detection & Grounding Verification

Prompting alone is never sufficient. Every answer passes through a four-stage guard **after** generation:

1. **Retrieval confidence gate** — a calibrated blend of the top rerank probability, the mean of the top-3
   fused scores, an **on-domain term** (best raw BGE cosine, rescaled over a calibration band), and evidence
   agreement. Below `min_retrieval_confidence` the pipeline refuses *before* spending a generation token.
   The on-domain term carries a third of the weight because the cross-encoder's sigmoid output is far too
   compressed to separate off-topic questions on its own.
2. **Citation enforcement** — `[n]` markers are parsed, validated against the evidence block (out-of-range
   markers are hallucinated references), and a citation-coverage ratio is computed over clinical sentences.
   The refusal sentence is detected by containment rather than prefix, because the model routinely wraps it
   inside the requested output template.
3. **Sentence-level grounding** — each answer sentence is embedded and compared against its cited passages;
   the mean maximum similarity is the *grounding score*. Sentences below threshold are listed explicitly as
   unsupported.
4. **Entity containment** — numbers, doses and percentages appearing in the answer must also appear in the
   evidence. A fabricated statistic is the single most dangerous failure mode in clinical RAG, and it is
   trivially detectable this way.

If confidence or grounding falls below threshold, the answer is replaced by the refusal message; borderline
answers are returned with an explicit warning banner.

In [23]:
# =============================================================================
# SECTION 16 — GROUNDING VERIFICATION
# =============================================================================


@dataclass
class GroundingReport:
    """Post-generation verification result."""

    grounding_score: float
    citation_coverage: float
    cited_indices: List[int]
    invalid_citations: List[int]
    unsupported_sentences: List[str]
    unsupported_numbers: List[str]
    hedged: bool                     # the answer body concedes it did not answer
    verdict: str                     # grounded | partially_grounded | ungrounded

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class GroundingVerifier:
    """Verifies that a generated answer is entailed by the retrieved evidence."""

    CITATION_PATTERN = re.compile(r"\[(\d{1,2})\]")
    # Phrases that concede the evidence did not answer the question. Grounding
    # similarity cannot catch these: a sentence that faithfully restates an
    # unrelated retrieved passage scores as well-grounded while answering nothing.
    HEDGE_PATTERN = re.compile(
        r"(not (?:explicitly|directly|specifically) (?:stated|mentioned|reported|provided|addressed|available)"
        r"|does not (?:specifically )?(?:report|state|mention|provide|address|contain)"
        r"|is not (?:stated|mentioned|available|provided|included) in the"
        r"|no (?:specific )?(?:data|information|evidence|figures?) (?:is |are )?(?:provided|available|given|reported))",
        re.IGNORECASE,
    )
    ANSWER_SECTION = re.compile(r"\*\*Answer:?\*\*(.*?)(?=\*\*Evidence summary|\*\*Limitations|$)",
                                re.IGNORECASE | re.DOTALL)
    NUMBER_PATTERN = re.compile(r"\b\d+(?:\.\d+)?\s?(?:%|mg|g|ml|mcg|weeks?|months?|years?|days?|hours?)?\b")
    SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")

    def __init__(self, config: RAGConfig, embedder: EmbeddingModel) -> None:
        self.config = config
        self.embedder = embedder

    # -- helpers --------------------------------------------------------------
    def _clinical_sentences(self, answer: str) -> List[str]:
        body = re.sub(r"\*\*(Answer|Evidence summary|Limitations):?\*\*", " ", answer)
        sentences = [s.strip(" -•\t") for s in self.SENTENCE_SPLIT.split(body)]
        return [s for s in sentences if len(s) > 35]

    @staticmethod
    def _normalise_number(token: str) -> str:
        return re.sub(r"\s+", "", token.lower())

    # -- public API -----------------------------------------------------------
    def _answer_body(self, answer: str) -> str:
        """The Answer section only — Limitations is *supposed* to state gaps."""
        match = self.ANSWER_SECTION.search(answer)
        return match.group(1) if match else answer

    def verify(self, answer: str, evidence: Sequence[RetrievedChunk]) -> GroundingReport:
        if not answer.strip() or answer.strip() == INSUFFICIENT_EVIDENCE_MESSAGE:
            return GroundingReport(1.0, 1.0, [], [], [], [], False, "grounded")

        n_evidence = len(evidence)
        cited = sorted({int(m) for m in self.CITATION_PATTERN.findall(answer)})
        valid = [c for c in cited if 1 <= c <= n_evidence]
        invalid = [c for c in cited if c not in valid]

        sentences = self._clinical_sentences(answer)
        cited_sentences = [s for s in sentences if self.CITATION_PATTERN.search(s)]
        citation_coverage = len(cited_sentences) / max(len(sentences), 1)

        # ---- sentence-level semantic grounding ------------------------------
        grounding_score = 0.0
        unsupported: List[str] = []
        if sentences and evidence:
            try:
                sentence_vectors = self.embedder.encode_passages(sentences, show_progress=False)
                evidence_vectors = self.embedder.encode_passages(
                    [item.text for item in evidence], show_progress=False
                )
                similarity = sentence_vectors @ evidence_vectors.T          # cosine (unit vectors)
                per_sentence = similarity.max(axis=1)
                grounding_score = float(per_sentence.mean())
                unsupported = [
                    sentence
                    for sentence, score in zip(sentences, per_sentence)
                    if score < self.config.min_grounding_score
                ]
            except Exception as exc:
                log_error("grounding", exc, answer[:100])
                grounding_score = 0.5

        # ---- numeric containment --------------------------------------------
        evidence_text = " ".join(item.text for item in evidence).lower()
        evidence_numbers = {self._normalise_number(n) for n in self.NUMBER_PATTERN.findall(evidence_text)}
        answer_numbers = {self._normalise_number(n) for n in self.NUMBER_PATTERN.findall(answer.lower())}
        unsupported_numbers = sorted(
            n for n in answer_numbers - evidence_numbers if len(n) > 1 and not n.isdigit()
        )[:10]

        hedged = bool(self.HEDGE_PATTERN.search(self._answer_body(answer)))

        if invalid or grounding_score < self.config.min_grounding_score * 0.8:
            verdict = "ungrounded"
        elif unsupported or unsupported_numbers or citation_coverage < 0.6 or hedged:
            verdict = "partially_grounded"
        else:
            verdict = "grounded"

        return GroundingReport(
            grounding_score=round(grounding_score, 4),
            citation_coverage=round(citation_coverage, 4),
            cited_indices=valid,
            invalid_citations=invalid,
            unsupported_sentences=unsupported[:5],
            unsupported_numbers=unsupported_numbers,
            hedged=hedged,
            verdict=verdict,
        )


class ConfidenceEstimator:
    """Turns raw retrieval scores into an interpretable [0, 1] confidence value.

    Four complementary signals are blended:
      * `top` — the best rerank probability (or fused score) available
      * `depth` — the mean of the top-3 results, penalising a single lucky hit
      * `domain` — the best *raw* dense cosine, rescaled over the calibration band.
        This is the term that actually detects off-topic questions. The
        cross-encoder's sigmoid output is heavily compressed (observed range
        ~0.50-0.73 across wildly different queries), so a gate built on it alone
        cannot separate "car timing belt" from "diagnostic features of GAD".
        Raw BGE cosine separates them cleanly.
      * `agreement` — how many of the top results come from distinct documents,
        because corroboration across independent sources is evidence strength
        rather than redundancy
    """

    def __init__(self, config: RAGConfig) -> None:
        self.config = config

    def _domain_score(self, results: Sequence[RetrievedChunk]) -> float:
        """Rescale the best raw dense cosine over the calibration band."""
        similarities = [r.dense_score for r in results if r.dense_score is not None]
        if not similarities:
            return 0.0
        span = max(self.config.dense_similarity_ceiling - self.config.dense_similarity_floor, 1e-6)
        return float(np.clip((max(similarities) - self.config.dense_similarity_floor) / span, 0.0, 1.0))

    def estimate(self, results: Sequence[RetrievedChunk]) -> Tuple[float, Dict[str, float]]:
        if not results:
            return 0.0, {"top": 0.0, "depth": 0.0, "domain": 0.0, "agreement": 0.0}
        scores = np.array(
            [r.rerank_score if r.rerank_score is not None else r.hybrid_score for r in results], dtype="float32"
        )
        top = float(scores[0])
        depth = float(scores[: min(3, len(scores))].mean())
        domain = self._domain_score(results)
        distinct_docs = len({r.metadata.get("doc_id") for r in results[:5]})
        agreement = distinct_docs / max(min(len(results), 5), 1)
        confidence = 0.35 * top + 0.20 * depth + 0.35 * domain + 0.10 * agreement
        return float(np.clip(confidence, 0.0, 1.0)), {
            "top": round(top, 4),
            "depth": round(depth, 4),
            "domain": round(domain, 4),
            "agreement": round(agreement, 4),
        }


GROUNDING_VERIFIER = GroundingVerifier(CONFIG, EMBEDDER)
CONFIDENCE_ESTIMATOR = ConfidenceEstimator(CONFIG)
print("Grounding verifier and confidence estimator ready.")
print(f"  refusal threshold (retrieval confidence): {CONFIG.min_retrieval_confidence}")
print(f"  refusal threshold (grounding score)     : {CONFIG.min_grounding_score}")

Grounding verifier and confidence estimator ready.
  refusal threshold (retrieval confidence): 0.55
  refusal threshold (grounding score)     : 0.42


---
## 17. Retrieval Pipeline (end-to-end orchestration)

`PsychiatryRAGPipeline.answer()` is the single entry point used by the evaluation cells, the example queries
and the Gradio app:

```
question → [risk check] → hybrid retrieve (50) → cross-encoder rerank (top-k)
         → confidence gate → grounded prompt → LLM → citation + grounding verification
         → AnswerResult(answer, confidence, evidence, citations, latency, tokens, verdict)
```

Every intermediate artefact is returned rather than discarded — that is what makes the system explainable.

In [24]:
# =============================================================================
# SECTION 17 — RAG PIPELINE
# =============================================================================


@dataclass
class AnswerResult:
    """Complete, explainable output of one RAG query."""

    question: str
    answer: str
    confidence: float
    confidence_components: Dict[str, float]
    evidence: List[RetrievedChunk]
    citations: List[Dict[str, Any]]
    grounding: Optional[GroundingReport]
    refused: bool
    latency: Dict[str, float]
    tokens: Dict[str, int]
    model_name: str
    warnings: List[str] = field(default_factory=list)

    def evidence_frame(self) -> pd.DataFrame:
        return pd.DataFrame([item.to_row() for item in self.evidence])

    def to_dict(self) -> Dict[str, Any]:
        return {
            "question": self.question,
            "answer": self.answer,
            "confidence": self.confidence,
            "refused": self.refused,
            "citations": self.citations,
            "grounding": self.grounding.to_dict() if self.grounding else None,
            "latency": self.latency,
            "tokens": self.tokens,
            "model": self.model_name,
            "warnings": self.warnings,
        }


class PsychiatryRAGPipeline:
    """End-to-end evidence-gated RAG pipeline."""

    def __init__(
        self,
        config: RAGConfig,
        retriever: HybridRetriever,
        reranker: CrossEncoderReranker,
        llm: LLMEngine,
        verifier: GroundingVerifier,
        confidence: ConfidenceEstimator,
        prompt_builder: PromptBuilder,
    ) -> None:
        self.config = config
        self.retriever = retriever
        self.reranker = reranker
        self.llm = llm
        self.verifier = verifier
        self.confidence = confidence
        self.prompt_builder = prompt_builder
        self.history: List[AnswerResult] = []

    # -- retrieval only (used heavily by the evaluator) ----------------------
    def retrieve(
        self,
        question: str,
        top_k: int = 10,
        use_reranker: bool = True,
        mode: str = "hybrid",
        dense_weight: Optional[float] = None,
        bm25_weight: Optional[float] = None,
        candidates: Optional[int] = None,
    ) -> Tuple[List[RetrievedChunk], Dict[str, float]]:
        timings: Dict[str, float] = {}
        started = time.perf_counter()
        pool = self.retriever.retrieve(
            question,
            k=candidates or self.config.retrieve_candidates,
            candidates=candidates or self.config.retrieve_candidates,
            dense_weight=dense_weight,
            bm25_weight=bm25_weight,
            mode=mode,
        )
        timings["retrieval_s"] = time.perf_counter() - started

        if use_reranker and self.reranker.available and pool:
            started = time.perf_counter()
            results = self.reranker.rerank(question, pool, top_k=top_k)
            timings["rerank_s"] = time.perf_counter() - started
        else:
            results = pool[:top_k]
            for rank, item in enumerate(results, start=1):
                item.rank = rank
            timings["rerank_s"] = 0.0
        return results, timings

    # -- full pipeline --------------------------------------------------------
    def answer(
        self,
        question: str,
        top_k: Optional[int] = None,
        use_reranker: bool = True,
        dense_weight: Optional[float] = None,
        bm25_weight: Optional[float] = None,
        min_confidence: Optional[float] = None,
        generate: bool = True,
    ) -> AnswerResult:
        top_k = top_k or self.config.top_k_default
        min_confidence = self.config.min_retrieval_confidence if min_confidence is None else min_confidence
        question = (question or "").strip()
        warnings_list: List[str] = []
        total_started = time.perf_counter()

        if not question:
            return AnswerResult(question, "Please enter a question.", 0.0, {}, [], [], None, True,
                                {"total_s": 0.0}, {"prompt": 0, "completion": 0, "total": 0}, self.llm.model_name)

        results, timings = self.retrieve(
            question, top_k=top_k, use_reranker=use_reranker,
            dense_weight=dense_weight, bm25_weight=bm25_weight,
        )
        confidence, components = self.confidence.estimate(results)

        # ---- gate 1: refuse before generating if retrieval is weak -----------
        if not results or confidence < min_confidence:
            timings["generation_s"] = 0.0
            timings["total_s"] = time.perf_counter() - total_started
            answer_text = INSUFFICIENT_EVIDENCE_MESSAGE
            if self.prompt_builder.is_risk_query(question):
                answer_text += SAFETY_NOTICE
            result = AnswerResult(
                question=question,
                answer=answer_text,
                confidence=round(confidence, 4),
                confidence_components=components,
                evidence=results,
                citations=[],
                grounding=None,
                refused=True,
                latency={k: round(v, 4) for k, v in timings.items()},
                tokens={"prompt": 0, "completion": 0, "total": 0},
                model_name=self.llm.model_name,
                warnings=[f"Retrieval confidence {confidence:.2f} < threshold {min_confidence:.2f}."],
            )
            self.history.append(result)
            return result

        if not generate:
            timings["generation_s"] = 0.0
            timings["total_s"] = time.perf_counter() - total_started
            return AnswerResult(question, "", round(confidence, 4), components, results, [], None, False,
                                {k: round(v, 4) for k, v in timings.items()},
                                {"prompt": 0, "completion": 0, "total": 0}, self.llm.model_name)

        # ---- generation ------------------------------------------------------
        _, used_evidence = self.prompt_builder.build_messages(question, results)
        generation = self.llm.generate(question, results)
        timings["generation_s"] = generation.seconds

        # ---- gate 2: verify grounding ---------------------------------------
        report = self.verifier.verify(generation.text, used_evidence)
        answer_text = generation.text

        # The model reliably emits the refusal sentence but often wraps it in the
        # requested output template ("**Answer:** I could not find sufficient…").
        # Matching on prefix alone silently recorded those as answered, so match on
        # containment and then normalise to the canonical refusal.
        refused = INSUFFICIENT_EVIDENCE_MESSAGE.lower() in answer_text.lower()
        if refused:
            answer_text = INSUFFICIENT_EVIDENCE_MESSAGE
            warnings_list.append("The model reported insufficient evidence; the answer was replaced by the refusal message.")

        if report.verdict == "ungrounded" and not refused:
            warnings_list.append(
                f"Answer rejected by the grounding verifier "
                f"(score {report.grounding_score:.2f}, invalid citations {report.invalid_citations})."
            )
            answer_text = INSUFFICIENT_EVIDENCE_MESSAGE
            refused = True
        elif report.verdict == "partially_grounded":
            if report.hedged:
                warnings_list.append(
                    "The answer itself concedes the retrieved evidence does not directly address the "
                    "question. Treat the surrounding statements as related context, not as an answer."
                )
            else:
                warnings_list.append(
                    "Some statements could not be matched confidently to a retrieved passage — verify "
                    "against the cited sources below."
                )
        if self.config.require_citations and not report.cited_indices and not refused:
            warnings_list.append("The generated answer contained no valid citations; treat it with caution.")

        citations = [
            {
                "marker": f"[{index}]",
                "citation": item.citation(),
                "source": item.metadata.get("source"),
                "organization": item.metadata.get("organization"),
                "year": item.metadata.get("year"),
                "url": item.metadata.get("url"),
                "chunk_id": item.chunk_id,
                "used": (index in report.cited_indices) if report else False,
            }
            for index, item in enumerate(used_evidence, start=1)
        ]

        if self.prompt_builder.is_risk_query(question):
            answer_text += SAFETY_NOTICE

        timings["total_s"] = time.perf_counter() - total_started
        result = AnswerResult(
            question=question,
            answer=answer_text,
            confidence=round(confidence, 4),
            confidence_components=components,
            evidence=results,
            citations=citations,
            grounding=report,
            refused=refused,
            latency={k: round(v, 4) for k, v in timings.items()},
            tokens={
                "prompt": generation.prompt_tokens,
                "completion": generation.completion_tokens,
                "total": generation.total_tokens,
            },
            model_name=generation.model_name,
            warnings=warnings_list,
        )
        self.history.append(result)
        return result


PIPELINE = PsychiatryRAGPipeline(
    CONFIG, RETRIEVER, RERANKER, LLM, GROUNDING_VERIFIER, CONFIDENCE_ESTIMATOR, PROMPT_BUILDER
)

_test = PIPELINE.answer("What are the diagnostic features of generalised anxiety disorder?", top_k=5)
print("=" * 78)
print("PIPELINE SMOKE TEST")
print("=" * 78)
print(f"confidence : {_test.confidence:.3f}  {_test.confidence_components}")
print(f"refused    : {_test.refused}")
print(f"grounding  : {_test.grounding.verdict if _test.grounding else 'n/a'} "
      f"(score {_test.grounding.grounding_score if _test.grounding else 0:.3f})")
print(f"latency    : {_test.latency}")
print(f"tokens     : {_test.tokens}")
print("-" * 78)
print(_test.answer[:1200])
print("-" * 78)
print("Top evidence:")
if _test.evidence:
    print(_test.evidence_frame()[["rank", "final_score", "source", "section", "title"]].to_string(index=False))

# Negative control: an out-of-domain question must trigger the refusal path.
_control = PIPELINE.answer("What is the recommended torque specification for a 2004 Honda Civic cylinder head?")
print("\nOut-of-domain control → refused =", _control.refused, "| confidence =", f"{_control.confidence:.3f}")

PIPELINE SMOKE TEST
confidence : 0.759  {'top': 0.7105, 'depth': 0.7097, 'domain': 0.7674, 'agreement': 1.0}
refused    : False
grounding  : grounded (score 0.673)
latency    : {'retrieval_s': 0.0634, 'rerank_s': 3.0739, 'generation_s': 29.915, 'total_s': 33.4826}
tokens     : {'prompt': 2192, 'completion': 188, 'total': 2380}
------------------------------------------------------------------------------
**Answer:** Generalized anxiety disorder (GAD) is characterized by excessive anxiety and worry about a number of life circumstances, along with physical symptoms like restlessness, fatigue, and difficulty concentrating [2]. It is also associated with anticipatory anxiety and chronic worry, which are core features of the disorder [1]. The diagnostic criteria emphasize the presence of these symptoms, along with physical symptoms such as dizziness and muscle tension [4].

**Evidence summary:** 
- GAD is distinguished by worry about multiple life circumstances and a cognitive aspect of anx

---
## 18. Evaluation — Retrieval

Metrics implemented from first principles (no metric library, so the definitions are auditable):

| Metric | Definition |
|---|---|
| **Recall@k** | fraction of relevant chunks appearing in the top-k |
| **Precision@k** | fraction of the top-k that are relevant |
| **Hit-Rate@k** | 1 if any relevant chunk is in the top-k |
| **MRR** | reciprocal rank of the first relevant result |
| **MAP** | mean of average precision over the ranked list |
| **nDCG@k** | binary-gain DCG normalised by the ideal ranking |
| **Latency** | wall-clock retrieval and rerank time per query (mean / p95) |

Configurations compared: **BM25 only**, **dense only**, **hybrid**, and **hybrid + cross-encoder** at
rerank depths 5 / 10 / 20. Fusion weights are tuned on the **validation** split; the **test** split is used
only for the final reported numbers.

In [25]:
# =============================================================================
# SECTION 18 — RETRIEVAL METRICS
# =============================================================================


class RetrievalMetrics:
    """Standard IR metrics computed over a ranked list of chunk ids."""

    @staticmethod
    def recall_at_k(retrieved: Sequence[str], relevant: Set[str], k: int) -> float:
        if not relevant:
            return 0.0
        hits = len(set(retrieved[:k]) & relevant)
        return hits / min(len(relevant), k) if k < len(relevant) else hits / len(relevant)

    @staticmethod
    def precision_at_k(retrieved: Sequence[str], relevant: Set[str], k: int) -> float:
        if k == 0:
            return 0.0
        return len(set(retrieved[:k]) & relevant) / k

    @staticmethod
    def hit_rate_at_k(retrieved: Sequence[str], relevant: Set[str], k: int) -> float:
        return 1.0 if set(retrieved[:k]) & relevant else 0.0

    @staticmethod
    def reciprocal_rank(retrieved: Sequence[str], relevant: Set[str]) -> float:
        for position, chunk_id in enumerate(retrieved, start=1):
            if chunk_id in relevant:
                return 1.0 / position
        return 0.0

    @staticmethod
    def average_precision(retrieved: Sequence[str], relevant: Set[str]) -> float:
        if not relevant:
            return 0.0
        hits = 0
        precision_sum = 0.0
        for position, chunk_id in enumerate(retrieved, start=1):
            if chunk_id in relevant:
                hits += 1
                precision_sum += hits / position
        return precision_sum / min(len(relevant), len(retrieved)) if hits else 0.0

    @staticmethod
    def ndcg_at_k(retrieved: Sequence[str], relevant: Set[str], k: int) -> float:
        gains = [1.0 if chunk_id in relevant else 0.0 for chunk_id in retrieved[:k]]
        dcg = sum(gain / math.log2(position + 1) for position, gain in enumerate(gains, start=1))
        ideal_hits = min(len(relevant), k)
        idcg = sum(1.0 / math.log2(position + 1) for position in range(1, ideal_hits + 1))
        return dcg / idcg if idcg > 0 else 0.0

    @classmethod
    def evaluate_query(cls, retrieved: Sequence[str], relevant: Set[str]) -> Dict[str, float]:
        return {
            "recall@1": cls.recall_at_k(retrieved, relevant, 1),
            "recall@5": cls.recall_at_k(retrieved, relevant, 5),
            "recall@10": cls.recall_at_k(retrieved, relevant, 10),
            "recall@20": cls.recall_at_k(retrieved, relevant, 20),
            "precision@5": cls.precision_at_k(retrieved, relevant, 5),
            "precision@10": cls.precision_at_k(retrieved, relevant, 10),
            "hit_rate@1": cls.hit_rate_at_k(retrieved, relevant, 1),
            "hit_rate@5": cls.hit_rate_at_k(retrieved, relevant, 5),
            "hit_rate@10": cls.hit_rate_at_k(retrieved, relevant, 10),
            "mrr": cls.reciprocal_rank(retrieved, relevant),
            "map": cls.average_precision(retrieved, relevant),
            "ndcg@5": cls.ndcg_at_k(retrieved, relevant, 5),
            "ndcg@10": cls.ndcg_at_k(retrieved, relevant, 10),
        }


class RetrievalEvaluator:
    """Runs metric sweeps across retrieval configurations."""

    def __init__(self, pipeline: PsychiatryRAGPipeline, config: RAGConfig) -> None:
        self.pipeline = pipeline
        self.config = config

    def evaluate_configuration(
        self,
        items: Sequence[BenchmarkItem],
        name: str,
        mode: str = "hybrid",
        use_reranker: bool = False,
        top_k: int = 20,
        dense_weight: Optional[float] = None,
        bm25_weight: Optional[float] = None,
        show_progress: bool = True,
    ) -> Dict[str, Any]:
        per_query: List[Dict[str, float]] = []
        latencies: List[float] = []
        iterator = tqdm(items, desc=f"eval:{name}", leave=False) if show_progress else items
        for item in iterator:
            try:
                started = time.perf_counter()
                results, timings = self.pipeline.retrieve(
                    item.question, top_k=top_k, use_reranker=use_reranker, mode=mode,
                    dense_weight=dense_weight, bm25_weight=bm25_weight,
                )
                latencies.append(time.perf_counter() - started)
                retrieved_ids = [r.chunk_id for r in results]
                per_query.append(RetrievalMetrics.evaluate_query(retrieved_ids, set(item.relevant_chunk_ids)))
            except Exception as exc:
                log_error("retrieval_eval", exc, item.question[:100])
        if not per_query:
            return {"configuration": name}
        frame = pd.DataFrame(per_query)
        summary = {"configuration": name, "n_queries": len(per_query)}
        summary.update({metric: round(float(frame[metric].mean()), 4) for metric in frame.columns})
        summary["latency_mean_ms"] = round(1000 * float(np.mean(latencies)), 1)
        summary["latency_p95_ms"] = round(1000 * float(np.percentile(latencies, 95)), 1)
        summary["_per_query"] = frame                 # retained for significance testing
        return summary

    @staticmethod
    def bootstrap_ci(
        values: Sequence[float], n_resamples: int = 2000, seed: int = GLOBAL_SEED
    ) -> Tuple[float, float]:
        """Percentile bootstrap 95% interval for the mean of a single sample."""
        array = np.asarray(values, dtype="float64")
        if array.size == 0:
            return 0.0, 0.0
        rng = np.random.RandomState(seed)
        means = array[rng.randint(0, array.size, size=(n_resamples, array.size))].mean(axis=1)
        return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

    @staticmethod
    def paired_bootstrap(
        treatment: Sequence[float],
        control: Sequence[float],
        n_resamples: int = 2000,
        seed: int = GLOBAL_SEED,
    ) -> Tuple[float, float, float]:
        """Paired bootstrap of the mean difference; returns (delta, lo95, hi95).

        The same queries are scored under both configurations, so the difference is
        paired and resampling is over queries. With a validation split of well under
        a hundred questions, metric gaps of a few thousandths are indistinguishable
        from noise, and this makes that explicit instead of inviting a coin-flip
        hyper-parameter choice.
        """
        treatment_array = np.asarray(treatment, dtype="float64")
        control_array = np.asarray(control, dtype="float64")
        differences = treatment_array - control_array
        rng = np.random.RandomState(seed)
        n = len(differences)
        if n == 0:
            return 0.0, 0.0, 0.0
        means = differences[rng.randint(0, n, size=(n_resamples, n))].mean(axis=1)
        return float(differences.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

    def tune_fusion_weights(
        self, items: Sequence[BenchmarkItem], grid: Sequence[float], metric: str = "ndcg@10"
    ) -> Dict[str, Any]:
        """Sweep the dense/BM25 weighting on the VALIDATION split only.

        Returns the sweep table plus a paired-bootstrap comparison of the best
        setting against the configured default, so the decision to change a
        hyper-parameter is evidence-based rather than rank-based.
        """
        rows: List[Dict[str, Any]] = []
        per_query: Dict[float, List[float]] = {}
        for dense_weight in tqdm(grid, desc="tuning fusion weights", leave=False):
            summary = self.evaluate_configuration(
                items,
                name=f"dense={dense_weight:.1f}/bm25={1 - dense_weight:.1f}",
                mode="hybrid",
                use_reranker=False,
                top_k=20,
                dense_weight=dense_weight,
                bm25_weight=1 - dense_weight,
                show_progress=False,
            )
            frame = summary.pop("_per_query", None)
            if frame is not None and metric in frame:
                per_query[round(dense_weight, 2)] = frame[metric].tolist()
            summary["dense_weight"] = dense_weight
            rows.append(summary)

        table = pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)
        default_weight = round(self.config.dense_weight, 2)
        best_weight = round(float(table.iloc[0]["dense_weight"]), 2)

        comparison: Dict[str, Any] = {"metric": metric, "best_weight": best_weight,
                                      "default_weight": default_weight, "significant": False}
        if best_weight != default_weight and best_weight in per_query and default_weight in per_query:
            delta, low, high = self.paired_bootstrap(per_query[best_weight], per_query[default_weight])
            comparison.update({"delta": delta, "ci_low": low, "ci_high": high, "significant": low > 0})
        return {"table": table, "comparison": comparison, "per_query": per_query}


EVALUATOR = RetrievalEvaluator(PIPELINE, CONFIG)
set_global_seeds(CONFIG.seed)

# --- Weight tuning on the VALIDATION split -----------------------------------
validation_sample = SPLITS["validation"][: min(len(SPLITS["validation"]), CONFIG.eval_retrieval_sample)]
WEIGHT_TUNING_RESULT = CACHE.get_or_build(
    PATHS.evaluation / f"weight_tuning_{CORPUS_EVAL_FINGERPRINT}.pkl",
    lambda: EVALUATOR.tune_fusion_weights(validation_sample, grid=[0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]),
    fmt="pickle",
    label=f"weight_tuning_{CORPUS_EVAL_FINGERPRINT}",
)
WEIGHT_TUNING = WEIGHT_TUNING_RESULT["table"]
WEIGHT_COMPARISON = WEIGHT_TUNING_RESULT["comparison"]

print("FUSION-WEIGHT TUNING (validation split, ranked by nDCG@10)")
print("=" * 78)
print(WEIGHT_TUNING[["configuration", "recall@5", "recall@10", "mrr", "ndcg@10", "latency_mean_ms"]]
      .to_string(index=False))

BEST_DENSE_WEIGHT = float(WEIGHT_COMPARISON.get("best_weight", CONFIG.dense_weight))
print("\nSIGNIFICANCE OF THE BEST SETTING vs THE CONFIGURED DEFAULT")
print("-" * 78)
if "delta" in WEIGHT_COMPARISON:
    print(f"  best on validation : dense={WEIGHT_COMPARISON['best_weight']:.1f}")
    print(f"  configured default : dense={WEIGHT_COMPARISON['default_weight']:.1f}")
    print(f"  paired delta ({WEIGHT_COMPARISON['metric']}) : "
          f"{WEIGHT_COMPARISON['delta']:+.4f} "
          f"[95% CI {WEIGHT_COMPARISON['ci_low']:+.4f}, {WEIGHT_COMPARISON['ci_high']:+.4f}] "
          f"over {len(validation_sample)} questions")
    if WEIGHT_COMPARISON["significant"]:
        print("  → the CI excludes zero: changing CONFIG.dense_weight is justified by the data.")
    else:
        print("  → the CI spans zero: the difference is not distinguishable from noise, so the\n"
              "    documented default is kept. Ranking alone would have flipped this setting between\n"
              "    runs; the interval is what makes the decision reproducible.")
else:
    print(f"  the configured default (dense={CONFIG.dense_weight:.1f}) already ranks first — nothing to change.")

19:02:29 | INFO    | psych-rag | cache MISS → building weight_tuning_00f98c7248_a1c407c885_b2


tuning fusion weights:   0%|          | 0/7 [00:00<?, ?it/s]

19:02:52 | INFO    | psych-rag | built weight_tuning_00f98c7248_a1c407c885_b2 in 23.0s


FUSION-WEIGHT TUNING (validation split, ranked by nDCG@10)
     configuration  recall@5  recall@10    mrr  ndcg@10  latency_mean_ms
dense=0.2/bm25=0.8    0.2191     0.2315 0.3483   0.2135             43.0
dense=0.4/bm25=0.6    0.2244     0.2247 0.3558   0.2121             41.3
dense=0.5/bm25=0.5    0.2186     0.2421 0.3335   0.2101             43.6
dense=0.6/bm25=0.4    0.2060     0.2389 0.2751   0.1998             61.3
dense=0.0/bm25=1.0    0.2133     0.2183 0.3116   0.1875             55.1
dense=0.8/bm25=0.2    0.2068     0.2203 0.2055   0.1670             44.5
dense=1.0/bm25=0.0    0.1655     0.2137 0.1706   0.1487             43.0

SIGNIFICANCE OF THE BEST SETTING vs THE CONFIGURED DEFAULT
------------------------------------------------------------------------------
  best on validation : dense=0.2
  configured default : dense=0.5
  paired delta (ndcg@10) : +0.0034 [95% CI -0.0194, +0.0248] over 69 questions
  → the CI spans zero: the difference is not distinguishable from noise, 

In [26]:
# =============================================================================
# SECTION 18b — CONFIGURATION COMPARISON ON THE TEST SPLIT
# =============================================================================
test_sample = SPLITS["test"][: min(len(SPLITS["test"]), CONFIG.eval_retrieval_sample)]


HEADLINE_METRICS = ("recall@5", "precision@5", "mrr", "ndcg@10")


def run_retrieval_benchmark() -> Dict[str, Any]:
    configurations: List[Dict[str, Any]] = [
        {"name": "BM25 only", "mode": "bm25", "use_reranker": False, "top_k": 20},
        {"name": "Dense only (FAISS)", "mode": "dense", "use_reranker": False, "top_k": 20},
        {"name": f"Hybrid ({CONFIG.dense_weight}/{CONFIG.bm25_weight})", "mode": "hybrid",
         "use_reranker": False, "top_k": 20},
    ]
    if RERANKER.available:
        configurations.extend(
            {"name": f"Hybrid + rerank (top-{depth})", "mode": "hybrid", "use_reranker": True, "top_k": depth}
            for depth in CONFIG.rerank_depths
        )
    rows: List[Dict[str, Any]] = []
    per_query: Dict[str, pd.DataFrame] = {}
    for configuration in configurations:
        summary = EVALUATOR.evaluate_configuration(
            test_sample,
            name=configuration["name"],
            mode=configuration["mode"],
            use_reranker=configuration["use_reranker"],
            top_k=configuration["top_k"],
        )
        frame = summary.pop("_per_query", None)
        if frame is not None:
            per_query[configuration["name"]] = frame
        rows.append(summary)

    table = pd.DataFrame(rows)

    # 95% intervals on the headline metrics. On a 64-question test split, a bare
    # point estimate overstates how much any single number can be trusted.
    interval_rows: List[Dict[str, Any]] = []
    for name, frame in per_query.items():
        for metric in HEADLINE_METRICS:
            if metric not in frame:
                continue
            low, high = EVALUATOR.bootstrap_ci(frame[metric].values)
            interval_rows.append({
                "configuration": name,
                "metric": metric,
                "mean": round(float(frame[metric].mean()), 4),
                "ci_low": round(low, 4),
                "ci_high": round(high, 4),
            })
    intervals = pd.DataFrame(interval_rows)

    # Paired test of the best reranked depth against the hybrid baseline.
    effect: Dict[str, Any] = {}
    baseline_name = next((n for n in per_query if n.startswith("Hybrid (")), None)
    reranked = [n for n in per_query if "rerank" in n]
    if baseline_name and reranked:
        best_name = max(reranked, key=lambda n: per_query[n]["ndcg@10"].mean())
        effect = {"baseline": baseline_name, "treatment": best_name, "metrics": {}}
        for metric in HEADLINE_METRICS:
            if metric not in per_query[best_name]:
                continue
            delta, low, high = EVALUATOR.paired_bootstrap(
                per_query[best_name][metric].values, per_query[baseline_name][metric].values
            )
            effect["metrics"][metric] = {
                "baseline": round(float(per_query[baseline_name][metric].mean()), 4),
                "treatment": round(float(per_query[best_name][metric].mean()), 4),
                "delta": round(delta, 4),
                "ci_low": round(low, 4),
                "ci_high": round(high, 4),
                "significant": bool(low > 0),
            }
    return {"table": table, "intervals": intervals, "effect": effect}


RETRIEVAL_BENCHMARK: Dict[str, Any] = CACHE.get_or_build(
    PATHS.evaluation / f"retrieval_results_{CORPUS_EVAL_FINGERPRINT}.pkl",
    run_retrieval_benchmark,
    fmt="pickle",
    label=f"retrieval_results_{CORPUS_EVAL_FINGERPRINT}",
)
RETRIEVAL_RESULTS: pd.DataFrame = RETRIEVAL_BENCHMARK["table"]
RETRIEVAL_INTERVALS: pd.DataFrame = RETRIEVAL_BENCHMARK["intervals"]
RERANK_EFFECT: Dict[str, Any] = RETRIEVAL_BENCHMARK["effect"]

display_columns = [
    "configuration", "n_queries", "recall@1", "recall@5", "recall@10",
    "precision@5", "precision@10", "mrr", "map", "ndcg@5", "ndcg@10",
    "hit_rate@5", "latency_mean_ms", "latency_p95_ms",
]
available_columns = [c for c in display_columns if c in RETRIEVAL_RESULTS.columns]

print("=" * 110)
print(f"RETRIEVAL EVALUATION — TEST SPLIT ({len(test_sample)} questions)")
print("=" * 110)
print(RETRIEVAL_RESULTS[available_columns].to_string(index=False))

if not RETRIEVAL_INTERVALS.empty:
    print("\n95% BOOTSTRAP INTERVALS ON THE HEADLINE METRICS")
    print("-" * 110)
    pivot = RETRIEVAL_INTERVALS.assign(
        interval=lambda f: f.apply(lambda r: f"{r['mean']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]", axis=1)
    ).pivot(index="configuration", columns="metric", values="interval")
    print(pivot.to_string())

if RERANK_EFFECT.get("metrics"):
    print(f"\nEFFECT OF CROSS-ENCODER RERANKING — {RERANK_EFFECT['treatment']} vs {RERANK_EFFECT['baseline']}")
    print("-" * 110)
    print(f"  {'metric':<12} {'baseline':>9} {'reranked':>9} {'delta':>9}   {'95% CI of the paired delta':<30} verdict")
    for metric, stats in RERANK_EFFECT["metrics"].items():
        relative = 100 * stats["delta"] / max(stats["baseline"], 1e-9)
        interval = f"[{stats['ci_low']:+.4f}, {stats['ci_high']:+.4f}]"
        verdict = "significant" if stats["significant"] else "not distinguishable from noise"
        print(f"  {metric:<12} {stats['baseline']:>9.4f} {stats['treatment']:>9.4f} "
              f"{stats['delta']:>+9.4f}   {interval:<30} {verdict} ({relative:+.1f}%)")
    print("\n  Intervals are paired bootstraps over the same test questions, so they measure\n"
          "  the reranker's effect rather than the variability of the question sample.")
RETRIEVAL_RESULTS[available_columns]

19:02:52 | INFO    | psych-rag | cache MISS → building retrieval_results_00f98c7248_a1c407c885_b2


eval:BM25 only:   0%|          | 0/82 [00:00<?, ?it/s]

eval:Dense only (FAISS):   0%|          | 0/82 [00:00<?, ?it/s]

eval:Hybrid (0.5/0.5):   0%|          | 0/82 [00:00<?, ?it/s]

eval:Hybrid + rerank (top-5):   0%|          | 0/82 [00:00<?, ?it/s]

eval:Hybrid + rerank (top-10):   0%|          | 0/82 [00:00<?, ?it/s]

eval:Hybrid + rerank (top-20):   0%|          | 0/82 [00:00<?, ?it/s]

19:17:05 | INFO    | psych-rag | built retrieval_results_00f98c7248_a1c407c885_b2 in 852.8s


RETRIEVAL EVALUATION — TEST SPLIT (82 questions)
           configuration  n_queries  recall@1  recall@5  recall@10  precision@5  precision@10    mrr    map  ndcg@5  ndcg@10  hit_rate@5  latency_mean_ms  latency_p95_ms
               BM25 only         82    0.1829    0.1974     0.2441       0.1049        0.0756 0.2883 0.1227  0.1691   0.1845      0.4390             18.4            39.6
      Dense only (FAISS)         82    0.0976    0.1766     0.2053       0.0878        0.0683 0.2095 0.1231  0.1452   0.1597      0.3415             25.0            32.5
        Hybrid (0.5/0.5)         82    0.2073    0.2207     0.2374       0.1171        0.0817 0.3279 0.1582  0.2052   0.2119      0.5000             60.7            88.8
 Hybrid + rerank (top-5)         82    0.2073    0.2134     0.1931       0.1171        0.0585 0.3006 0.1532  0.2032   0.1875      0.4512           3441.0          3997.4
Hybrid + rerank (top-10)         82    0.2073    0.2134     0.2651       0.1171        0.0915 0.3166 

,configuration,n_queries,recall@1,recall@5,recall@10,precision@5,precision@10,mrr,map,ndcg@5,ndcg@10,hit_rate@5,latency_mean_ms,latency_p95_ms
0,BM25 only,82,0.1829,0.1974,0.2441,0.1049,0.0756,0.2883,0.1227,0.1691,0.1845,0.4390,18.4,39.6
1,Dense only (FAISS),82,0.0976,0.1766,0.2053,0.0878,0.0683,0.2095,0.1231,0.1452,0.1597,0.3415,25.0,32.5
2,Hybrid (0.5/0.5),82,0.2073,0.2207,0.2374,0.1171,0.0817,0.3279,0.1582,0.2052,0.2119,0.5000,60.7,88.8
3,Hybrid + rerank (top-5),82,0.2073,0.2134,0.1931,0.1171,0.0585,0.3006,0.1532,0.2032,0.1875,0.4512,3441.0,3997.4
4,Hybrid + rerank (top-10),82,0.2073,0.2134,0.2651,0.1171,0.0915,0.3166,0.1589,0.2032,0.2239,0.4512,3423.0,3911.8
5,Hybrid + rerank (top-20),82,0.2073,0.2134,0.2651,0.1171,0.0915,0.3215,0.1703,0.2032,0.2239,0.4512,3425.7,3873.2


---
## 19. Evaluation — RAG Quality (RAGAS)

Retrieval metrics measure whether the *right evidence* was found; they say nothing about whether the *answer*
is faithful to it. Section 19 closes that gap with four generation-level metrics:

| Metric | Question it answers |
|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? |
| **Context Precision** | Are the retrieved passages actually relevant (is the context free of noise)? |
| **Context Recall** | Does the retrieved context contain what the reference answer needs? |
| **Answer Relevancy** | Does the answer address the question that was asked? |
| **Answer Correctness** | How close is the answer to the reference answer? |

**Two execution paths.** RAGAS is used when it imports cleanly *and* a judge LLM is available; RAGAS by
default expects an OpenAI key, which this project deliberately does not use, so the notebook wires the local
Qwen model and the local BGE embeddings into RAGAS via LangChain wrappers. If any part of that stack is
unavailable (very common on a fresh Colab runtime), the notebook falls back to a **native implementation of
the same five metrics** built on the local cross-encoder and embedding model. The evaluation path used is
always reported alongside the numbers, so results are never silently incomparable.

In [27]:
# =============================================================================
# SECTION 19 — RAG EVALUATION (RAGAS + NATIVE FALLBACK)
# =============================================================================


def build_generation_eval_set(items: Sequence[BenchmarkItem], limit: int) -> List[Dict[str, Any]]:
    """Run the full pipeline over a sample of test questions, collecting artefacts."""
    records: List[Dict[str, Any]] = []
    sample = list(items)[:limit]
    for item in tqdm(sample, desc="Generating answers for RAG evaluation", leave=False):
        try:
            result = PIPELINE.answer(item.question, top_k=CONFIG.top_k_default)
            records.append(
                {
                    "question": item.question,
                    "answer": result.answer,
                    "contexts": [chunk.text for chunk in result.evidence],
                    "ground_truth": item.expected_answer,
                    "reference_chunk_ids": item.relevant_chunk_ids,
                    "retrieved_chunk_ids": [chunk.chunk_id for chunk in result.evidence],
                    "confidence": result.confidence,
                    "refused": result.refused,
                    "grounding_score": result.grounding.grounding_score if result.grounding else None,
                    "citation_coverage": result.grounding.citation_coverage if result.grounding else None,
                    "verdict": result.grounding.verdict if result.grounding else "n/a",
                    "latency_s": result.latency.get("total_s", 0.0),
                    "prompt_tokens": result.tokens.get("prompt", 0),
                    "completion_tokens": result.tokens.get("completion", 0),
                }
            )
        except Exception as exc:
            log_error("generation_eval", exc, item.question[:100])
    return records


set_global_seeds(CONFIG.seed)
GENERATION_RECORDS: List[Dict[str, Any]] = CACHE.get_or_build(
    PATHS.evaluation / f"generation_records_{CORPUS_EVAL_FINGERPRINT}.pkl",
    lambda: build_generation_eval_set(SPLITS["test"], CONFIG.eval_generation_sample),
    fmt="pickle",
    label="generation_records",
)
GENERATION_FRAME = pd.DataFrame(GENERATION_RECORDS)

print(f"Generated {len(GENERATION_RECORDS)} answers on the test split.")
if not GENERATION_FRAME.empty:
    print(f"  refusal rate        : {100 * GENERATION_FRAME['refused'].mean():.1f}%")
    print(f"  mean confidence     : {GENERATION_FRAME['confidence'].mean():.3f}")
    print(f"  mean grounding score: {GENERATION_FRAME['grounding_score'].dropna().mean():.3f}")
    print(f"  mean latency        : {GENERATION_FRAME['latency_s'].mean():.2f} s")
    print(f"  mean tokens/query   : {(GENERATION_FRAME['prompt_tokens'] + GENERATION_FRAME['completion_tokens']).mean():.0f}")
    print("\n  Grounding verdicts:")
    print("   " + GENERATION_FRAME["verdict"].value_counts().to_string().replace("\n", "\n   "))

19:17:05 | INFO    | psych-rag | cache MISS → building generation_records


Generating answers for RAG evaluation:   0%|          | 0/24 [00:00<?, ?it/s]

19:29:20 | INFO    | psych-rag | built generation_records in 735.3s


Generated 24 answers on the test split.
  refusal rate        : 0.0%
  mean confidence     : 0.742
  mean grounding score: 0.769
  mean latency        : 30.64 s
  mean tokens/query   : 2241

  Grounding verdicts:
   verdict
   grounded              20
   partially_grounded     4


In [28]:
# =============================================================================
# SECTION 19b — METRIC IMPLEMENTATIONS
# =============================================================================


class NativeRagMetrics:
    """Local, dependency-light implementation of the RAGAS metric family.

    Each metric is computed with the models already loaded in this notebook:
      * faithfulness      — mean max cosine similarity of answer claims to context
                            sentences, sharpened by the cross-encoder when available
      * context_precision — share of retrieved passages that are relevant to the
                            question (cross-encoder relevance, else cosine)
      * context_recall    — share of reference-answer sentences covered by the context
      * answer_relevancy  — cosine similarity between question and answer embeddings
      * answer_correctness— cosine similarity between answer and reference answer
    """

    def __init__(self, embedder: EmbeddingModel, reranker: CrossEncoderReranker, config: RAGConfig) -> None:
        self.embedder = embedder
        self.reranker = reranker
        self.config = config

    def _rescale(self, value: float) -> float:
        """Map a raw cosine onto [0, 1] over the calibration band.

        Binary thresholding saturated every score at 1.000 on the first run, which
        made the metric useless. Grading over the same band the confidence gate
        uses keeps the numbers comparable and informative.
        """
        span = max(self.config.dense_similarity_ceiling - self.config.dense_similarity_floor, 1e-6)
        return float(np.clip((value - self.config.dense_similarity_floor) / span, 0.0, 1.0))

    @staticmethod
    def _split_sentences(text: str) -> List[str]:
        cleaned = re.sub(r"\*\*(Answer|Evidence summary|Limitations):?\*\*", " ", text or "")
        cleaned = re.sub(r"\[\d{1,2}\]", " ", cleaned)
        return [s.strip(" -•\t") for s in re.split(r"(?<=[.!?])\s+", cleaned) if len(s.strip()) > 30]

    def _cosine_matrix(self, left: Sequence[str], right: Sequence[str]) -> Optional[np.ndarray]:
        if not left or not right:
            return None
        left_vectors = self.embedder.encode_passages(list(left), show_progress=False)
        right_vectors = self.embedder.encode_passages(list(right), show_progress=False)
        return left_vectors @ right_vectors.T

    def faithfulness(self, answer: str, contexts: Sequence[str]) -> float:
        claims = self._split_sentences(answer)
        context_sentences = [s for context in contexts for s in self._split_sentences(context)]
        similarity = self._cosine_matrix(claims, context_sentences)
        if similarity is None:
            return float("nan")
        return float(np.mean([self._rescale(v) for v in similarity.max(axis=1)]))

    def context_precision(self, question: str, contexts: Sequence[str]) -> float:
        if not contexts:
            return float("nan")
        similarity = self._cosine_matrix([question], contexts)
        if similarity is None:
            return float("nan")
        return float(np.mean([self._rescale(v) for v in similarity[0]]))

    def context_recall(self, ground_truth: str, contexts: Sequence[str]) -> float:
        reference_sentences = self._split_sentences(ground_truth)
        context_sentences = [s for context in contexts for s in self._split_sentences(context)]
        similarity = self._cosine_matrix(reference_sentences, context_sentences)
        if similarity is None:
            return float("nan")
        return float(np.mean([self._rescale(v) for v in similarity.max(axis=1)]))

    def answer_relevancy(self, question: str, answer: str) -> float:
        similarity = self._cosine_matrix([answer], [question])
        return float(similarity[0][0]) if similarity is not None else float("nan")

    def answer_correctness(self, answer: str, ground_truth: str) -> float:
        similarity = self._cosine_matrix([answer], [ground_truth])
        return float(similarity[0][0]) if similarity is not None else float("nan")

    def evaluate(self, records: Sequence[Dict[str, Any]]) -> pd.DataFrame:
        rows: List[Dict[str, Any]] = []
        for record in tqdm(records, desc="Native RAG metrics", leave=False):
            if record.get("refused"):
                continue                                  # refusals are scored separately
            try:
                rows.append(
                    {
                        "question": record["question"],
                        "faithfulness": self.faithfulness(record["answer"], record["contexts"]),
                        "context_precision": self.context_precision(record["question"], record["contexts"]),
                        "context_recall": self.context_recall(record["ground_truth"], record["contexts"]),
                        "answer_relevancy": self.answer_relevancy(record["question"], record["answer"]),
                        "answer_correctness": self.answer_correctness(record["answer"], record["ground_truth"]),
                    }
                )
            except Exception as exc:
                log_error("native_rag_metrics", exc, record["question"][:80])
        return pd.DataFrame(rows)


def run_ragas_evaluation(records: Sequence[Dict[str, Any]]) -> Optional[pd.DataFrame]:
    """Attempt a true RAGAS evaluation driven by the local LLM and embeddings."""
    missing = [name for name in ("ragas", "datasets") if not CAPABILITIES.get(name)]
    if missing:
        logger.info("RAGAS unavailable (missing: %s) — using the native metric implementation.",
                    ", ".join(missing))
        return None
    if not LLM.available:
        logger.info("No local judge LLM available — RAGAS requires one; using native metrics.")
        return None
    try:
        import nest_asyncio

        nest_asyncio.apply()
        from datasets import Dataset
        from ragas import evaluate as ragas_evaluate
        from ragas.metrics import (
            answer_relevancy,
            context_precision,
            context_recall,
            faithfulness,
        )
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
        from transformers import pipeline as hf_pipeline

        judge = hf_pipeline(
            "text-generation",
            model=LLM.model,
            tokenizer=LLM.tokenizer,
            max_new_tokens=384,
            do_sample=False,
            return_full_text=False,
        )
        judge_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=judge))
        judge_embeddings = LangchainEmbeddingsWrapper(
            HuggingFaceEmbeddings(model_name=EMBEDDER.model_name, model_kwargs={"device": DEVICE})
        )
        usable = [r for r in records if not r["refused"]]
        if not usable:
            return None
        dataset = Dataset.from_dict(
            {
                "question": [r["question"] for r in usable],
                "answer": [r["answer"] for r in usable],
                "contexts": [r["contexts"] for r in usable],
                "ground_truth": [r["ground_truth"] for r in usable],
            }
        )
        outcome = ragas_evaluate(
            dataset,
            metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=judge_llm,
            embeddings=judge_embeddings,
            raise_exceptions=False,
        )
        frame = outcome.to_pandas()
        logger.info("RAGAS evaluation completed on %d records", len(frame))
        return frame
    except Exception as exc:
        log_error("ragas", exc, "falling back to native metrics")
        return None


def evaluate_rag_quality() -> Dict[str, Any]:
    """Run RAGAS if possible, otherwise the native implementation."""
    frame = run_ragas_evaluation(GENERATION_RECORDS)
    backend = "ragas"
    if frame is None or frame.empty:
        frame = NativeRagMetrics(EMBEDDER, RERANKER, CONFIG).evaluate(GENERATION_RECORDS)
        backend = "native"
    numeric = frame.select_dtypes(include=[np.number])
    summary = {column: round(float(numeric[column].mean()), 4) for column in numeric.columns}
    return {"backend": backend, "per_question": frame, "summary": summary}


set_global_seeds(CONFIG.seed)
RAG_EVALUATION: Dict[str, Any] = CACHE.get_or_build(
    PATHS.evaluation / f"rag_evaluation_{CORPUS_EVAL_FINGERPRINT}.pkl",
    evaluate_rag_quality,
    fmt="pickle",
    label="rag_evaluation",
)

RAGAS_SUMMARY = RAG_EVALUATION["summary"]
RAGAS_FRAME = RAG_EVALUATION["per_question"]

print("=" * 78)
print(f"RAG QUALITY EVALUATION  (backend: {RAG_EVALUATION['backend'].upper()})")
print("=" * 78)
if RAGAS_SUMMARY:
    for metric, value in RAGAS_SUMMARY.items():
        bar = "█" * int(round(max(0.0, min(value, 1.0)) * 34))
        print(f"  {metric:<22} {value:6.3f}  {bar}")
else:
    print("  No generation-level metrics could be computed (all queries were refused).")

# Refusal behaviour is itself a headline safety metric.
if not GENERATION_FRAME.empty:
    print("\nSAFETY / ABSTENTION BEHAVIOUR")
    print("-" * 78)
    print(f"  refusal rate on in-domain test questions : {100 * GENERATION_FRAME['refused'].mean():.1f}%")
    print(f"  answers flagged 'grounded'               : "
          f"{100 * (GENERATION_FRAME['verdict'] == 'grounded').mean():.1f}%")
    print(f"  answers flagged 'partially_grounded'     : "
          f"{100 * (GENERATION_FRAME['verdict'] == 'partially_grounded').mean():.1f}%")
    print(f"  mean citation coverage                   : "
          f"{GENERATION_FRAME['citation_coverage'].dropna().mean():.3f}")

EVALUATION_SUMMARY: Dict[str, Any] = {
    "retrieval": RETRIEVAL_RESULTS.to_dict("records"),
    "rag_backend": RAG_EVALUATION["backend"],
    "rag_metrics": RAGAS_SUMMARY,
    "generation_stats": {
        "n": int(len(GENERATION_FRAME)),
        "refusal_rate": float(GENERATION_FRAME["refused"].mean()) if not GENERATION_FRAME.empty else None,
        "mean_latency_s": float(GENERATION_FRAME["latency_s"].mean()) if not GENERATION_FRAME.empty else None,
        "mean_total_tokens": float(
            (GENERATION_FRAME["prompt_tokens"] + GENERATION_FRAME["completion_tokens"]).mean()
        ) if not GENERATION_FRAME.empty else None,
    },
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(PATHS.evaluation / "evaluation_summary.json").write_text(
    json.dumps(EVALUATION_SUMMARY, indent=2, default=str), encoding="utf-8"
)
print(f"\nEvaluation summary written to {PATHS.evaluation / 'evaluation_summary.json'}")

19:29:20 | INFO    | psych-rag | cache MISS → building rag_evaluation
19:29:20 | INFO    | psych-rag | RAGAS unavailable (missing: ragas) — using the native metric implementation.


Native RAG metrics:   0%|          | 0/24 [00:00<?, ?it/s]

19:30:05 | INFO    | psych-rag | built rag_evaluation in 44.7s


RAG QUALITY EVALUATION  (backend: NATIVE)
  faithfulness            0.887  ██████████████████████████████
  context_precision       0.759  ██████████████████████████
  context_recall          0.687  ███████████████████████
  answer_relevancy        0.842  █████████████████████████████
  answer_correctness      0.740  █████████████████████████

SAFETY / ABSTENTION BEHAVIOUR
------------------------------------------------------------------------------
  refusal rate on in-domain test questions : 0.0%
  answers flagged 'grounded'               : 83.3%
  answers flagged 'partially_grounded'     : 16.7%
  mean citation coverage                   : 0.793

Evaluation summary written to /content/drive/MyDrive/psychiatry_rag/06_evaluation/evaluation_summary.json


---
## 20. Visualisations

Eleven publication-quality figures (matplotlib only, no seaborn), each saved to the plots cache so the Gradio
Evaluation Dashboard can serve them directly.

In [29]:
# =============================================================================
# SECTION 20 — VISUALISATIONS
# =============================================================================

PLOT_PATHS: Dict[str, Path] = {}
PALETTE = ["#2E5A87", "#C1666B", "#4E937A", "#E8A33D", "#7D6B91", "#5B8C5A", "#B85C38", "#3E7C8A"]


def save_figure(figure: plt.Figure, key: str, title: str) -> Path:
    """Persist a figure to the plots cache and register it for the dashboard."""
    path = PATHS.plots / f"{key}.png"
    figure.savefig(path, bbox_inches="tight", facecolor="white")
    PLOT_PATHS[title] = path
    plt.close(figure)
    return path


def annotate_bars(axis: plt.Axes, bars, fmt: str = "{:.0f}", offset: float = 0.01) -> None:
    """Write the value on top of each bar (small, unobtrusive)."""
    for bar in bars:
        height = bar.get_height()
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            height + offset * max(1.0, abs(height)),
            fmt.format(height),
            ha="center", va="bottom", fontsize=8,
        )


# --- 1. Corpus size by source -------------------------------------------------
figure, axes = plt.subplots(1, 2, figsize=(13, 4.6))
source_chunks = CORPUS["source"].value_counts()
bars = axes[0].bar(range(len(source_chunks)), source_chunks.values,
                   color=[PALETTE[i % len(PALETTE)] for i in range(len(source_chunks))])
axes[0].set_xticks(range(len(source_chunks)))
axes[0].set_xticklabels(source_chunks.index, rotation=35, ha="right")
axes[0].set_ylabel("Chunks")
axes[0].set_title("Corpus size by source (chunks)")
annotate_bars(axes[0], bars)

source_tokens = CORPUS.groupby("source")["n_tokens"].sum().sort_values(ascending=False)
bars = axes[1].bar(range(len(source_tokens)), source_tokens.values / 1000,
                   color=[PALETTE[i % len(PALETTE)] for i in range(len(source_tokens))])
axes[1].set_xticks(range(len(source_tokens)))
axes[1].set_xticklabels(source_tokens.index, rotation=35, ha="right")
axes[1].set_ylabel("Tokens (thousands)")
axes[1].set_title("Corpus volume by source (approx. tokens)")
annotate_bars(axes[1], bars, fmt="{:.0f}")
figure.suptitle("Corpus composition", fontweight="bold")
save_figure(figure, "01_corpus_by_source", "Corpus size by source")

# --- 2. Documents by publication year ----------------------------------------
figure, axis = plt.subplots(figsize=(9, 4.2))
years = CORPUS.drop_duplicates("doc_id")["year"].dropna().astype(int)
if len(years):
    counts = years.value_counts().sort_index()
    axis.bar(counts.index, counts.values, color=PALETTE[0], width=0.75)
    axis.set_xlabel("Publication year")
    axis.set_ylabel("Documents")
    axis.set_title("Documents by publication year")
    axis.set_xticks(counts.index[:: max(1, len(counts) // 12)])
else:
    axis.text(0.5, 0.5, "No publication years available", ha="center", va="center")
save_figure(figure, "02_documents_by_year", "Documents by publication year")

# --- 3. Chunk length distribution --------------------------------------------
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
lengths = CORPUS["n_tokens"].values
axes[0].hist(lengths, bins=42, color=PALETTE[2], edgecolor="white", linewidth=0.5)
axes[0].axvline(float(np.mean(lengths)), color=PALETTE[1], linestyle="--",
                label=f"mean = {np.mean(lengths):.0f}")
axes[0].axvline(CONFIG.chunk_target_tokens, color=PALETTE[3], linestyle=":",
                label=f"target = {CONFIG.chunk_target_tokens}")
axes[0].set_xlabel("Tokens per chunk (approx.)")
axes[0].set_ylabel("Chunks")
axes[0].set_title("Chunk length distribution")
axes[0].legend(fontsize=8)

by_source = [CORPUS.loc[CORPUS["source"] == s, "n_tokens"].values for s in source_chunks.index]
axes[1].boxplot(by_source, showfliers=False, patch_artist=True,
                boxprops={"facecolor": PALETTE[0], "alpha": 0.55},
                medianprops={"color": PALETTE[1], "linewidth": 1.6})
axes[1].set_xticklabels(source_chunks.index, rotation=35, ha="right")
axes[1].set_ylabel("Tokens per chunk")
axes[1].set_title("Chunk length by source")
save_figure(figure, "03_chunk_length", "Chunk length distribution")

# --- 4. Chunk-size experiment -------------------------------------------------
figure, axis = plt.subplots(figsize=(9, 4.2))
if not CHUNK_EXPERIMENT.empty:
    axis.plot(CHUNK_EXPERIMENT["target_tokens"], CHUNK_EXPERIMENT["n_chunks"],
              marker="o", color=PALETTE[0], label="chunks produced")
    axis.set_xlabel("Target chunk size (tokens)")
    axis.set_ylabel("Chunks produced", color=PALETTE[0])
    twin = axis.twinx()
    twin.plot(CHUNK_EXPERIMENT["target_tokens"], CHUNK_EXPERIMENT["mean_tokens"],
              marker="s", color=PALETTE[1], label="mean chunk length")
    twin.set_ylabel("Mean chunk length (tokens)", color=PALETTE[1])
    twin.grid(False)
    axis.axvline(CONFIG.chunk_target_tokens, color=PALETTE[3], linestyle=":", alpha=0.8)
    axis.set_title("Chunk-size experiment (fragmentation vs granularity)")
save_figure(figure, "04_chunk_size_experiment", "Chunk-size experiment")

# --- 5. Embedding generation time --------------------------------------------
figure, axis = plt.subplots(figsize=(8, 4.2))
seconds = float(EMBEDDING_TIMING.get("seconds", 0.0) or 0.0)
n_chunks = int(EMBEDDING_TIMING.get("chunks", len(CORPUS)) or len(CORPUS))
throughput = n_chunks / seconds if seconds > 0 else 0.0
bars = axis.bar(
    ["Total time (s)", "Chunks / second", "Chunks (hundreds)"],
    [seconds, throughput, n_chunks / 100],
    color=[PALETTE[0], PALETTE[2], PALETTE[4]],
)
annotate_bars(axis, bars, fmt="{:.1f}")
axis.set_title(f"Embedding generation — {EMBEDDER.model_name} on {DEVICE.upper()}")
axis.set_ylabel("Value")
save_figure(figure, "05_embedding_time", "Embedding generation time")

# --- 6. Retrieval latency -----------------------------------------------------
figure, axis = plt.subplots(figsize=(9.5, 4.4))
if "latency_mean_ms" in RETRIEVAL_RESULTS.columns:
    labels = RETRIEVAL_RESULTS["configuration"].tolist()
    positions = np.arange(len(labels))
    width = 0.38
    bars_mean = axis.bar(positions - width / 2, RETRIEVAL_RESULTS["latency_mean_ms"], width,
                         label="mean", color=PALETTE[0])
    bars_p95 = axis.bar(positions + width / 2, RETRIEVAL_RESULTS["latency_p95_ms"], width,
                        label="p95", color=PALETTE[1])
    annotate_bars(axis, bars_mean)
    annotate_bars(axis, bars_p95)
    axis.set_xticks(positions)
    axis.set_xticklabels(labels, rotation=22, ha="right")
    axis.set_ylabel("Latency (ms)")
    axis.set_title("Retrieval latency by configuration")
    axis.legend(fontsize=8)
save_figure(figure, "06_retrieval_latency", "Retrieval latency")

# --- 7. Recall comparison -----------------------------------------------------
figure, axis = plt.subplots(figsize=(10, 4.6))
recall_columns = [c for c in ["recall@1", "recall@5", "recall@10"] if c in RETRIEVAL_RESULTS.columns]
if recall_columns:
    labels = RETRIEVAL_RESULTS["configuration"].tolist()
    positions = np.arange(len(labels))
    width = 0.8 / len(recall_columns)
    for index, column in enumerate(recall_columns):
        offset = (index - (len(recall_columns) - 1) / 2) * width
        axis.bar(positions + offset, RETRIEVAL_RESULTS[column], width,
                 label=column, color=PALETTE[index % len(PALETTE)])
    axis.set_xticks(positions)
    axis.set_xticklabels(labels, rotation=22, ha="right")
    axis.set_ylabel("Recall")
    axis.set_ylim(0, 1.05)
    axis.set_title("Recall@k across retrieval configurations (test split)")
    axis.legend(fontsize=8)
save_figure(figure, "07_recall_comparison", "Recall comparison")

# --- 8. Precision comparison --------------------------------------------------
figure, axis = plt.subplots(figsize=(10, 4.6))
precision_columns = [c for c in ["precision@5", "precision@10"] if c in RETRIEVAL_RESULTS.columns]
if precision_columns:
    labels = RETRIEVAL_RESULTS["configuration"].tolist()
    positions = np.arange(len(labels))
    width = 0.8 / len(precision_columns)
    for index, column in enumerate(precision_columns):
        offset = (index - (len(precision_columns) - 1) / 2) * width
        axis.bar(positions + offset, RETRIEVAL_RESULTS[column], width,
                 label=column, color=PALETTE[(index + 2) % len(PALETTE)])
    axis.set_xticks(positions)
    axis.set_xticklabels(labels, rotation=22, ha="right")
    axis.set_ylabel("Precision")
    axis.set_title("Precision@k across retrieval configurations (test split)")
    axis.legend(fontsize=8)
save_figure(figure, "08_precision_comparison", "Precision comparison")

# --- 9. MRR / nDCG before vs after reranking ---------------------------------
figure, axis = plt.subplots(figsize=(9.5, 4.6))
if "mrr" in RETRIEVAL_RESULTS.columns:
    labels = RETRIEVAL_RESULTS["configuration"].tolist()
    positions = np.arange(len(labels))
    width = 0.38
    bars_mrr = axis.bar(positions - width / 2, RETRIEVAL_RESULTS["mrr"], width, label="MRR", color=PALETTE[0])
    ndcg_column = "ndcg@10" if "ndcg@10" in RETRIEVAL_RESULTS.columns else None
    if ndcg_column:
        bars_ndcg = axis.bar(positions + width / 2, RETRIEVAL_RESULTS[ndcg_column], width,
                             label="nDCG@10", color=PALETTE[1])
        annotate_bars(axis, bars_ndcg, fmt="{:.2f}")
    annotate_bars(axis, bars_mrr, fmt="{:.2f}")
    axis.set_xticks(positions)
    axis.set_xticklabels(labels, rotation=22, ha="right")
    axis.set_ylim(0, 1.05)
    axis.set_ylabel("Score")
    axis.set_title("Ranking quality before vs after cross-encoder reranking")
    axis.legend(fontsize=8)
save_figure(figure, "09_mrr_rerank", "MRR before vs after reranking")

# --- 10. RAG quality metrics --------------------------------------------------
figure, axis = plt.subplots(figsize=(9, 4.4))
if RAGAS_SUMMARY:
    names = list(RAGAS_SUMMARY)
    values = [RAGAS_SUMMARY[name] for name in names]
    bars = axis.barh(names, values, color=[PALETTE[i % len(PALETTE)] for i in range(len(names))])
    for bar, value in zip(bars, values):
        axis.text(min(value + 0.015, 1.02), bar.get_y() + bar.get_height() / 2,
                  f"{value:.3f}", va="center", fontsize=9)
    axis.set_xlim(0, 1.08)
    axis.set_xlabel("Score")
    axis.set_title(f"RAG quality metrics ({RAG_EVALUATION['backend']} backend)")
else:
    axis.text(0.5, 0.5, "No RAG metrics available", ha="center", va="center")
save_figure(figure, "10_rag_metrics", "RAG quality metrics")

# --- 11. Source & topic distribution -----------------------------------------
figure, axes = plt.subplots(1, 2, figsize=(13.5, 5))
organizations = CORPUS["organization"].value_counts().head(7)
axes[0].pie(organizations.values, labels=[o[:26] for o in organizations.index], autopct="%1.1f%%",
            colors=PALETTE[: len(organizations)], startangle=110,
            wedgeprops={"edgecolor": "white", "linewidth": 1.2}, textprops={"fontsize": 8})
axes[0].set_title("Chunk share by publishing organisation")

topic_counts = CORPUS["topic"].value_counts().head(12).sort_values()
labels = [PSYCHIATRY_TOPICS.get(t, {}).get("label", t.replace("_", " ")) for t in topic_counts.index]
bars = axes[1].barh(range(len(topic_counts)), topic_counts.values, color=PALETTE[0])
axes[1].set_yticks(range(len(topic_counts)))
axes[1].set_yticklabels([label[:34] for label in labels], fontsize=8)
axes[1].set_xlabel("Chunks")
axes[1].set_title("Top disorders represented in the corpus")
for bar, value in zip(bars, topic_counts.values):
    axes[1].text(value + max(topic_counts.values) * 0.01, bar.get_y() + bar.get_height() / 2,
                 str(value), va="center", fontsize=8)
save_figure(figure, "11_source_topic_distribution", "Source and topic distribution")

# --- 12. Confidence / grounding behaviour ------------------------------------
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
if not GENERATION_FRAME.empty:
    axes[0].hist(GENERATION_FRAME["confidence"].dropna(), bins=14, color=PALETTE[0],
                 edgecolor="white", linewidth=0.6)
    axes[0].axvline(CONFIG.min_retrieval_confidence, color=PALETTE[1], linestyle="--",
                    label=f"refusal threshold = {CONFIG.min_retrieval_confidence}")
    axes[0].set_xlabel("Retrieval confidence")
    axes[0].set_ylabel("Questions")
    axes[0].set_title("Confidence distribution (test split)")
    axes[0].legend(fontsize=8)

    verdicts = GENERATION_FRAME["verdict"].value_counts()
    bars = axes[1].bar(range(len(verdicts)), verdicts.values,
                       color=[PALETTE[i % len(PALETTE)] for i in range(len(verdicts))])
    axes[1].set_xticks(range(len(verdicts)))
    axes[1].set_xticklabels(verdicts.index, rotation=12)
    axes[1].set_ylabel("Answers")
    axes[1].set_title("Grounding verdicts")
    annotate_bars(axes[1], bars)
save_figure(figure, "12_confidence_grounding", "Confidence and grounding behaviour")

print(f"Saved {len(PLOT_PATHS)} figures to {PATHS.plots}")
for title, path in PLOT_PATHS.items():
    print(f"  • {title:<42} {path.name}")

Saved 12 figures to /content/drive/MyDrive/psychiatry_rag/07_plots
  • Corpus size by source                      01_corpus_by_source.png
  • Documents by publication year              02_documents_by_year.png
  • Chunk length distribution                  03_chunk_length.png
  • Chunk-size experiment                      04_chunk_size_experiment.png
  • Embedding generation time                  05_embedding_time.png
  • Retrieval latency                          06_retrieval_latency.png
  • Recall comparison                          07_recall_comparison.png
  • Precision comparison                       08_precision_comparison.png
  • MRR before vs after reranking              09_mrr_rerank.png
  • RAG quality metrics                        10_rag_metrics.png
  • Source and topic distribution              11_source_topic_distribution.png
  • Confidence and grounding behaviour         12_confidence_grounding.png


---
## 21. Gradio Application

A five-tab interface over the same `PsychiatryRAGPipeline` object used for evaluation — no duplicated logic,
so what the UI shows is exactly what was measured.

| Tab | Contents |
|---|---|
| **1 · Ask** | Question box, answer with inline citations, confidence gauge, evidence cards, citation list |
| **2 · Retrieved Documents** | Full ranked table: hybrid / dense / BM25 / rerank scores plus all metadata |
| **3 · Corpus Explorer** | Filter by source, topic, section and document type; keyword search; inspect any chunk |
| **4 · Evaluation Dashboard** | Retrieval metrics, RAG metrics, latency, token usage and every figure |
| **5 · System Information** | Corpus statistics, models, hardware, configuration, cache status, error log |

In [30]:
# =============================================================================
# SECTION 21 — GRADIO APPLICATION
# =============================================================================
import gradio as gr  # noqa: E402

LAST_RETRIEVAL: Dict[str, Any] = {"question": "", "frame": pd.DataFrame(), "result": None}

CUSTOM_CSS = """
.gradio-container {font-family: 'Inter', 'Segoe UI', system-ui, sans-serif;}
#answer-box {border-left: 4px solid #2E5A87; padding-left: 14px;}
footer {visibility: hidden;}
"""


def _confidence_badge(confidence: float, refused: bool) -> str:
    if refused:
        return f"<span style='color:#B85C38;font-weight:600'>REFUSED — confidence {confidence:.2f}</span>"
    if confidence >= 0.65:
        colour, label = "#4E937A", "HIGH"
    elif confidence >= CONFIG.min_retrieval_confidence:
        colour, label = "#E8A33D", "MODERATE"
    else:
        colour, label = "#C1666B", "LOW"
    filled = int(round(confidence * 20))
    bar = "█" * filled + "░" * (20 - filled)
    return f"<span style='color:{colour};font-weight:600'>{label} — {confidence:.2f}</span><br><code>{bar}</code>"


def _evidence_markdown(result: AnswerResult) -> str:
    if not result.evidence:
        return "_No passages passed the retrieval threshold._"
    blocks: List[str] = []
    for index, item in enumerate(result.evidence, start=1):
        year = item.metadata.get("year")
        year_text = int(year) if isinstance(year, (int, float)) and not pd.isna(year) else "n.d."
        rerank_text = f" · rerank **{item.rerank_score:.3f}**" if item.rerank_score is not None else ""
        url = item.metadata.get("url") or ""
        link = f"[source]({url})" if url else ""
        body = item.text.split("\n", 1)[-1]
        blocks.append(
            f"**[{index}] {item.metadata.get('title', 'Untitled')[:110]}**  \n"
            f"*{item.metadata.get('organization')} · {year_text} · "
            f"{item.metadata.get('document_type')} · section: `{item.metadata.get('section')}`* {link}  \n"
            f"scores — hybrid **{item.hybrid_score:.3f}** · dense {item.dense_score:.3f} · "
            f"bm25 {item.bm25_score:.3f}{rerank_text}  \n"
            f"> {body[:620]}{'…' if len(body) > 620 else ''}"
        )
    return "\n\n---\n\n".join(blocks)


def _citation_markdown(result: AnswerResult) -> str:
    if not result.citations:
        return "_No citations were produced for this answer._"
    lines = ["| Marker | Used | Citation | Link |", "|---|---|---|---|"]
    for citation in result.citations:
        used = "✅" if citation["used"] else "—"
        url = citation.get("url") or ""
        link = f"[open]({url})" if url else ""
        lines.append(f"| {citation['marker']} | {used} | {citation['citation'][:130]} | {link} |")
    return "\n".join(lines)


def ui_ask(
    question: str,
    top_k: int,
    use_reranker: bool,
    dense_weight: float,
    min_confidence: float,
) -> Tuple[str, str, str, str, pd.DataFrame]:
    """Gradio callback for Tab 1."""
    try:
        result = PIPELINE.answer(
            question,
            top_k=int(top_k),
            use_reranker=bool(use_reranker),
            dense_weight=float(dense_weight),
            bm25_weight=1.0 - float(dense_weight),
            min_confidence=float(min_confidence),
        )
    except Exception as exc:
        log_error("ui_ask", exc, question[:120])
        return (f"An error occurred: {exc}", "", "", "", pd.DataFrame())

    frame = result.evidence_frame()
    LAST_RETRIEVAL.update({"question": question, "frame": frame, "result": result})

    answer_markdown = result.answer
    if result.warnings:
        answer_markdown += "\n\n" + "\n".join(f"> ⚠️ {w}" for w in result.warnings)
    if result.grounding:
        answer_markdown += (
            f"\n\n<sub>Grounding: **{result.grounding.verdict}** "
            f"(score {result.grounding.grounding_score:.2f}, "
            f"citation coverage {result.grounding.citation_coverage:.2f}) · "
            f"latency {result.latency.get('total_s', 0):.2f}s · "
            f"{result.tokens.get('total', 0)} tokens · model `{result.model_name}`</sub>"
        )
    return (
        answer_markdown,
        _confidence_badge(result.confidence, result.refused),
        _evidence_markdown(result),
        _citation_markdown(result),
        frame,
    )


def ui_last_retrieval() -> Tuple[str, pd.DataFrame]:
    """Gradio callback for Tab 2."""
    frame = LAST_RETRIEVAL.get("frame", pd.DataFrame())
    if frame.empty:
        return "_Ask a question in Tab 1 to populate this view._", pd.DataFrame()
    result: Optional[AnswerResult] = LAST_RETRIEVAL.get("result")
    header = (
        f"### Retrieval trace\n"
        f"**Query:** {LAST_RETRIEVAL['question']}\n\n"
        f"**Retriever:** hybrid (dense {CONFIG.dense_weight} / bm25 {CONFIG.bm25_weight}) · "
        f"**Reranker:** `{RERANKER.model_name}` · "
        f"**Candidates rescored:** {CONFIG.retrieve_candidates}"
    )
    if result is not None:
        header += (
            f"\n\n**Confidence:** {result.confidence:.3f} "
            f"({result.confidence_components}) · "
            f"**Retrieval:** {result.latency.get('retrieval_s', 0) * 1000:.0f} ms · "
            f"**Rerank:** {result.latency.get('rerank_s', 0) * 1000:.0f} ms"
        )
    return header, frame


CORPUS_EXPLORER_COLUMNS = [
    "chunk_id", "source", "organization", "year", "topic", "section",
    "document_type", "title", "n_tokens", "url",
]


def ui_explore_corpus(
    source: str, topic: str, section: str, document_type: str, keyword: str, limit: int
) -> Tuple[str, pd.DataFrame]:
    """Gradio callback for Tab 3."""
    frame = CORPUS.copy()
    if source != "All":
        frame = frame[frame["source"] == source]
    if topic != "All":
        frame = frame[frame["topic"] == topic]
    if section != "All":
        frame = frame[frame["section"] == section]
    if document_type != "All":
        frame = frame[frame["document_type"] == document_type]
    if keyword.strip():
        pattern = re.escape(keyword.strip())
        frame = frame[
            frame["text"].str.contains(pattern, case=False, na=False)
            | frame["title"].str.contains(pattern, case=False, na=False)
        ]
    summary = (
        f"**{len(frame):,} chunks** match "
        f"({frame['doc_id'].nunique():,} documents, "
        f"{int(frame['n_tokens'].sum()):,} approx. tokens)."
    )
    return summary, frame[CORPUS_EXPLORER_COLUMNS].head(int(limit))


def ui_inspect_chunk(chunk_id: str) -> str:
    """Show the full text and metadata of one chunk."""
    row = CORPUS[CORPUS["chunk_id"] == chunk_id.strip()]
    if row.empty:
        return "_Chunk not found. Copy a `chunk_id` from the table above._"
    record = row.iloc[0]
    year = record["year"]
    year_text = int(year) if pd.notna(year) else "n.d."
    return (
        f"### {record['title']}\n"
        f"**Organisation:** {record['organization']} · **Year:** {year_text} · "
        f"**Type:** {record['document_type']}  \n"
        f"**Topic:** {record['topic']} · **Section:** `{record['section']}` "
        f"({record['section_heading']}) · **Tokens:** {record['n_tokens']}  \n"
        f"**Licence:** {record['license_note'] or 'not specified'}  \n"
        f"**URL:** {record['url']}\n\n---\n\n{record['text']}"
    )


def _retrieval_table() -> pd.DataFrame:
    columns = [c for c in [
        "configuration", "n_queries", "recall@1", "recall@5", "recall@10",
        "precision@5", "precision@10", "mrr", "map", "ndcg@10",
        "hit_rate@5", "latency_mean_ms", "latency_p95_ms",
    ] if c in RETRIEVAL_RESULTS.columns]
    return RETRIEVAL_RESULTS[columns]


def _rag_table() -> pd.DataFrame:
    if not RAGAS_SUMMARY:
        return pd.DataFrame([{"metric": "unavailable", "score": None}])
    return pd.DataFrame(
        [{"metric": metric, "score": value} for metric, value in RAGAS_SUMMARY.items()]
    )


def _system_markdown() -> str:
    cache_report = CACHE.report()
    hits = int((cache_report["status"] == "hit").sum()) if not cache_report.empty else 0
    built = int((cache_report["status"] == "built").sum()) if not cache_report.empty else 0
    generation_stats = EVALUATION_SUMMARY.get("generation_stats", {})
    return f"""
### Corpus
| Metric | Value |
|---|---|
| Documents (raw → cleaned) | {len(RAW_DOCUMENTS):,} → {len(CLEAN_DOCUMENTS):,} |
| Indexed chunks | {len(CORPUS):,} |
| Approx. tokens | {int(CORPUS['n_tokens'].sum()):,} |
| Distinct sources | {CORPUS['source'].nunique()} |
| Distinct organisations | {CORPUS['organization'].nunique()} |
| Canonical sections | {CORPUS['section'].nunique()} |
| Clinical topics | {CORPUS['topic'].nunique()} |
| Year range | {int(CORPUS['year'].min()) if CORPUS['year'].notna().any() else 'n/a'} – {int(CORPUS['year'].max()) if CORPUS['year'].notna().any() else 'n/a'} |

### Models
| Component | Model | Notes |
|---|---|---|
| Embeddings | `{EMBEDDER.model_name}` | dim {EMBEDDER.dimension}, normalised, max_seq {CONFIG.embedding_max_seq_length} |
| Vector index | FAISS | {FAISS_INDEX.index_type}, {FAISS_INDEX.size:,} vectors |
| Sparse index | BM25-Okapi | {BM25_INDEX.n_documents:,} documents |
| Reranker | `{RERANKER.model_name}` | {'active' if RERANKER.available else 'unavailable — hybrid ordering used'} |
| Generator | `{LLM.model_name}` | backend `{LLM.backend}` |

### Retrieval configuration
| Parameter | Value |
|---|---|
| Chunk target / overlap | {CONFIG.chunk_target_tokens} / {CONFIG.chunk_overlap_tokens} tokens |
| Fusion weights (dense / BM25) | {CONFIG.dense_weight} / {CONFIG.bm25_weight} |
| Candidate pool → rerank depths | {CONFIG.retrieve_candidates} → {list(CONFIG.rerank_depths)} |
| Refusal thresholds (confidence / grounding) | {CONFIG.min_retrieval_confidence} / {CONFIG.min_grounding_score} |
| Benchmark questions (train/val/test) | {len(SPLITS['train'])} / {len(SPLITS['validation'])} / {len(SPLITS['test'])} |

### Hardware & runtime
| Item | Value |
|---|---|
| Device | {DEVICE.upper()} ({GPU_NAME}) |
| GPU memory | {GPU_MEMORY_GB} GB |
| Python / PyTorch | {HARDWARE_INFO['python']} / {HARDWARE_INFO['torch']} |
| Colab | {IN_COLAB} |
| Cache root | `{PATHS.base}` |
| Cache status | {hits} artefacts loaded, {built} rebuilt |
| Seed | {CONFIG.seed} |

### Measured performance
| Item | Value |
|---|---|
| RAG metric backend | {EVALUATION_SUMMARY.get('rag_backend')} |
| Mean end-to-end latency | {generation_stats.get('mean_latency_s') or 0:.2f} s |
| Mean tokens per query | {generation_stats.get('mean_total_tokens') or 0:.0f} |
| Refusal rate (in-domain) | {100 * (generation_stats.get('refusal_rate') or 0):.1f}% |
| HTTP requests during ingestion | {HTTP.request_count:,} |
| Non-fatal errors logged | {len(ERROR_LOG)} |

### Package versions
{chr(10).join(f'- `{name}` {version}' for name, version in sorted(VERSION_MANIFEST.items()))}
"""


def ui_error_log() -> pd.DataFrame:
    if not ERROR_LOG:
        return pd.DataFrame([{"stage": "—", "error_type": "none", "message": "No errors logged."}])
    return pd.DataFrame(ERROR_LOG)[["timestamp", "stage", "error_type", "message", "context"]]


EXAMPLE_QUESTIONS = [
    "What are the diagnostic features of generalised anxiety disorder?",
    "Which psychosocial interventions are recommended for schizophrenia?",
    "What does the evidence say about lithium in the maintenance treatment of bipolar disorder?",
    "How should follow-up be conducted after a self-harm presentation?",
    "What are the risk factors for post-traumatic stress disorder?",
    "Which medications are used first line for obsessive-compulsive disorder?",
    "What is the reported prevalence of eating disorders in adolescents?",
    "How is ADHD diagnosed in adults?",
]


def build_interface() -> gr.Blocks:
    """Assemble the five-tab Gradio application."""
    with gr.Blocks(title="Evidence-Based Psychiatry RAG Assistant", css=CUSTOM_CSS,
                   theme=gr.themes.Soft(primary_hue="blue")) as demo:
        gr.Markdown(
            """
# 🧠 Evidence-Based Psychiatry RAG Assistant
Answers are generated **only** from retrieved passages of open clinical literature
(PubMed · Europe PMC · WHO · ICD-11 · NIMH · MedlinePlus). When the retrieved evidence is insufficient,
the assistant says so instead of guessing.

> ⚠️ **Not a medical device.** This is an information-retrieval demonstration, not clinical advice.
> If you may be at risk of harm, contact your local emergency services or a crisis line immediately.
"""
        )

        # ------------------------------------------------------------------ Tab 1
        with gr.Tab("1 · Ask a psychiatry question"):
            with gr.Row():
                with gr.Column(scale=3):
                    question_box = gr.Textbox(
                        label="Your question",
                        placeholder="e.g. What are the diagnostic features of generalised anxiety disorder?",
                        lines=3,
                    )
                    with gr.Row():
                        ask_button = gr.Button("Retrieve evidence and answer", variant="primary")
                        clear_button = gr.Button("Clear")
                    gr.Examples(examples=EXAMPLE_QUESTIONS, inputs=question_box, label="Example questions")
                with gr.Column(scale=1):
                    top_k_slider = gr.Slider(3, 20, value=CONFIG.top_k_default, step=1, label="Passages in context (top-k)")
                    rerank_toggle = gr.Checkbox(value=RERANKER.available, label="Cross-encoder reranking",
                                                interactive=RERANKER.available)
                    weight_slider = gr.Slider(0.0, 1.0, value=CONFIG.dense_weight, step=0.05,
                                              label="Dense weight (BM25 = 1 − dense)")
                    confidence_slider = gr.Slider(0.0, 0.9, value=CONFIG.min_retrieval_confidence, step=0.02,
                                                  label="Refusal threshold")
                    confidence_display = gr.HTML(label="Confidence")
            answer_display = gr.Markdown(label="Answer", elem_id="answer-box")
            with gr.Accordion("Retrieved evidence", open=True):
                evidence_display = gr.Markdown()
            with gr.Accordion("Citations", open=False):
                citation_display = gr.Markdown()
            evidence_table = gr.Dataframe(label="Evidence table", wrap=True, interactive=False)

            ask_button.click(
                ui_ask,
                inputs=[question_box, top_k_slider, rerank_toggle, weight_slider, confidence_slider],
                outputs=[answer_display, confidence_display, evidence_display, citation_display, evidence_table],
            )
            question_box.submit(
                ui_ask,
                inputs=[question_box, top_k_slider, rerank_toggle, weight_slider, confidence_slider],
                outputs=[answer_display, confidence_display, evidence_display, citation_display, evidence_table],
            )
            clear_button.click(
                lambda: ("", "", "", "", pd.DataFrame(), ""),
                outputs=[answer_display, confidence_display, evidence_display, citation_display,
                         evidence_table, question_box],
            )

        # ------------------------------------------------------------------ Tab 2
        with gr.Tab("2 · Retrieved documents"):
            gr.Markdown("Full scoring trace for the most recent query — dense, sparse, fused and reranked scores.")
            refresh_button = gr.Button("Refresh from last query", variant="secondary")
            retrieval_header = gr.Markdown()
            retrieval_table = gr.Dataframe(wrap=True, interactive=False)
            refresh_button.click(ui_last_retrieval, outputs=[retrieval_header, retrieval_table])

        # ------------------------------------------------------------------ Tab 3
        with gr.Tab("3 · Corpus explorer"):
            gr.Markdown("Browse every indexed chunk with its provenance metadata.")
            with gr.Row():
                source_dropdown = gr.Dropdown(["All"] + sorted(CORPUS["source"].unique()), value="All", label="Source")
                topic_dropdown = gr.Dropdown(["All"] + sorted(CORPUS["topic"].unique()), value="All", label="Topic")
                section_dropdown = gr.Dropdown(["All"] + sorted(CORPUS["section"].unique()), value="All", label="Section")
                type_dropdown = gr.Dropdown(["All"] + sorted(CORPUS["document_type"].unique()), value="All",
                                            label="Document type")
            with gr.Row():
                keyword_box = gr.Textbox(label="Keyword filter", placeholder="e.g. sertraline")
                limit_slider = gr.Slider(10, 300, value=60, step=10, label="Rows to display")
                explore_button = gr.Button("Apply filters", variant="primary")
            explorer_summary = gr.Markdown()
            explorer_table = gr.Dataframe(wrap=True, interactive=False)
            with gr.Accordion("Inspect a single chunk", open=False):
                chunk_input = gr.Textbox(label="chunk_id", placeholder="paste a chunk_id from the table above")
                inspect_button = gr.Button("Show full chunk")
                chunk_display = gr.Markdown()

            explore_button.click(
                ui_explore_corpus,
                inputs=[source_dropdown, topic_dropdown, section_dropdown, type_dropdown, keyword_box, limit_slider],
                outputs=[explorer_summary, explorer_table],
            )
            inspect_button.click(ui_inspect_chunk, inputs=chunk_input, outputs=chunk_display)

        # ------------------------------------------------------------------ Tab 4
        with gr.Tab("4 · Evaluation dashboard"):
            gr.Markdown(
                f"""
### Retrieval metrics — held-out **test** split ({len(test_sample)} questions)
Fusion weights were tuned on the validation split; the test split was used once, for these numbers.
"""
            )
            gr.Dataframe(value=_retrieval_table(), wrap=True, interactive=False)
            if not RETRIEVAL_INTERVALS.empty:
                gr.Markdown("**95% bootstrap intervals on the headline metrics** "
                            "(percentile bootstrap over the test questions)")
                gr.Dataframe(value=RETRIEVAL_INTERVALS, interactive=False)
            if RERANK_EFFECT.get("metrics"):
                _lines = [
                    f"- `{m}` {s['baseline']:.4f} → {s['treatment']:.4f} "
                    f"(**{s['delta']:+.4f}**, 95% CI [{s['ci_low']:+.4f}, {s['ci_high']:+.4f}]) — "
                    + ("significant" if s["significant"] else "not distinguishable from noise")
                    for m, s in RERANK_EFFECT["metrics"].items()
                ]
                gr.Markdown(f"**Reranking effect** ({RERANK_EFFECT['treatment']} vs "
                            f"{RERANK_EFFECT['baseline']}, paired bootstrap)\n\n" + "\n".join(_lines))
            gr.Markdown(f"### RAG quality metrics — backend: `{RAG_EVALUATION['backend']}`")
            gr.Dataframe(value=_rag_table(), interactive=False)
            gr.Markdown("### Fusion-weight tuning (validation split)")
            gr.Dataframe(
                value=WEIGHT_TUNING[[c for c in ["configuration", "recall@5", "recall@10", "mrr", "ndcg@10"]
                                     if c in WEIGHT_TUNING.columns]],
                interactive=False,
            )
            if "delta" in WEIGHT_COMPARISON:
                gr.Markdown(
                    f"Best validation setting `dense={WEIGHT_COMPARISON['best_weight']:.1f}` vs configured "
                    f"default `dense={WEIGHT_COMPARISON['default_weight']:.1f}`: "
                    f"paired delta **{WEIGHT_COMPARISON['delta']:+.4f}** "
                    f"[95% CI {WEIGHT_COMPARISON['ci_low']:+.4f}, {WEIGHT_COMPARISON['ci_high']:+.4f}] — "
                    + ("**significant**, the default should change."
                       if WEIGHT_COMPARISON["significant"]
                       else "**not distinguishable from noise**, so the documented default is kept.")
                )
            gr.Markdown("### Figures")
            gr.Gallery(
                value=[str(path) for path in PLOT_PATHS.values()],
                label="Evaluation figures", columns=2, height=560, object_fit="contain",
            )

        # ------------------------------------------------------------------ Tab 5
        with gr.Tab("5 · System information"):
            gr.Markdown(_system_markdown())
            with gr.Accordion("Non-fatal error log", open=False):
                gr.Dataframe(value=ui_error_log(), wrap=True, interactive=False)
            with gr.Accordion("Data sources and licensing", open=False):
                gr.Markdown(
                    """
| Source | Access | Licence / reuse |
|---|---|---|
| PubMed | NCBI E-utilities | Abstracts freely accessible for research |
| Europe PMC | REST API, OA subset only | CC-BY / CC0 filtered at ingestion |
| MedlinePlus | NLM web service | U.S. Government work — public domain |
| NIMH | Public topic pages | U.S. Government work — public domain |
| WHO (mhGAP, reports) | IRIS PDF download | CC BY-NC-SA 3.0 IGO |
| ICD-11 Chapter 06 | WHO ICD API (if credentials) or bundled scaffold | WHO terms; descriptors author-written |
| NICE / APA / VA-DoD | Metadata + URL only | Deliberately not ingested |

**Documented substitutions:** the ICD-11 API requires OAuth credentials, so a bundled Chapter 06 scaffold plus
public-domain NIMH/MedlinePlus content is used by default; NICE guidance is indexed as reference-only records;
WHO PDF URLs are tried against multiple candidates and degrade to reference-only records on failure.
"""
                )

    return demo


APP = build_interface()
print("Gradio interface built. Launching…")

19:30:15 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
19:30:15 | INFO    | httpx | HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


Gradio interface built. Launching…


In [31]:
# =============================================================================
# SECTION 21b — LAUNCH
# =============================================================================
# `share=True` produces a public link (required for Colab). Set
# CONFIG.gradio_share = False for a local-only server.
# =============================================================================
try:
    APP.queue(max_size=16).launch(
        share=CONFIG.gradio_share,
        debug=False,
        show_error=True,
        inline=True,
    )
except Exception as exc:
    log_error("gradio_launch", exc, "retrying without share link")
    APP.launch(share=False, debug=False, show_error=True, inline=True)

19:30:16 | INFO    | httpx | HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
19:30:16 | INFO    | httpx | HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
19:30:17 | INFO    | httpx | HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()


19:30:17 | INFO    | httpx | HTTP Request: GET https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_linux_amd64 "HTTP/1.1 200 OK"
19:30:17 | INFO    | httpx | HTTP Request: HEAD https://8b19310676e09093ef.gradio.live "HTTP/1.1 200 OK"


* Running on public URL: https://8b19310676e09093ef.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 22. Example Queries

The cell below exercises the pipeline programmatically across a spread of clinical intents plus two negative
controls (an out-of-domain question and an unanswerable one) so that the abstention behaviour is visible
without touching the UI.

In [32]:
# =============================================================================
# SECTION 22 — EXAMPLE QUERIES & NEGATIVE CONTROLS
# =============================================================================

DEMO_QUERIES: List[Tuple[str, str]] = [
    ("diagnosis", "What are the diagnostic features of generalised anxiety disorder?"),
    ("treatment", "Which psychosocial interventions are recommended for schizophrenia?"),
    ("medications", "What does the evidence say about lithium in bipolar disorder maintenance?"),
    ("follow-up", "How should patients be followed up after a self-harm presentation?"),
    ("epidemiology", "What is the reported prevalence of depression in primary care?"),
    ("NEGATIVE CONTROL — out of domain", "What is the best way to replace a car's timing belt?"),
    ("NEGATIVE CONTROL — unanswerable", "What will the 2035 global suicide rate be, precisely?"),
]


def run_demo_queries(queries: Sequence[Tuple[str, str]]) -> pd.DataFrame:
    """Execute demonstration queries and summarise pipeline behaviour."""
    rows: List[Dict[str, Any]] = []
    for intent, question in queries:
        result = PIPELINE.answer(question, top_k=CONFIG.top_k_default)
        print("=" * 100)
        print(f"[{intent}] {question}")
        print("-" * 100)
        print(f"confidence {result.confidence:.3f} | refused {result.refused} | "
              f"grounding {result.grounding.verdict if result.grounding else 'n/a'} | "
              f"{result.latency.get('total_s', 0):.2f}s | {result.tokens.get('total', 0)} tokens")
        print(result.answer[:900])
        if result.evidence and not result.refused:
            top = result.evidence[0]
            print(f"\ntop evidence → [{top.metadata.get('organization')}] "
                  f"{str(top.metadata.get('title'))[:80]} ({top.metadata.get('url')})")
        print()
        rows.append(
            {
                "intent": intent,
                "question": question[:70],
                "confidence": round(result.confidence, 3),
                "refused": result.refused,
                "verdict": result.grounding.verdict if result.grounding else "n/a",
                "grounding": result.grounding.grounding_score if result.grounding else None,
                "citations_used": sum(1 for c in result.citations if c["used"]),
                "latency_s": round(result.latency.get("total_s", 0.0), 2),
                "total_tokens": result.tokens.get("total", 0),
            }
        )
    return pd.DataFrame(rows)


set_global_seeds(CONFIG.seed)
DEMO_RESULTS = run_demo_queries(DEMO_QUERIES)
print("=" * 100)
print("DEMONSTRATION SUMMARY")
print("=" * 100)
print(DEMO_RESULTS.to_string(index=False))
print(
    "\nExpected behaviour: in-domain questions produce cited, grounded answers; both negative controls "
    "should be refused with low confidence."
)

19:30:17 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


[diagnosis] What are the diagnostic features of generalised anxiety disorder?
----------------------------------------------------------------------------------------------------
confidence 0.759 | refused False | grounding grounded | 33.31s | 2380 tokens
**Answer:** Generalized anxiety disorder (GAD) is characterized by excessive anxiety and worry about a number of life circumstances, along with physical symptoms like restlessness, fatigue, and difficulty concentrating [2]. It is also associated with anticipatory anxiety and chronic worry, which are core features of the disorder [1]. The diagnostic criteria emphasize the presence of these symptoms, along with physical symptoms such as dizziness and muscle tension [4].

**Evidence summary:** 
- GAD is distinguished by worry about multiple life circumstances and a cognitive aspect of anxiety [2].
- Symptoms include worry, social and performance fears, and physical symptoms like palpitations and dizziness [4].
- The GAD-7 scale is a tool

---
## 23. Limitations

**Corpus**
* **Abstract-heavy.** Most PubMed content is abstracts, not full text. Abstracts compress away exactly the
  operational detail (doses, monitoring schedules, contraindications) clinicians most need.
* **Licensing gaps by design.** NICE, APA and VA/DoD guidance — arguably the highest-value content for these
  questions — is indexed as reference-only records. The assistant can point at them but cannot quote them.
* **ICD-11 scaffold.** Without OAuth credentials the ICD-11 layer carries codes, titles, hierarchy and
  author-written descriptors rather than the official CDDR text.
* **English-only, high-income bias.** Sources skew to English-language, Western guidance; global applicability
  is limited.
* **Availability drift.** WHO IRIS URLs and NIMH page structures change; a future run may ingest a slightly
  different corpus, which is why every artefact is versioned in the cache.

**Benchmark**
* Questions are **synthesised from the corpus**, so they measure self-consistency of retrieval rather than
  agreement with independent clinical judgment. Absolute metric values are therefore optimistic; the
  *relative* comparisons (BM25 vs dense vs hybrid vs reranked) are the trustworthy signal.
* Relevance labels are heuristic (topic + section, or document + section), not expert-annotated. Chunks that
  are genuinely relevant but sit in a different section are scored as false positives.
* Reference answers are extractive, which flatters answer-correctness scoring.

**System**
* **Grounding ≠ truth.** The verifier measures whether the answer restates the evidence, not whether the
  evidence is correct, current or applicable to a given patient.
* Confidence is a heuristic blend of retrieval signals, not a calibrated probability; the thresholds are
  tuned, not validated against clinician judgement.
* Reranking roughly doubles query latency, and the LLM dominates end-to-end time on a T4.
* No multi-hop reasoning, no query decomposition, no conversational memory — each question is answered
  independently.
* **Not clinically validated. Not a medical device.**

---
## 24. Future Improvements

| Priority | Improvement | Rationale |
|---|---|---|
| High | **Clinician-annotated benchmark** (100–200 questions with graded relevance) | Removes the self-consistency bias in every current metric |
| High | **Full-text ingestion via PMC OA bulk packages** | Replaces abstracts with methods-and-results depth |
| High | **Query decomposition + multi-hop retrieval** | "Compare first-line treatments for OCD and PTSD" needs two evidence sets |
| Medium | **Fine-tuned domain reranker** (train on the benchmark's train split) | Cross-encoders gain the most from in-domain supervision |
| Medium | **HyDE / query expansion with MeSH terms** | Bridges lay-language questions to clinical vocabulary |
| Medium | **NLI-based faithfulness** (DeBERTa-MNLI entailment per claim) | A stricter grounding check than embedding similarity |
| Medium | **Calibrated confidence** via isotonic regression on labelled outcomes | Turns the heuristic score into an actual probability |
| Medium | **Recency and evidence-quality weighting** (GRADE-aware boosting) | A 2024 meta-analysis should outrank a 2015 narrative review |
| Low | **Approximate index (HNSW/IVF-PQ) + incremental ingestion** | Needed beyond ~10⁵ chunks and for nightly refreshes |
| Low | **vLLM or TGI serving, structured JSON output** | Throughput and machine-readable citations |
| Low | **Multilingual corpus and embeddings** | WHO mhGAP exists in many languages; global reach |

---
## 25. Code Quality & Reproducibility Notes

* **Object-oriented core.** `BaseConnector`, `TextCleaner`, `Deduplicator`, `SemanticChunker`,
  `BenchmarkBuilder`, `EmbeddingModel`, `FaissIndex`, `BM25Index`, `HybridRetriever`,
  `CrossEncoderReranker`, `PromptBuilder`, `LLMEngine`, `GroundingVerifier`, `ConfidenceEstimator`,
  `RetrievalEvaluator`, `NativeRagMetrics` and `PsychiatryRAGPipeline` each own one responsibility.
* **Type hints and docstrings** throughout; PEP 8 formatting; no duplicated logic between the evaluation
  cells and the UI (both call the same pipeline object).
* **Deterministic.** One seed constant drives Python, NumPy and PyTorch, plus the benchmark split and the
  MinHash permutations.
* **Idempotent.** Every expensive stage goes through `CacheManager.get_or_build`, so the notebook can be
  re-run top to bottom without manual cleanup. Set `CONFIG.force_rebuild = True` to invalidate everything.
* **Fail-soft.** Every network call, parser, model load and UI callback is wrapped; failures are logged to
  `ERROR_LOG` and surfaced in Tab 5 rather than raised.
* **Auditable.** Package versions, hardware, configuration, connector statistics, cleaning statistics and
  evaluation summaries are all written to the cache directory.

### Reproducing this notebook
```
Runtime → Change runtime type → GPU (T4 or better)
Runtime → Run all
```
First run: ~10–25 minutes (downloads + embeddings + evaluation). Subsequent runs: under a minute from cache.

---
> **Final reminder.** This project is an engineering demonstration of retrieval-augmented generation over open
> medical literature. It is not clinically validated, is not a medical device, and must not be used to make
> diagnostic or treatment decisions. If you or someone else may be at risk of harm, contact your local
> emergency services or a crisis line immediately.